# Suficiencia energética del SIN 2027-2030 y expansión de mínimo costo

**Tutorial del Trabajo Final · Taller de Programación en Python (MECA 4110), 2026-2 · Universidad de los Andes**
Carlos Andrés Terrassa Vallejo (código 200518213) · Esteban Darío Herrera León (código 202424968) · Jair Armando Castañeda Rodríguez (código 202616631)

Este tutorial responde qué expansión de mínimo costo necesita el Sistema Interconectado Nacional (SIN) de Colombia para que su energía firme cubra la demanda proyectada entre 2026 y 2030. Para eso mide cuánta energía firme falta en cada período y resuelve un modelo de optimización con variables enteras. Todo el código está en este notebook, y los datos vienen en la carpeta `datos_tutorial/` del .zip: corre de principio a fin sin internet.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cterrassa/suficiencia-sin/blob/main/Tutorial_Suficiencia_SIN.ipynb) [![Abrir en Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/cterrassa/suficiencia-sin/HEAD?urlpath=lab/tree/Tutorial_Suficiencia_SIN.ipynb) · [Versión web, ya ejecutada](https://cterrassa.github.io/suficiencia-sin/) · [App interactiva](https://cterrassa.github.io/suficiencia-sin/app/) · [Repositorio](https://github.com/cterrassa/suficiencia-sin)

**Contenido**

1. [Contexto y motivación](#contexto): el problema, la pregunta, los antecedentes y por qué Python
2. [Datos](#datos): instalación, fuentes, una consulta en vivo, limpieza, revisión y uniones
3. [Metodología y desarrollo](#metodologia): homologación con RapidFuzz, costos, balance de energía firme y modelo de expansión
4. [Resultados y visualización](#resultados): siete figuras, dos animaciones y la respuesta a la pregunta
5. [Uso de IA para ir más allá de la metodología](#mas-alla): la consulta, la respuesta completa, lo que va más allá de lo que recomendaron los profesores y dos mejoras con su efecto
6. [Ejercicios para el lector](#ejercicios), que se revisan solos, con sus [soluciones](#soluciones)

[Cómo atendimos la evaluación de la propuesta](#evaluacion) · [Declaración de uso de IA](#ia) · [Referencias](#referencias)

**Cómo está hecho.** Cada paso sigue el mismo orden: un texto que explica qué se va a hacer y por qué, el código, y una lectura del resultado. Cada sección trae una tabla con las funciones y los métodos que usa: qué hacen, con qué parámetros los llamamos y por qué. Las funciones propias tienen docstring.

**Qué se necesita:** Python 3.14 con los paquetes de la sección 2.1 y el notebook al lado de la carpeta `datos_tutorial/`, como vienen en el .zip. Internet es opcional: con conexión, el notebook consulta además la API de XM en vivo, y si falta algún archivo de datos lo baja de GitHub y lo verifica con su huella SHA-256. Con los botones de Colab o de Binder corre en el navegador, sin instalar nada. Para mover los supuestos del modelo sin correr nada está la [app interactiva](https://cterrassa.github.io/suficiencia-sin/app/), que abre en el navegador.

<a id="contexto" name="contexto"></a>
## 1. Contexto y motivación

Esta sección plantea el problema, la pregunta que lo responde, lo que ya se sabe del tema y por qué Python es la herramienta adecuada. De aquí salen los datos que busca la sección 2.

### El problema

El SIN depende del agua. De los 23,2 GW de capacidad efectiva que tenía en octubre de 2026, sin contar la generación distribuida, 13,2 eran hidráulicos (XM).

Cuando llega El Niño, llueve menos en los Andes colombianos y bajan los caudales de los ríos que alimentan los embalses (Poveda et al., 2011): el sistema queda en manos del agua embalsada y de las térmicas. El precio lo refleja. En los meses de hidrología seca, el precio de bolsa real promedia el doble que en los normales (sección 2.5), y Ríos et al. (2024) muestran que ese encarecimiento del mercado mayorista llega también a las tarifas de los usuarios.

Para asegurar que haya con qué atender la demanda en un año seco, la CREG creó en 2006 el Cargo por Confiabilidad (Resolución 071 de 2006). Cada planta tiene una energía firme, la ENFICC: la que puede garantizar con la hidrología crítica de la serie histórica, si es hidráulica, o con su disponibilidad y su combustible, si es térmica. En subastas, el sistema compra obligaciones de energía firme (OEF) para cada período de diciembre a noviembre; quien las vende recibe un pago fijo a cambio de entregar esa energía cuando el precio de bolsa supera el precio de escasez. Por eso saber si el sistema alcanza es un asunto de energía firme: si la ENFICC del parque, más la de los proyectos que se comprometieron en las subastas, cubre la demanda que proyecta la UPME.

Hay razones para dudarlo:
- **Los proyectos llegan tarde.** De la OEF que la subasta de 2019 asignó a plantas nuevas, el 47% entró tarde o no ha entrado (sección 3.1).
- **La demanda subió.** La proyección media de la UPME para 2029-2030 creció 5% entre la que usó la CREG para convocar la última subasta (enero de 2025, según el Documento CREG 901 202) y la de julio de 2026.
- **El operador ve un faltante.** Según la prensa, el Boletín 364 de XM estima hasta 7.669 GWh de faltante de energía firme en 2026-2027.
- **Hay un El Niño fuerte en curso:** el índice ONI llegó a 2,46 °C en agosto-octubre de 2026 (sección 2.3).

Un faltante de energía firme no es un apagón seguro: en un año de lluvias normales la hidráulica genera más que su energía firme. Es la exposición a un racionamiento en el próximo año seco. Y construir toma años: una térmica de ciclo abierto que se decida hoy no opera antes de diciembre de 2028 (sección 3.2). Si hay que actuar, la decisión es de ahora.

### La pregunta

Una sola pregunta guía el tutorial. Debajo están lo que abarca y el orden en que se responde.

> **¿Qué expansión de mínimo costo necesita el Sistema Interconectado Nacional (SIN) para que su energía firme cubra la demanda proyectada por la UPME en cada período del Cargo por Confiabilidad entre 2026-2027 y 2029-2030?**

**Qué abarca:**
- **Unidad:** el SIN de Colombia; no incluye las zonas no interconectadas.
- **Período:** los cuatro períodos del Cargo por Confiabilidad, de diciembre de 2026 a noviembre de 2030, con datos al 4 de noviembre de 2026.
- **Variables:** la energía firme del parque (ENFICC) y de los proyectos con OEF de las subastas, con sus retrasos; la demanda total proyectada por la UPME (revisión de julio de 2026), en sus escenarios medio y alto, y los costos de cada tecnología (Catálogo Tecnológico de la UPME y PIEG 2025-2039).
- **Alcance:** energía firme, la que el sistema garantiza en un año seco como los de El Niño, con un año de hidrología normal como contraste. No incluye la potencia hora a hora, la red de transmisión ni el precio de bolsa.

La respuesta tiene tres partes, en este orden: cuánta energía firme falta y en qué período (sección 3.3); qué bloques se construyen y cuándo (sección 3.4), y cuánto cuestan y qué energía no alcanza a cubrirse a tiempo. Son partes de una sola respuesta, porque el modelo de expansión toma la brecha como insumo. La sección 4.8 responde en ese orden.

**El Niño y el año normal.** El balance principal usa la energía firme (ENFICC), la que el parque garantiza con hidrología crítica: es el escenario de El Niño, y es el que dimensiona la expansión. Como contraste, la sección 3.3 calcula también un año de hidrología normal, en el que la hidráulica genera más que su energía firme; así se ve que la brecha firme mide el riesgo del año seco, no un faltante seguro. Los tres escenarios (base, estrés y optimista) cambian la demanda y los retrasos de los proyectos; la hidrología cambia entre El Niño y el año normal.

### Antecedentes

Cuatro líneas de trabajo enmarcan el tutorial: el diseño del mercado de energía firme, cómo le ha ido en Colombia, la planeación oficial y el emparejamiento de registros sin un código común.

- **El diseño del mercado de energía firme.** Cramton y Stoft (2007) describen el mecanismo que Colombia adoptó en 2006: obligaciones de energía firme que se venden en una subasta de reloj descendente y funcionan como una opción de compra con precio de ejercicio igual al precio de escasez. Cramton, Ockenfels y Stoft (2013) explican por qué un mercado de energía con precios topados necesita un mecanismo así para pagar la capacidad que solo se usa en la escasez.
- **Cómo le ha ido.** Harbord y Pagnozzi (2014) revisan las subastas de Colombia y de Nueva Inglaterra como lecciones para el diseño británico. Juvinao Navarro (2021) concluye que la inestabilidad regulatoria del cargo redistribuyó rentas hacia los grupos de interés mejor organizados y recomienda un regulador más independiente.
- **La planeación oficial.** La UPME publica el Plan Indicativo de Expansión de la Generación (PIEG 2025-2039), que optimiza la expansión de largo plazo y trae su propio balance de energía firme. Este tutorial usa sus costos, sus factores firmes y su tasa de descuento. Los modelos de expansión de mínimo costo con variables enteras son la herramienta estándar de esa planeación (Koltsaklis y Dagoumas, 2018).
- **Emparejar registros sin un código común** es un problema clásico de la estadística. Fellegi y Sunter (1969) lo formalizan como una decisión entre aceptar el par, rechazarlo o revisarlo a mano: la misma lógica de los dos umbrales de la sección 3.1.

**Qué agrega este trabajo.** Un balance de energía firme por período, reproducible desde datos públicos, con tres piezas que no aparecen juntas en los documentos oficiales: los retrasos medidos en las subastas reales, gracias a la homologación (sección 3.1); la oferta física del parque, y no solo las obligaciones asignadas (sección 3.3), y una expansión que respeta las fechas en que cada tecnología puede llegar (sección 3.4).

### ¿Por qué Python?

El problema junta cuatro tareas que suelen vivir en herramientas distintas: bajar datos de varias API, emparejar nombres de plantas sin un código común, optimizar con variables enteras y mostrar los resultados.

Python las resuelve en un solo flujo, que se vuelve a correr cuando XM publica un mes nuevo:
- **`requests`** consulta la API de XM y baja los datos que falten. **`pandas`** limpia las tablas mensuales y valida cada unión (`merge(..., validate=...)`), para que una llave repetida no duplique filas sin aviso.
- **`rapidfuzz`** calcula en C++, con todos los núcleos del procesador, los puntajes de similitud entre miles de nombres (`process.cdist`). En una hoja de cálculo eso sería impráctico.
- **`scipy.optimize.milp`** resuelve el modelo de expansión con HiGHS, un solver de código abierto que no pide las licencias de GAMS o CPLEX; **`numpy`** arma la matriz del problema.
- **`plotly`** produce figuras interactivas, que se pueden explorar pasando el cursor.

La sección 2.1 dice cómo instalar cada paquete, quién lo desarrolla y dónde está su documentación.

**Lo que sigue.** Para responder la pregunta hacen falta tres insumos: la energía firme del parque y de los proyectos (XM), la demanda proyectada (UPME) y los costos de las tecnologías (UPME). La sección 2 los consigue, los revisa y los deja listos.

<a id="datos" name="datos"></a>
## 2. Datos

La sección 1 pidió tres insumos: la energía firme, la demanda proyectada y los costos. Esta sección instala lo necesario, los consigue de fuentes públicas y los deja limpios, revisados y unidos para las secciones 3 y 4.

Las fuentes son XM (el operador del sistema), la UPME (el planeador), NOAA (el índice de El Niño), el DANE (el IPC) y la CREG (las reglas).

### 2.1 Instalación y configuración

Antes de los datos, el entorno: cómo correr el notebook en el navegador o en su equipo, y una sola celda que carga todos los paquetes.

**La vía corta: Colab o Binder.** Los dos botones de la portada abren el notebook en el navegador, sin instalar nada:
- **Colab** pide una cuenta de Google que lo tenga habilitado. La primera celda de código detecta Colab e instala RapidFuzz, el único paquete de la tabla que Colab no trae.
- **Binder** (mybinder.org) no pide cuenta. Arma un entorno con Python 3.14 y las versiones exactas de la tabla, a partir de los archivos `runtime.txt` y `requirements.txt` del repositorio, y abre el notebook en JupyterLab. La primera vez puede tardar varios minutos en arrancar; la sesión se cierra tras unos diez minutos sin uso y no guarda los cambios.

**Requisitos para correrlo en su equipo.** Python 3.14; lo probamos con Python 3.14.7 en Windows 11 (noviembre de 2026). En macOS y Linux los pasos son equivalentes, pero no los probamos. Para abrir el notebook hace falta JupyterLab o VS Code con su extensión de Jupyter. Internet no hace falta si el notebook está al lado de la carpeta `datos_tutorial/`, como viene en el .zip.

En Windows (PowerShell):

```powershell
python -m venv $env:USERPROFILE\.venvs\suficiencia_sin
& $env:USERPROFILE\.venvs\suficiencia_sin\Scripts\Activate.ps1
python -m pip install pandas==3.0.3 numpy==2.5.4 requests==2.34.2 rapidfuzz==3.14.6 scipy==1.18.1 plotly==7.1.0 jupyterlab
python -m jupyter lab
```

Si PowerShell no deja correr `Activate.ps1`, se habilita solo para esa ventana con `Set-ExecutionPolicy -Scope Process Bypass`.

En macOS o Linux (Terminal):

```bash
python3 -m venv ~/.venvs/suficiencia_sin
source ~/.venvs/suficiencia_sin/bin/activate
python -m pip install pandas==3.0.3 numpy==2.5.4 requests==2.34.2 rapidfuzz==3.14.6 scipy==1.18.1 plotly==7.1.0 jupyterlab
python -m jupyter lab
```

| Paquete | Versión | Para qué lo usamos | Quién lo desarrolla | Documentación oficial |
|---|---|---|---|---|
| pandas | 3.0.3 | Tablas: lectura, limpieza, uniones y agregaciones | pandas Development Team (proyecto comunitario) | [pandas.pydata.org/docs](https://pandas.pydata.org/docs/) |
| NumPy | 2.5.4 | Vectores y matrices del modelo de expansión y de la simulación del ejercicio 3 | NumPy Developers | [numpy.org/doc/stable](https://numpy.org/doc/stable/) |
| requests | 2.34.2 | La consulta a la API de XM y la descarga de los datos | Creado por Kenneth Reitz; hoy en la Python Software Foundation | [requests.readthedocs.io](https://requests.readthedocs.io/) |
| RapidFuzz | 3.14.6 | Similitud de nombres en la homologación | Max Bachmann | [rapidfuzz.github.io/RapidFuzz](https://rapidfuzz.github.io/RapidFuzz/) |
| SciPy | 1.18.1 | `milp`, que llama al solver HiGHS | SciPy Developers; HiGHS nació en la Universidad de Edimburgo | [scipy.optimize.milp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.milp.html) y [HiGHS](https://highs.dev/) |
| Plotly | 7.1.0 | Figuras interactivas | Plotly (empresa) | [plotly.com/python](https://plotly.com/python/) |

La primera celda solo actúa en Colab: instala RapidFuzz con `pip`. La segunda importa todo lo que usa el notebook: primero la biblioteca estándar de Python y después los paquetes de la tabla.

In [1]:
# En Google Colab: instalar RapidFuzz, el único paquete de la tabla que Colab no trae
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz==3.14.6"], check=True)
    print("Colab: RapidFuzz instalado")

In [2]:
# Biblioteca estándar
import hashlib
import json
import os
import re
import time
import unicodedata
from importlib.metadata import PackageNotFoundError, version
from io import StringIO
from pathlib import Path

# Paquetes de la tabla de la sección 2.1
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from IPython.display import display
from plotly.subplots import make_subplots
from rapidfuzz import fuzz, process
from scipy.optimize import Bounds, LinearConstraint, milp

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
SEMILLA = 2026   # semilla de todas las simulaciones (secciones 5 y 6): el notebook da siempre los mismos resultados

La tercera celda comprueba las versiones instaladas.

**Funciones y métodos de esta celda**

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `importlib.metadata.version(paquete)` | Devuelve la versión instalada de un paquete; lanza `PackageNotFoundError` si no está | El nombre del paquete en PyPI | Viene con Python: no hace falta importar cada paquete para saber su versión |
| `revisar_paquetes(probadas)` (propia) | Compara cada versión instalada con la que probamos | Un diccionario `{paquete: versión}` | Atrapa `PackageNotFoundError` para reportar el paquete que falta en lugar de detener el notebook |

In [3]:
PROBADAS = {"pandas": "3.0.3", "numpy": "2.5.4", "requests": "2.34.2", "rapidfuzz": "3.14.6", "scipy": "1.18.1",
            "plotly": "7.1.0"}


def revisar_paquetes(probadas):
    """Compara la versión instalada de cada paquete con la que probamos.

    probadas: diccionario {paquete: versión probada}, con el nombre del paquete en PyPI.
    Devuelve una fila por paquete con la versión instalada ("no instalado" si falta) y si coincide.
    Una versión distinta suele funcionar; si algo falla, es lo primero que hay que revisar.
    """
    filas = []
    for paquete, probada in probadas.items():
        try:
            instalada = version(paquete)
        except PackageNotFoundError:  # el paquete no está: se reporta en lugar de detener el notebook
            instalada = "no instalado"
        filas.append({"paquete": paquete, "probada": probada, "instalada": instalada,
                      "coincide": instalada == probada})
    return pd.DataFrame(filas).set_index("paquete")


revisar_paquetes(PROBADAS)

,probada,instalada,coincide
paquete,,,
pandas,3.0.3,3.0.3,True
numpy,2.5.4,2.5.4,True
requests,2.34.2,2.34.2,True
rapidfuzz,3.14.6,3.14.6,True
scipy,1.18.1,1.18.1,True
plotly,7.1.0,7.1.0,True


### 2.2 Las fuentes y el paquete de datos

De dónde sale cada dato y cómo llega al notebook. Casi todo viene en un paquete de datos verificado, que trae el .zip; la API de XM se consulta además en vivo (sección 2.3).

| Fuente | Qué se toma | Cómo llega al notebook |
|---|---|---|
| [XM, API de datos](https://github.com/EquipoAnaliticaXM/API_XM) | OEF por planta de octubre de 2026 | Consulta en vivo (sección 2.3) |
| XM, API y SIMEM | Demanda, aportes hídricos, embalse y precio de bolsa (mensual, 2000-2026); ENFICC y capacidad por planta; OEF de los proyectos nuevos; generación y ENFICC de las hidráulicas por período desde 2010 (año normal) | Paquete de datos |
| [UPME, proyección de demanda 2026-2040](https://www.upme.gov.co/simec/planeacion-energetica/proyeccion-de-demanda-v2/) (rev. julio de 2026) | Demanda mensual por escenario y por definición | Paquete de datos |
| [UPME, Catálogo Tecnológico Colombiano](https://api.upme.gov.co/catalogo_tecnologias/api/techs_year/) | Costos por tecnología | Paquete de datos, ya procesados (sección 3.2) |
| [UPME, PIEG 2025-2039](https://docs.upme.gov.co/SIMEC/Energia%20Electrica/PIEG/2025-2039/PIEG_2025_2039_Version_final.pdf) | Factores de energía firme, tasa de descuento, vidas útiles y años mínimos de entrada | Supuestos de la sección 3 |
| [NOAA, índice ONI](https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt) | Fases de El Niño y La Niña | Paquete de datos (sección 2.3) |
| [DANE](https://www.dane.gov.co/), IPC | Precio de bolsa en pesos constantes | Paquete de datos |
| XM, SIMEM y registro de proyectos de la UPME | Las tres bases que se homologan y los retrasos de las subastas | Paquete de datos |
| [CREG, Resolución 071 de 2006](https://gestornormativo.creg.gov.co/gestor/entorno/docs/resolucion_creg_0071_2006.htm) | Las reglas de la ENFICC, las OEF y las subastas | Contexto |

**Por qué un paquete de datos.** Bajar la OEF y la ENFICC por planta, día a día desde 2006, y la generación por planta, hora a hora desde 2010, toma unas dos horas con la API de XM. Por eso el notebook consulta en vivo solo la API de XM (sección 2.3) y lee lo demás de un paquete de 1,2 MB que viene en el .zip y también está publicado en GitHub. El paquete lo armó el pipeline completo del proyecto con las mismas fuentes: son las tablas limpias y algunos insumos intermedios, no los resultados del análisis. Aparte trae los resultados del pipeline, solo para que la sección 3 compruebe que llega a lo mismo.

**Cómo se verifica.** El notebook guarda la huella SHA-256 del manifiesto (`manifiesto.csv`), y el manifiesto, la de cada archivo. Si un archivo cambia en un solo byte, su huella cambia y el notebook se detiene en lugar de seguir con datos distintos. Si la carpeta `datos_tutorial/` ya existe al lado del notebook, solo se baja lo que falta.

**Funciones y métodos de esta sección**

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `requests.get(url, timeout)` | Hace una solicitud HTTP GET y devuelve la respuesta | `timeout=60` segundos | Sin `timeout`, una conexión colgada detiene el notebook para siempre |
| `respuesta.raise_for_status()` | Lanza un error si la respuesta no fue exitosa (404, 500...) | Ninguno | Así un error del servidor no se guarda como si fuera el archivo |
| `hashlib.sha256(bytes).hexdigest()` | Calcula la huella SHA-256 de un contenido | Los bytes del archivo | Cambia por completo si cambia un solo byte |
| `huella(ruta)`, `descargar(nombre)` y `obtener_datos()` (propias) | Calculan la huella, bajan un archivo con reintentos y dejan verificado todo el paquete | El nombre del archivo; 3 intentos | Una red intermitente no debería tumbar el notebook al primer intento |

In [4]:
# Dónde están los datos: la carpeta local si ya existe; si no, el repositorio público
URL_DATOS = os.environ.get("URL_DATOS_SUFICIENCIA",   # otra copia del paquete, solo para pruebas
                           "https://raw.githubusercontent.com/cterrassa/suficiencia-sin/main/datos_tutorial/")
CARPETA = Path("datos_tutorial")
HUELLA_MANIFIESTO = "1d3533e63b83ba4cb14418e5c98e4d555f353fcefd3a613e91b38a889416aaf3"   # SHA-256 de manifiesto.csv


def huella(ruta):
    """Huella SHA-256 de un archivo, en hexadecimal."""
    return hashlib.sha256(Path(ruta).read_bytes()).hexdigest()


def descargar(nombre, intentos=3):
    """Baja un archivo del repositorio de datos a CARPETA; reintenta si la conexión falla."""
    for k in range(1, intentos + 1):
        try:
            respuesta = requests.get(URL_DATOS + nombre, timeout=60)
            respuesta.raise_for_status()
            (CARPETA / nombre).write_bytes(respuesta.content)
            return
        except requests.RequestException as error:
            if k == intentos:
                raise RuntimeError(f"No se pudo bajar {nombre}: {error}") from error
            time.sleep(2 * k)

In [5]:
def obtener_datos():
    """Deja en CARPETA los archivos del manifiesto, cada uno verificado con su huella SHA-256.

    Baja solo lo que falta o no coincide. Si el manifiesto no tiene la huella esperada, se detiene:
    los datos no serían los que usa este tutorial. Devuelve el manifiesto.
    """
    CARPETA.mkdir(exist_ok=True)
    manifiesto = CARPETA / "manifiesto.csv"
    if not manifiesto.exists() or huella(manifiesto) != HUELLA_MANIFIESTO:
        descargar("manifiesto.csv")
    if huella(manifiesto) != HUELLA_MANIFIESTO:
        raise ValueError("manifiesto.csv no coincide con la huella esperada")
    m = pd.read_csv(manifiesto)
    for archivo, sha in zip(m["archivo"], m["sha256"]):
        ruta = CARPETA / archivo
        if not ruta.exists() or huella(ruta) != sha:
            descargar(archivo)
        if huella(ruta) != sha:
            raise ValueError(f"{archivo} no coincide con su huella SHA-256")
    return m


manifiesto = obtener_datos()
print(f"{len(manifiesto)} archivos verificados, {manifiesto['bytes'].sum() / 1e6:.1f} MB")
manifiesto[["archivo", "bytes"]]

18 archivos verificados, 1.2 MB


,archivo,bytes
0,candidatos_costos.csv,4017
1,dane_ipc_mensual.csv,8064
2,entradas_sin_subasta.csv,117
3,homologacion_bases.csv,168616
4,homologacion_etiquetas.csv,4749
5,metadatos.json,516
6,noaa_oni.ascii.txt,23972
7,pipeline_brechas_escenarios.csv,3731
8,pipeline_expansion_resumen.csv,17193
9,proyectos_nuevos_oef.csv,4951


**Lectura.** Los 18 archivos pesan 1,2 MB y todos coinciden con su huella. Los más grandes son la OEF diaria por planta de octubre de 2026 (el respaldo de la consulta en vivo de 2.3), el Catálogo Tecnológico de la UPME (la fuente original de los costos de la sección 3.2), la proyección mensual de demanda y las bases de la homologación; los demás son tablas de unas decenas o cientos de filas.

### 2.3 La API de XM y el índice de El Niño

La API de XM se consulta en vivo, con una copia en el paquete por si no responde. El índice ONI de NOAA se lee del paquete: con él se marcan los meses de El Niño y La Niña que usa la sección 2.5.

**La API de XM.** Recibe una solicitud POST con la métrica, la entidad y las fechas (a lo sumo 31 días) y devuelve un JSON: una lista de días y, dentro de cada día, una entrada por planta con su código y su valor. `consultar_xm()` la convierte en una tabla larga (`fecha`, `codigo`, `valor`). El ejemplo pide la OEF por planta de octubre de 2026.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `requests.post(url, json, timeout)` | Envía una solicitud POST con un cuerpo en JSON | El cuerpo que pide la API y `timeout=120` | La API de XM solo acepta POST; algunas consultas tardan más de un minuto |
| `pd.to_datetime(serie)` | Convierte texto en fechas | Las fechas que devuelve la API (`"2026-10-01"`) | Sin fechas de verdad no se puede agrupar por mes |
| `pd.to_numeric(serie, errors="coerce")` | Convierte texto en números; lo que no es número queda como faltante | `errors="coerce"` | La API devuelve a veces los valores como texto |
| `consultar_xm(metrica, entidad, desde, hasta, respaldo)` (propia) | Consulta la API con reintentos y, si no responde, lee el respaldo | `"ObligEnerFirme"`, `"Recurso"` y octubre de 2026 | La OEF se publica por planta (`Recurso`) y por día |

In [6]:
URL_XM = "https://servapibi.xm.com.co/daily"


def consultar_xm(metrica, entidad, desde, hasta, respaldo, intentos=3):
    """Consulta una métrica diaria de la API de XM y la devuelve como tabla larga (fecha, codigo, valor).

    metrica y entidad: códigos de XM, por ejemplo "ObligEnerFirme" y "Recurso".
    desde, hasta: fechas "AAAA-MM-DD", a lo sumo 31 días entre sí (límite de la API).
    respaldo: CSV con la misma tabla, que se usa si la API no responde. Devuelve (tabla, origen).
    """
    cuerpo = {"MetricId": metrica, "Entity": entidad, "Filter": [], "StartDate": desde, "EndDate": hasta}
    for k in range(1, intentos + 1):
        try:
            respuesta = requests.post(URL_XM, json=cuerpo, timeout=120)
            respuesta.raise_for_status()
            filas = []
            for dia in respuesta.json()["Items"]:
                for e in dia["DailyEntities"]:
                    v = e.get("Values", e)
                    filas.append({"fecha": dia["Date"], "codigo": v.get("code", v.get("Code")), "valor": v["Value"]})
            t = pd.DataFrame(filas)
            t["fecha"], t["valor"] = pd.to_datetime(t["fecha"]), pd.to_numeric(t["valor"], errors="coerce")
            return t, "API de XM, en vivo"
        except (requests.RequestException, KeyError, ValueError) as error:
            ultimo = error
            time.sleep(2 * k)
    return pd.read_csv(respaldo, parse_dates=["fecha"]), f"respaldo del paquete ({type(ultimo).__name__})"

In [7]:
oef_oct, origen = consultar_xm("ObligEnerFirme", "Recurso", "2026-10-01", "2026-10-31",
                               respaldo=CARPETA / "xm_oef_recurso_2026_10.csv")
oef_oct = oef_oct.query("valor > 0")   # la API trae todas las plantas del listado, la mayoría en cero
print(f"Origen: {origen}")
print(f"{len(oef_oct):,} filas planta-día; {oef_oct['codigo'].nunique()} plantas con OEF")
print(f"OEF total de octubre de 2026: {oef_oct['valor'].sum() / 1e6:,.1f} GWh (la API da kWh por día)")
oef_oct.head()

Origen: respaldo del paquete (ProxyError)
17,050 filas planta-día; 590 plantas con OEF
OEF total de octubre de 2026: 8,413.1 GWh (la API da kWh por día)


,fecha,codigo,valor
0,2026-10-01,2QBW,341118.83
1,2026-10-01,2QRL,15042.84
2,2026-10-01,2QV2,131165.37
3,2026-10-01,2R22,0.01
4,2026-10-01,2S6Q,2002.20


**Lectura.** En octubre de 2026 tenían OEF 589 plantas: 17.062 filas planta-día, porque cada planta con OEF aparece todos los días. La OEF total del mes fue de 8.205 GWh, unos 265 GWh-día. Ese total incluye a las plantas menores, a las que la API les reporta como "OEF" su generación del mes y no una energía firme. Por eso la sección 3.3 no suma esta cifra: suma la ENFICC del parque y le da a cada menor un factor firme.

**El índice ONI de NOAA.** Es la anomalía de la temperatura del Pacífico ecuatorial en trimestres móviles (`DJF` es diciembre-enero-febrero). Cada trimestre se asigna a su mes central. Un mes es de El Niño cuando el ONI pasa de +0,5 °C durante al menos cinco trimestres seguidos, y de La Niña cuando baja de −0,5 °C; es el criterio de NOAA. El notebook lee la copia del paquete, cortada en la fecha de corte del proyecto (4 de noviembre de 2026), para que los resultados no cambien cuando NOAA publique un trimestre nuevo. La fuente original está enlazada en la tabla de la sección 2.2.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `pd.read_csv(texto, sep=r"\s+")` | Lee una tabla de texto | `sep=r"\s+"`: uno o más espacios | El archivo de NOAA separa las columnas con espacios, no con comas |
| `Series.map(diccionario)` | Reemplaza cada valor por el del diccionario | `MES_CENTRAL`: trimestre → mes central | Deja el trimestre como un mes, comparable con las demás series |
| `Series.shift()` y `cumsum()` | Desplazan la serie un lugar y acumulan | — | Juntos numeran cada racha de meses seguidos que cumplen la condición |
| `groupby(...).transform("size")` | Le da a cada fila el tamaño de su grupo | Las rachas | Dice cuántos meses dura la racha de cada mes |
| `leer_oni(texto)` y `rachas(serie, umbral)` (propias) | Leen el archivo de NOAA y marcan las rachas de al menos cinco meses | `umbral=±0,5`, `minimo=5` | Es la definición de NOAA |

In [8]:
MES_CENTRAL = {"DJF": 1, "JFM": 2, "FMA": 3, "MAM": 4, "AMJ": 5, "MJJ": 6,
               "JJA": 7, "JAS": 8, "ASO": 9, "SON": 10, "OND": 11, "NDJ": 12}


def leer_oni(texto):
    """Convierte el archivo de NOAA (columnas SEAS, YR, TOTAL y ANOM) en una serie mensual de ONI."""
    d = pd.read_csv(StringIO(texto), sep=r"\s+")
    d["mes"] = pd.to_datetime(dict(year=d["YR"], month=d["SEAS"].map(MES_CENTRAL), day=1))
    return d.rename(columns={"SEAS": "trimestre", "ANOM": "oni"})[["mes", "trimestre", "oni"]]


def rachas(serie, umbral, minimo=5):
    """Marca los meses que pertenecen a una racha de al menos `minimo` valores seguidos que cumplen
    la condición: serie >= umbral si el umbral es positivo, serie <= umbral si es negativo."""
    cumple = serie >= umbral if umbral > 0 else serie <= umbral
    racha = (cumple != cumple.shift()).cumsum()            # número de cada racha
    return cumple & (cumple.groupby(racha).transform("size") >= minimo)


oni = leer_oni((CARPETA / "noaa_oni.ascii.txt").read_text(encoding="utf-8"))
oni["fase"] = np.select([rachas(oni["oni"], 0.5), rachas(oni["oni"], -0.5)], ["nino", "nina"], "neutro")
print(f"{len(oni)} meses de ONI, de {oni['mes'].min():%Y-%m} a {oni['mes'].max():%Y-%m}")

921 meses de ONI, de 1950-01 a 2026-09


In [9]:
# Un renglón por episodio de El Niño: inicio, fin, duración y ONI máximo
episodio = (oni["fase"] != oni["fase"].shift()).cumsum()
episodios = (oni[oni["fase"] == "nino"].groupby(episodio)
             .agg(inicio=("mes", "min"), fin=("mes", "max"), meses=("mes", "size"), oni_maximo=("oni", "max")))
episodios[episodios["inicio"] >= "1990-01-01"].reset_index(drop=True)

,inicio,fin,meses,oni_maximo
0,1991-06-01,1992-06-01,13,1.54
1,1994-10-01,1995-02-01,5,0.97
2,1997-05-01,1998-04-01,12,2.37
3,2002-07-01,2003-02-01,8,1.19
4,2004-08-01,2005-02-01,7,0.71
5,2006-09-01,2007-01-01,5,0.88
6,2009-07-01,2010-03-01,9,1.50
7,2014-10-01,2016-05-01,20,2.59
8,2018-09-01,2019-06-01,10,1.05
9,2019-10-01,2020-03-01,6,0.75


**Lectura.** Desde 1990 hubo doce episodios de El Niño. Los más fuertes fueron los de 1997-1998 (ONI máximo de 2,37), 2014-2016 (2,59, y el más largo: 20 meses) y 2023-2024 (1,99). El que empezó en mayo de 2026 ya llegó a 2,46 en agosto-octubre, cerca del máximo de 2015-2016, y seguía en curso a la fecha de corte. La sección 2.5 usa estas fases para describir el precio de bolsa.

### 2.4 Carga, limpieza y descripción

El paquete trae las tablas en CSV. `leer()` las abre y convierte las fechas, porque un CSV guarda todo como texto. Primero, la tabla mensual del sistema que publica XM: una fila por mes desde enero de 2000.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `pd.read_csv(ruta, parse_dates, dtype)` | Lee un CSV | `parse_dates` con las columnas de fecha; `dtype=str` en los códigos | Fechas como fechas; un código como "0001" no debe perder sus ceros |
| `DataFrame.info()` | Muestra columnas, tipos y valores no faltantes | — | Es la primera mirada a una tabla: tamaño, tipos y huecos |
| `DataFrame.isna().sum()` | Cuenta los faltantes de cada columna | — | Dice qué columnas tienen huecos y cuántos |
| `DataFrame.describe()` | Estadísticas descriptivas: media, desviación, mínimo, cuartiles y máximo | Las columnas numéricas que interesan | Resume cada variable en una sola tabla |
| `leer(nombre, fechas)` (propia) | Lee un CSV del paquete con sus fechas | El nombre del archivo | Evita repetir la ruta y la conversión de fechas |

In [10]:
def leer(nombre, fechas=(), **opciones):
    """Lee un CSV del paquete de datos y convierte a fecha las columnas de `fechas`."""
    return pd.read_csv(CARPETA / nombre, parse_dates=list(fechas), **opciones)


sistema = leer("xm_sistema_mensual.csv", ["mes"])
sistema.info()

<class 'pandas.DataFrame'>
RangeIndex: 323 entries, 0 to 322
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   mes                         323 non-null    datetime64[us]
 1   periodo_cxc                 323 non-null    str           
 2   demanda_real_gwh            322 non-null    float64       
 3   demanda_sin_gwh             323 non-null    float64       
 4   aportes_gwh                 322 non-null    float64       
 5   aportes_media_hist_gwh      322 non-null    float64       
 6   volumen_util_fin_mes_gwh    322 non-null    float64       
 7   capacidad_util_fin_mes_gwh  322 non-null    float64       
 8   importaciones_gwh           304 non-null    float64       
 9   exportaciones_gwh           292 non-null    float64       
 10  enficc_sistema_gwh          240 non-null    float64       
 11  precio_bolsa_cop_kwh        322 non-null    float64       
 12  preci

**Lectura.** 323 meses, de enero de 2000 a noviembre de 2026, y 20 columnas. Las fechas quedaron como fechas (`datetime64`), los indicadores de mes completo como booleanos y lo demás como números. Algunas columnas tienen menos datos porque empiezan después: la ENFICC del sistema, el precio de escasez y el umbral de escasez, en diciembre de 2006, con el Cargo por Confiabilidad; los precios de escasez superior e inferior, en 2025 (Resolución CREG 101 066 de 2024).

In [11]:
# Faltantes por columna (solo las que tienen alguno) y meses incompletos
faltantes = sistema.isna().sum()
print(faltantes[faltantes > 0].to_string())
print("\nMeses incompletos:", ", ".join(sistema.loc[~sistema["mes_completo"], "mes"].dt.strftime("%Y-%m")))

# Limpieza: solo meses completos, y nombres cortos para describir
completo = sistema[sistema["mes_completo"]].rename(columns={
    "demanda_sin_gwh": "demanda_GWh", "aportes_gwh": "aportes_GWh",
    "volumen_util_fin_mes_gwh": "embalse_GWh", "precio_bolsa_cop_kwh": "precio_COP_kWh"})
completo[["demanda_GWh", "aportes_GWh", "embalse_GWh", "precio_COP_kWh"]].describe().round(1)

demanda_real_gwh                1
aportes_gwh                     1
aportes_media_hist_gwh          1
volumen_util_fin_mes_gwh        1
capacidad_util_fin_mes_gwh      1
importaciones_gwh              19
exportaciones_gwh              31
enficc_sistema_gwh             83
precio_bolsa_cop_kwh            1
precio_escasez_cop_kwh         84
umbral_escasez_cop_kwh         84
precio_escasez_sup_cop_kwh    302
precio_escasez_inf_cop_kwh    302
dias_enficc                    83

Meses incompletos: 2026-11


,demanda_GWh,aportes_GWh,embalse_GWh,precio_COP_kWh
count,322.0,322.0,322.0,322.0
mean,5160.1,4672.0,10754.6,189.7
std,1104.9,1988.6,2332.1,203.7
min,3278.3,1122.5,2728.5,35.1
25%,4274.7,3246.0,9318.7,72.0
50%,5077.1,4580.5,11004.2,118.6
75%,5985.8,5817.2,12494.1,199.7
max,7701.5,12536.3,15533.0,1529.1


**Lectura.** Solo noviembre de 2026 está incompleto: la tabla se cortó el 4 de noviembre, así que se descarta. Los demás faltantes vienen de donde empieza cada serie; las importaciones y exportaciones no se usan en el análisis. En los 322 meses completos, la demanda promedió 5.160 GWh al mes, entre 3.278 y 7.702. Los aportes hídricos varían mucho más: su desviación estándar (1.989 GWh) es el 43% de su media (4.672). Y el precio de bolsa en pesos corrientes tiene una media de 190 COP/kWh pero una mediana de 119: unos pocos meses de escasez, con picos de hasta 1.529, jalan la media hacia arriba.

Después, la proyección de demanda de la UPME: una fila por mes, definición de demanda y escenario.

In [12]:
demanda_upme = leer("upme_demanda_mensual.csv", ["mes"])
print(f"{len(demanda_upme):,} filas, de {demanda_upme['mes'].min():%Y-%m} a {demanda_upme['mes'].max():%Y-%m}")
print("Definiciones:", ", ".join(demanda_upme["definicion"].unique()))
print("Escenarios:", ", ".join(demanda_upme["escenario"].unique()))
(demanda_upme.query("mes.dt.year == 2030")
             .pivot_table(index="definicion", columns="escenario", values="energia_gwh", aggfunc="sum")
             .round(0))

3,540 filas, de 2026-04 a 2040-12
Definiciones: SIN, SIN+GCE+ME, SIN+GCE+ME+GD_PNUMA, SIN+GCE+ME+GD_UPME
Escenarios: ic68_inf, ic68_sup, ic95_inf, ic95_sup, medio


escenario,ic68_inf,ic68_sup,ic95_inf,ic95_sup,medio
definicion,,,,,
SIN,90274.0,101050.0,85638.0,105733.0,95638.0
SIN+GCE+ME,100312.0,111087.0,95675.0,115771.0,105675.0
SIN+GCE+ME+GD_PNUMA,86522.0,97297.0,81885.0,101981.0,91885.0
SIN+GCE+ME+GD_UPME,95583.0,106358.0,90946.0,111042.0,100946.0


**Lectura.** La proyección va de abril de 2026 a diciembre de 2040, con cuatro definiciones de demanda y cinco escenarios: el medio y los bordes de las bandas de 68% y 95%. La definición importa: en 2030 el escenario medio va de 91.885 GWh (restando la generación distribuida con el supuesto del PNUMA) a 105.675 (sin restarla). El diagnóstico usa la demanda total, SIN + grandes consumidores especiales + movilidad eléctrica − generación distribuida (`SIN+GCE+ME+GD_UPME`): 100.946 GWh en 2030. En el pipeline fue la definición que mejor reprodujo la demanda observada de abril a octubre de 2026, y el equipo la adoptó el 5 de noviembre de 2026.

**Revisión de todas las tablas del paquete.** Antes de unir las fuentes, cada tabla pasa por la misma revisión: tamaño, tipos, celdas faltantes, llaves y filas repetidas, y valores extremos. La llave son las columnas que deben identificar cada fila una sola vez. Los valores extremos se cuentan con la regla de Tukey en las columnas numéricas continuas, fuera de tres rangos intercuartílicos: con 1,5, la regla usual, saldría marcada buena parte de un parque con muchas plantas pequeñas y pocas grandes. Las tablas de resultados del pipeline solo sirven para comprobar, y el ONI y el Catálogo ya se revisaron al leerlos en la sección 2.3.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `DataFrame.select_dtypes("number")` | Toma las columnas numéricas | — | Los valores extremos solo tienen sentido en números |
| `DataFrame.nunique()` | Cuenta los valores distintos de cada columna | Más de 10 | Deja por fuera los indicadores 0/1 y los parámetros con pocos valores, donde «extremo» no significa nada |
| `DataFrame.quantile(q)` | Cuantiles de cada columna | 0,25 y 0,75 | Dan el rango intercuartílico (RIC) |
| `DataFrame.duplicated(subset)` | Marca las filas cuya llave ya apareció antes | La llave de cada tabla | Una llave repetida duplicaría filas en las uniones |
| `revisar_tabla(nombre, llave, fechas)` (propia) | Revisa una tabla y resume la revisión en una fila | La llave y las fechas de cada tabla | La misma revisión para las doce tablas |

In [13]:
def revisar_tabla(nombre, llave, fechas=(), **opciones):
    """Revisa una tabla del paquete: tamaño, tipos, faltantes, llaves y filas repetidas, y valores extremos.

    llave: columnas que deben identificar cada fila una sola vez. Los valores extremos se cuentan en las columnas
    numéricas con más de 10 valores distintos, con la regla de Tukey: fuera de [Q1 − 3·RIC, Q3 + 3·RIC].
    """
    t = leer(nombre, fechas, **opciones)
    numericas = t.select_dtypes("number").loc[:, lambda d: d.nunique() > 10]
    q1, q3 = numericas.quantile(0.25), numericas.quantile(0.75)
    extremos = ((numericas < q1 - 3 * (q3 - q1)) | (numericas > q3 + 3 * (q3 - q1))).sum()
    return {"tabla": nombre, "filas": len(t), "columnas": t.shape[1],
            "tipos": ", ".join(f"{n} {tipo}" for tipo, n in t.dtypes.astype(str).value_counts().items()),
            "celdas faltantes": int(t.isna().sum().sum()), "llave": " + ".join(llave),
            "llaves repetidas": int(t.duplicated(llave).sum()), "filas repetidas": int(t.duplicated().sum()),
            "valores extremos": ", ".join(f"{c} ({n})" for c, n in extremos[extremos > 0].items()) or "ninguno"}

In [14]:
TABLAS = [("xm_sistema_mensual.csv", ["mes"], ["mes"], {}),
          ("upme_demanda_mensual.csv", ["mes", "definicion", "escenario"], ["mes"], {}),
          ("dane_ipc_mensual.csv", ["mes"], ["mes"], {}),
          ("xm_oef_recurso_2026_10.csv", ["fecha", "codigo"], ["fecha"], {}),
          ("xm_plantas_mes_referencia.csv", ["codigo"], [], {"dtype": {"codigo": str}}),
          ("proyectos_nuevos_oef.csv", ["planta"], [], {}),
          ("entradas_sin_subasta.csv", ["grupo"], [], {}),
          ("candidatos_costos.csv", ["tecnologia", "escenario_costos"], [], {}),
          ("homologacion_bases.csv", ["fuente", "id"], ["fecha", "fecha_alta"], {"dtype": {"id": str}}),
          ("homologacion_etiquetas.csv", ["tarea", "id_izq", "id_der"], [], {"dtype": {"id_izq": str, "id_der": str}}),
          ("retrasos_subastas.csv", ["codigo_oef", "mecanismo"], ["compromiso", "fecha_entrada"], {}),
          ("xm_hidraulica_periodos.csv", ["periodo"], [], {})]
revision = pd.DataFrame([revisar_tabla(nombre, llave, fechas, **opciones)
                         for nombre, llave, fechas, opciones in TABLAS]).set_index("tabla")
revision

,filas,columnas,tipos,celdas faltantes,llave,llaves repetidas,filas repetidas,valores extremos
tabla,,,,,,,,
xm_sistema_mensual.csv,323,20,"15 float64, 2 bool, 1 datetime64[us], 1 str, 1...",994,mes,0,0,"capacidad_util_fin_mes_gwh (5), importaciones_..."
upme_demanda_mensual.csv,3540,5,"2 str, 2 float64, 1 datetime64[us]",0,mes + definicion + escenario,0,0,ninguno
dane_ipc_mensual.csv,321,3,"1 datetime64[us], 1 float64, 1 str",0,mes,0,0,ninguno
xm_oef_recurso_2026_10.csv,17050,3,"1 datetime64[us], 1 str, 1 float64",0,fecha + codigo,0,0,valor (2296)
xm_plantas_mes_referencia.csv,631,8,"5 str, 3 float64",30,codigo,0,0,"capacidad_mw (55), enficc_gwh_dia (155), oef_g..."
proyectos_nuevos_oef.csv,41,13,"6 str, 5 float64, 1 bool, 1 int64",0,planta,0,0,"oef_2029_2030 (1), oef_gwh_dia (1)"
entradas_sin_subasta.csv,2,5,"2 str, 2 float64, 1 int64",0,grupo,0,0,ninguno
candidatos_costos.csv,18,23,"15 float64, 4 str, 4 int64",0,tecnologia + escenario_costos,0,0,ninguno
homologacion_bases.csv,2600,9,"5 str, 2 float64, 2 datetime64[us]",4598,fuente + id,0,0,"capacidad_mw (347), oef_gwh_dia (1)"


**Lectura.** Ninguna tabla tiene llaves ni filas repetidas, así que no hay duplicados que limpiar; además, cada unión del notebook usa `validate`, que se detendría si apareciera uno. Los tipos quedaron bien: fechas como fechas y códigos como texto. Los faltantes y los valores extremos sí piden decisiones:

| Tabla | Hallazgo | Decisión | Por qué |
|---|---|---|---|
| Sistema mensual de XM | Noviembre de 2026 trae solo 3 días, y por eso su ENFICC sale como valor extremo | Se descarta en los análisis mensuales | La tabla se cortó el 4 de noviembre: un mes incompleto no es comparable |
| Sistema mensual de XM | 994 celdas faltantes, casi todas de series que empiezan después de 2000: la ENFICC y el precio de escasez desde diciembre de 2006, y los precios de escasez superior e inferior desde 2025 | No se imputan | Son meses en que la variable no existía; imputarlos sería inventar datos |
| Sistema mensual de XM | 18 meses con precio de bolsa extremo: El Niño de 2015-2016 y de 2023-2024, septiembre a diciembre de 2024 y julio a octubre de 2026 | Se conservan | Son episodios reales de escasez, justo el fenómeno que motiva el trabajo |
| Sistema mensual de XM | Importaciones extremas en 38 meses y capacidad útil de los embalses baja en 5 meses de 2000-2002 | Se conservan | No entran en el análisis. Las importaciones son pequeñas la mayoría de los meses (mediana de 1,8 GWh), así que cualquier mes con importaciones importantes sale como extremo |
| OEF de octubre de 2026 (API de XM) | 2.296 valores extremos | Se filtran los ceros (sección 2.3) y se conservan los extremos | La API lista todas las plantas, la mayoría en cero; los valores altos son las plantas grandes |
| Plantas del mes de referencia | 30 celdas faltantes: 13 plantas registradas sin pruebas y una térmica sin capacidad, y 4 unidades fuera del listado de XM. Capacidad y ENFICC muy asimétricas | Las que no operan no suman (sección 3.3); los extremos se conservan | Sin pruebas no hay generación ni firmeza; el parque real tiene muchas plantas pequeñas y pocas grandes |
| Bases de la homologación | 4.598 celdas faltantes en capacidad, OEF, agente y fechas | Se conservan | Cada base trae atributos distintos; la revisión de atributos de la sección 3.1 da «no aplica» cuando falta el dato |
| Retrasos de las subastas | 91 faltantes en la fecha de entrada y el retraso | Se conservan | Son los proyectos que no han entrado: el faltante es la información |
| Hidráulica por período | 2012-2013 promedia 26,8 plantas con ENFICC, frente a 53 a 79 en los demás períodos | Se conserva | Generación y ENFICC se comparan para las mismas plantas, y su relación (1,22) es igual a la del período siguiente |

### 2.5 Unir las fuentes

Tres uniones preparan el análisis. Cada una usa `merge` con `validate`, que falla si la llave no es única donde debe serlo, y comprueba cuántas filas se pierden.

**a) El precio de bolsa en pesos constantes y la hidrología.** El precio de XM está en pesos corrientes: para comparar 2000 con 2026 hay que llevarlo a pesos de un mismo mes (septiembre de 2026) con el IPC del DANE. Luego cada mes se clasifica por su hidrología: aportes por debajo del 80% de su media histórica (seca), entre 80% y 110% (normal) o por encima (húmeda). Y se le pega la fase del ONI de la sección 2.3.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `DataFrame.merge(otra, on, how, validate)` | Une dos tablas por una llave | `on="mes"`, `how="inner"` y `validate="one_to_one"` | Un mes del sistema tiene un solo IPC y una sola fase; si no, el error salta aquí |
| `pd.cut(serie, bins, labels)` | Clasifica valores en intervalos | Cortes en 80% y 110% | Son los umbrales de condición hidrológica que usa XM |
| `groupby(...).agg(...)` | Resume por grupo | Media y número de meses por condición y por fase | Una fila por categoría |
| `precio_real(sistema, ipc, base)` (propia) | Deflacta el precio y clasifica la hidrología | Base: septiembre de 2026 | El último mes con IPC publicado a la fecha de corte |

In [15]:
ipc = leer("dane_ipc_mensual.csv", ["mes"])


def precio_real(sistema, ipc, base="2026-09-01"):
    """Precio de bolsa en pesos constantes de `base` y condición hidrológica de cada mes completo."""
    s = sistema[sistema["mes_completo"] & sistema["precio_bolsa_cop_kwh"].notna()]
    d = s.merge(ipc[["mes", "ipc"]], on="mes", how="inner", validate="one_to_one")
    print(f"Meses con precio: {len(s)}; con IPC: {len(d)} (el DANE publica el IPC con un mes de rezago)")
    deflactor = ipc.set_index("mes").loc[base, "ipc"] / d["ipc"]
    aportes_pct = 100 * d["aportes_gwh"] / d["aportes_media_hist_gwh"]
    return pd.DataFrame({
        "mes": d["mes"], "precio_real": d["precio_bolsa_cop_kwh"] * deflactor,
        "umbral_escasez_real": d["umbral_escasez_cop_kwh"] * deflactor,
        "condicion": pd.cut(aportes_pct, [-np.inf, 80, 110, np.inf], labels=["Seca", "Normal", "Húmeda"]).astype(str)})


precios = precio_real(sistema, ipc).merge(oni[["mes", "fase"]], on="mes", how="inner", validate="one_to_one")
precios["sobre_umbral"] = precios["precio_real"] > precios["umbral_escasez_real"]   # activa las OEF
pd.concat([precios.groupby(col).agg(meses=("precio_real", "size"), precio_medio=("precio_real", "mean"),
                                    meses_sobre_umbral=("sobre_umbral", "sum"))
           for col in ["condicion", "fase"]], keys=["Hidrología", "Fase del ONI"]).round(0)

Meses con precio: 322; con IPC: 321 (el DANE publica el IPC con un mes de rezago)


meses  precio_medio  meses_sobre_umbral
Hidrología   Húmeda     92         207.0                   0
             Normal    146         264.0                   2
             Seca       83         528.0                   8
Fase del ONI neutro    159         276.0                   2
             nina       81         226.0                   0
             nino       81         485.0                   8

**Lectura.** De los 322 meses con precio, 321 tienen IPC: el de octubre de 2026 todavía no estaba publicado, y la unión lo deja por fuera sin duplicar ni perder nada más. En pesos de septiembre de 2026, el precio medio de los meses secos (528 COP/kWh) duplica el de los normales (264) y casi triplica el de los húmedos (207). Con el ONI pasa algo parecido: 485 en El Niño frente a 226 en La Niña. El precio superó el umbral de escasez, el que activa las OEF, en 10 meses: 8 secos y 2 normales, y 8 de ellos en El Niño. Es asociación, no causalidad: la hidrología mueve a la vez los aportes, el embalse y el precio.

**b) De meses calendario a períodos del cargo.** La UPME proyecta por mes calendario; el Cargo por Confiabilidad funciona por períodos de diciembre a noviembre. Restar una cosa de la otra sin alinearlas mezcla un año con otro. `periodo_cxc()` le asigna a cada mes su período (diciembre de 2026 abre el período 2026-2027), y la suma por período deja la demanda en la misma unidad que la energía firme. Los dos escenarios de demanda son el medio y el borde superior de la banda de 68% de la UPME.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `Series.dt.year` y `dt.month` | Extraen el año y el mes de una fecha | — | El período depende de si el mes es diciembre |
| `Series.dt.days_in_month` | Días de cada mes | — | La energía firme se mide en GWh por día; hay que saber cuántos días tiene cada período |
| `periodo_cxc(meses)` (propia) | Período del cargo de cada mes | Fechas | Un diciembre pertenece al período que empieza ese año |

In [16]:
PERIODOS = ["2026-2027", "2027-2028", "2028-2029", "2029-2030"]
DEFINICION = "SIN+GCE+ME+GD_UPME"   # demanda total: SIN + grandes consumidores + movilidad eléctrica − GD
ESCENARIOS_DEMANDA = {"media": "medio", "alta": "ic68_sup"}


def periodo_cxc(meses):
    """Período del Cargo por Confiabilidad (diciembre a noviembre) de cada mes: dic-2026 → "2026-2027"."""
    inicio = meses.dt.year - (meses.dt.month < 12)
    return inicio.astype(str) + "-" + (inicio + 1).astype(str)


d = demanda_upme[(demanda_upme["definicion"] == DEFINICION)
                 & demanda_upme["escenario"].isin(ESCENARIOS_DEMANDA.values())].copy()
d["periodo"], d["dias"] = periodo_cxc(d["mes"]), d["mes"].dt.days_in_month
demanda = (d[d["periodo"].isin(PERIODOS)]
           .groupby(["escenario", "periodo"], as_index=False)
           .agg(demanda_gwh=("energia_gwh", "sum"), dias=("dias", "sum"), meses=("mes", "size")))
demanda["escenario_demanda"] = demanda["escenario"].map({v: k for k, v in ESCENARIOS_DEMANDA.items()})
calendario = d[d["mes"].dt.year == 2027].groupby("escenario")["energia_gwh"].sum()["medio"]
periodo_1 = demanda.set_index(["escenario", "periodo"]).loc[("medio", "2026-2027"), "demanda_gwh"]
print(f"Escenario medio: año calendario 2027, {calendario:,.0f} GWh; período 2026-2027, {periodo_1:,.0f} GWh")
demanda.pivot(index="periodo", columns="escenario_demanda", values="demanda_gwh").round(0)

Escenario medio: año calendario 2027, 93,571 GWh; período 2026-2027, 93,508 GWh

escenario_demanda,alta,media
periodo,,
2026-2027,96231.0,93508.0
2027-2028,100007.0,96284.0
2028-2029,103273.0,98700.0
2029-2030,106107.0,100759.0


**Lectura.** El período 2026-2027 suma 93.508 GWh en el escenario medio, y el año calendario 2027, 93.571. La diferencia es pequeña porque el período cambia un diciembre por otro; lo que importa es no mezclar los dos calendarios al restar la energía firme, que se asigna por período. La demanda media pasa de 93.508 GWh en 2026-2027 a 100.759 en 2029-2030 (+7,8%), y la alta, de 96.231 a 106.107 (+10,3%). Son las demandas del balance de la sección 3.3.

**c) Lo observado frente a lo proyectado.** Para saber si el escenario medio es razonable, se compara con la demanda que XM ya midió (`DemaSIN`) en los meses en que hay proyección y dato a la vez.

In [17]:
medio = demanda_upme.query("definicion == @DEFINICION and escenario == 'medio'")[["mes", "energia_gwh"]]
observado = sistema.loc[sistema["mes_completo"], ["mes", "demanda_sin_gwh"]]
control = observado.merge(medio, on="mes", how="inner", validate="one_to_one")
control["desvio_%"] = 100 * (control["demanda_sin_gwh"] / control["energia_gwh"] - 1)
control.set_index(control["mes"].dt.strftime("%b-%Y")).drop(columns="mes").round(1)

,demanda_sin_gwh,energia_gwh,desvio_%
mes,,,
Apr-2026,7217.2,7169.1,0.7
May-2026,7664.6,7725.0,-0.8
Jun-2026,7298.0,7425.3,-1.7
Jul-2026,7701.5,7837.0,-1.7
Aug-2026,7696.6,7874.8,-2.3
Sep-2026,7531.9,7766.5,-3.0
Oct-2026,7529.5,8010.8,-6.0


**Lectura.** De abril a septiembre de 2026, la demanda observada quedó entre 0,7% por encima y 3,0% por debajo del escenario medio de la demanda total; en octubre, 6,0% por debajo. El escenario medio, entonces, no es optimista: la UPME suma grandes consumidores nuevos y movilidad eléctrica que la demanda observada todavía no muestra. Por eso la demanda alta se trata como un escenario de estrés, no como la base.

**Lo que sigue.** Los datos quedaron revisados y listos: la demanda de la UPME por período del cargo, el parque y los proyectos de XM, los costos y las bases que hay que emparejar. La sección 3 los usa para construir el método, paso a paso.

<a id="metodologia" name="metodologia"></a>
## 3. Metodología y desarrollo

Con los datos de la sección 2, cuatro pasos responden la pregunta. Cada uno explica primero la idea y después la programa:

1. **Homologación (3.1):** emparejar plantas y proyectos entre bases que no comparten código, para medir cuánto se retrasan los proyectos de las subastas.
2. **Costos (3.2):** convertir la inversión y la operación de cada tecnología en un costo anual por MWh de energía firme.
3. **Balance de energía firme (3.3):** cuánta energía firme falta, y desde cuándo, en tres escenarios.
4. **Expansión de mínimo costo (3.4 y 3.5):** con qué bloques se cierra esa brecha al menor costo, y qué cambia si cambian los supuestos.

Al final de 3.3, 3.4 y 3.5, `comprobar()` compara los resultados con los del pipeline completo del proyecto, que los calculó con su propio código. Si alguna cifra no coincide, el notebook se detiene.

In [18]:
def comprobar(calculado, esperado, que, tolerancia=0.5):
    """Compara dos tablas (o series) numéricas alineadas por su índice.

    Se detiene si falta alguna fila o si alguna diferencia absoluta pasa de `tolerancia`
    (en las unidades de la tabla); si no, informa la diferencia máxima.
    """
    calculado, esperado = calculado.align(esperado, join="outer")
    diferencia = (calculado - esperado).abs()
    maxima = float(np.nanmax(diferencia.to_numpy()))
    if diferencia.isna().to_numpy().any() or maxima > tolerancia:
        raise AssertionError(f"{que}: no coincide con el pipeline (diferencia máxima {maxima:,.3f})")
    print(f"✓ {que}: coincide con el pipeline (diferencia máxima {maxima:.2g})")

### 3.1 Homologación de plantas y proyectos con RapidFuzz

La OEF, la ENFICC y la capacidad de las plantas que ya operan traen el código de XM y se unen al listado de forma exacta. Pero hay tres bases sin un código común:

| Base | Qué se sabe de cada registro |
|---|---|
| **OEF** (bitácora de SIMEM): proyectos `PROG#####` con obligaciones asignadas | Solo la razón social de la empresa, la OEF y las fechas |
| **UPME** (registro de proyectos de generación, septiembre de 2026) | Nombre, tecnología, MW, municipio y fecha prevista |
| **XM** (listado de recursos de generación) | Nombre, tecnología, agente, estado y fecha de entrada |

Sin emparejarlas no se puede saber si un proyecto con OEF ya entró en operación, ni cuándo. El método tiene cuatro pasos:
1. **Normalizar** los nombres, porque "PARQUE SOLAR LA LOMA" y "La Loma Solar" son la misma planta escrita distinto.
2. **Puntuar** cada par con dos medidas de RapidFuzz, solo dentro del mismo bloque de tecnología, y quedarse con los tres mejores candidatos de cada registro.
3. **Revisar atributos:** etapa, capacidad, energía firme posible y fechas. Un par con un atributo incompatible se descarta por alto que sea su puntaje.
4. **Escoger el umbral** con 205 pares revisados a mano: el que da la mejor combinación de precisión y exhaustividad.

**Normalizar.** `normalizar()` pasa a mayúsculas, quita tildes, razones sociales (S.A.S., E.S.P.), conectores, palabras genéricas (PARQUE, SOLAR, PCH...) y capacidades ("200 MW"), y convierte los números romanos en arábigos.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `unicodedata.normalize("NFKD", texto)` | Separa cada letra de su tilde | `"NFKD"` y después `.encode("ascii", "ignore")` | Así "Á" queda en "A" y "Ñ" en "N" |
| `re.sub(patron, reemplazo, texto)` | Reemplaza lo que cumple una expresión regular | Capacidades (`\d+ MW`), signos y la cola "Y SU CONEXIÓN..." | Lo que no identifica a la planta no debe sumar puntaje |
| `re.findall(r"\b\d+\b", texto)` | Encuentra todos los números sueltos | El nombre normalizado | Los números distinguen etapas: "LLANOS 8" no es "LLANOS 6" |
| `normalizar(nombre)` y `numeros(nombre)` (propias) | Dejan solo lo que identifica a la planta, y sus números | El nombre tal como viene | Son el primer paso de la homologación |

In [19]:
FORMAS_JURIDICAS = {"SAS", "SA", "ESP", "SCA", "CIA", "LTDA", "ZOMAC", "SC", "BIC", "EU"}
CONECTORES = {"DE", "DEL", "LA", "EL", "LOS", "LAS", "Y", "A", "AL", "EN", "SU"}
GENERICAS = {"PARQUE", "PLANTA", "CENTRAL", "PROYECTO", "SOLAR", "FOTOVOLTAICO", "FOTOVOLTAICA", "PHOTOVOLTAIC",
             "FV", "PV", "EOLICO", "EOLICA", "PCH", "PEQUENA", "HIDROELECTRICA", "HIDROELECTRICO", "TERMICA",
             "TERMICO", "TERMOELECTRICA", "GENERACION", "GENERADORA", "ENERGIA", "ENERGIAS", "RENOVABLES",
             "AUTOG", "AGPE", "MW"}
DESCARTAR = FORMAS_JURIDICAS | CONECTORES | GENERICAS
UNIDADES = ["", "I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX"]
ROMANOS = {"X" * (n // 10) + UNIDADES[n % 10]: n for n in range(1, 40)}   # "XIV" → 14


def sin_tildes(texto):
    """Quita tildes y diacríticos y pasa a mayúsculas: "Pequeña Central" → "PEQUENA CENTRAL"."""
    return unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode().upper()

In [20]:
def normalizar(nombre):
    """Deja solo las palabras que identifican la planta: "Parque Solar Colombia XIV S.A.S." → "COLOMBIA 14"."""
    if not isinstance(nombre, str) or not nombre.strip():
        return ""
    s = re.sub(r"\bY SU CONEXION\b.*$", " ", sin_tildes(nombre))        # "... Y SU CONEXIÓN A LA S/E ..."
    s = re.sub(r"\d+(?:[.,]\d+)?\s*(?:MWP?|MWAC|KWP?|KV)\b", " ", s)    # capacidad o tensión
    s = re.sub(r"[^A-Z0-9]+", " ", s.replace(".", ""))                  # S.A.S. → SAS; signos → espacio
    fichas = []
    for f in s.split():
        if f in DESCARTAR:
            continue
        if f.startswith("TERMO") and len(f) > 7:                         # TERMOCAIMAN → CAIMAN
            f = f[5:]
        fichas.append(str(ROMANOS[f]) if f in ROMANOS else f)
    return " ".join(fichas) or re.sub(r"[^A-Z0-9]+", " ", sin_tildes(nombre)).strip()


def numeros(nombre_norm):
    """Números sueltos de un nombre normalizado: etapas ("LLANOS 8") o unidades ("RUSA 2")."""
    return frozenset(re.findall(r"\b\d+\b", nombre_norm))

In [21]:
ejemplos = ["PARQUE DE GENERACIÓN FOTOVOLTAICO LA PRIMAVERA DE 57 MW", "Parque Solar Colombia XIV S.A.S. - E.S.P.",
            "BOSQUES SOLARES DE LOS LLANOS 8 Y SU CONEXIÓN A LA SUBESTACIÓN OCOA 115 KV",
            "PEQUEÑA CENTRAL HIDROELÉCTRICA AKUA 1", "PLANTA TÉRMICA EL TESORITO II 200 MW"]
pd.DataFrame({"nombre": ejemplos, "normalizado": [normalizar(n) for n in ejemplos],
              "números": [sorted(numeros(normalizar(n))) for n in ejemplos]})

,nombre,normalizado,números
0,PARQUE DE GENERACIÓN FOTOVOLTAICO LA PRIMAVERA...,PRIMAVERA,[]
1,Parque Solar Colombia XIV S.A.S. - E.S.P.,COLOMBIA 14,[14]
2,BOSQUES SOLARES DE LOS LLANOS 8 Y SU CONEXIÓN ...,BOSQUES SOLARES LLANOS 8,[8]
3,PEQUEÑA CENTRAL HIDROELÉCTRICA AKUA 1,AKUA 1,[1]
4,PLANTA TÉRMICA EL TESORITO II 200 MW,TESORITO 2,[2]


**Lectura.** De cada nombre queda solo lo que lo identifica: "PRIMAVERA", "COLOMBIA 14", "BOSQUES SOLARES LLANOS 8", "AKUA 1" y "TESORITO 2". "SOLARES" se conserva a propósito: es parte de la marca "Bosques Solares de los Llanos". Los números que quedan (14, 8, 1 y 2) distinguen etapas y unidades.

**Puntuar.** RapidFuzz da un puntaje de 0 a 100 a cada par de nombres. Se usan dos medidas:
- `token_set_ratio` compara los conjuntos de palabras: da 100 si las palabras de un nombre están contenidas en el otro, sin importar el orden. Ayuda ("LOMA" frente a "LOMA") y engaña: "LOMA 2", una segunda etapa, también saca 100.
- `WRatio` combina varias medidas (cadena completa, parcial y por palabras) y se queda con la mejor, ponderada.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `process.cdist(consultas, opciones, scorer, workers)` | Calcula el puntaje de todos los pares entre dos listas de nombres | `scorer=fuzz.token_set_ratio` (y otra vez con `fuzz.WRatio`) y `workers=-1` | `workers=-1` usa todos los núcleos; la matriz completa evita un ciclo en Python sobre cada par |
| `np.argsort(-fila, kind="stable")` | Ordena de mayor a menor y devuelve las posiciones | `kind="stable"` | Ante un empate, conserva el orden original: el resultado no cambia entre corridas |
| `candidatos(izq, der, top_k)` (propia) | Los `top_k` mejores candidatos de cada registro, dentro del mismo bloque de tecnología | `top_k=3` | Basta para que la pareja correcta casi siempre esté entre los candidatos |

In [22]:
def candidatos(izq, der, top_k=3):
    """Los `top_k` mejores candidatos de `der` para cada registro de `izq`, dentro del mismo bloque de tecnología.

    Ordena por token_set_ratio y desempata con WRatio. Devuelve una fila por par, con las columnas de
    cada lado y los sufijos _izq y _der.
    """
    tsr = process.cdist(izq["nombre_norm"], der["nombre_norm"], scorer=fuzz.token_set_ratio, workers=-1)
    wr = process.cdist(izq["nombre_norm"], der["nombre_norm"], scorer=fuzz.WRatio, workers=-1)
    ti, td = izq["tecnologia"].to_numpy(), der["tecnologia"].to_numpy()
    bloque = (ti[:, None] == td[None, :]) | (ti == "desconocida")[:, None] | (td == "desconocida")[None, :]
    clave = np.where(bloque, tsr * 1000 + wr, -1.0)                  # fuera del bloque, nunca es candidato
    filas = [(i, j, tsr[i, j], wr[i, j]) for i in range(len(izq))
             for j in np.argsort(-clave[i], kind="stable")[:top_k] if clave[i, j] >= 0]
    pares = pd.DataFrame(filas, columns=["i", "j", "token_set_ratio", "wratio"])
    a = izq.add_suffix("_izq").iloc[pares["i"]].reset_index(drop=True)
    b = der.add_suffix("_der").iloc[pares["j"]].reset_index(drop=True)
    return pd.concat([pares.drop(columns=["i", "j"]), a, b], axis=1)

**Revisar atributos.** Cada par pasa cinco pruebas. Cada una da verdadero, falso o "no aplica" (faltante) cuando no hay con qué comparar:

| Prueba | Regla | Ejemplo que descarta |
|---|---|---|
| `numeros_ok` | Las etapas coinciden; un número de un solo lado, distinto de 1, es otra etapa | "TESORITO 2" frente a "TESORITO" |
| `capacidad_ok` | Capacidades iguales con ±10% o ±2 MW | Un proyecto de 150 MW frente a un parque de 2 MW |
| `factor_ok` | OEF / capacidad dentro de un rango posible para la tecnología | 0,4 GWh-día de OEF en un autogenerador de 100 kW |
| `fecha_ok` | La planta de XM no operaba más de un año antes de que apareciera el proyecto | Una planta que opera desde 2015 no es un proyecto de 2019 |
| `codigo_ok` | La planta no tenía ya OEF con su propio código cuando se le asignó al proyecto | Plantas con OEF propia desde antes |

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `np.where(condicion, si, no)` | Elige elemento a elemento entre dos valores | Faltante (`np.nan`) cuando un lado no tiene el dato | "No aplica" no es lo mismo que "falla": no debe descartar el par |
| `np.maximum(a, b)` | Máximo elemento a elemento | 10% de la capacidad y 2 MW | Una tolerancia relativa sola sería demasiado estrecha en las plantas pequeñas |
| `atributos(c)` (propia) | Añade las cinco pruebas y la columna `conflicto` | La tabla de candidatos | Un conflicto descarta el par por alto que sea su puntaje |

In [23]:
FACTOR_ADMISIBLE = {"solar": (0.01, 0.35), "eolica": (0.02, 0.60), "hidraulica": (0.02, 1.0),
                    "termica": (0.20, 1.05), "biomasa": (0.05, 1.0), "desconocida": (0.01, 1.05)}


def numeros_ok(a, b):
    """True si los números coinciden, False si son otra etapa y NaN si no hay evidencia (solo un "1" o nada)."""
    if a and b:
        return a == b
    solo = a or b
    return np.nan if not solo or solo == {"1"} else False

In [24]:
def atributos(c):
    """Añade a cada par las cinco pruebas (True, False o NaN si no aplica) y si tiene algún conflicto."""
    c = c.copy()
    c["numeros_ok"] = [numeros_ok(numeros(a), numeros(b)) for a, b in zip(c["nombre_norm_izq"], c["nombre_norm_der"])]
    ci, cd = c["capacidad_mw_izq"], c["capacidad_mw_der"]
    tolerancia = np.maximum(0.10 * np.maximum(ci, cd), 2.0)
    c["capacidad_ok"] = np.where(ci.notna() & cd.notna(), (ci - cd).abs() <= tolerancia, np.nan)
    oef = c["oef_gwh_dia_izq"].fillna(c["oef_gwh_dia_der"])                 # OEF del lado que la tiene
    factor = oef * 1e3 / 24 / np.where(c["oef_gwh_dia_izq"].notna(), cd, ci)  # MW medios / capacidad del otro
    tec = np.where(c["tecnologia_izq"] == "desconocida", c["tecnologia_der"], c["tecnologia_izq"])
    limites = np.array([FACTOR_ADMISIBLE[t] for t in tec])
    c["factor_ok"] = np.where(factor.notna(), (factor >= limites[:, 0]) & (factor <= limites[:, 1]), np.nan)
    c["fecha_ok"] = c["codigo_ok"] = np.nan
    if (c["fuente_der"] == "xm").all():
        alta = c["fecha_alta_izq"]                                           # cuándo apareció el proyecto
        c["fecha_ok"] = np.where(c["fecha_der"].notna() & alta.notna(), c["fecha_der"] >= alta - pd.Timedelta(days=365), np.nan)
        if (c["fuente_izq"] == "oef").all():
            c["codigo_ok"] = np.where(c["fecha_alta_der"].notna() & alta.notna(), c["fecha_alta_der"] > alta, np.nan)
    pruebas = ["numeros_ok", "capacidad_ok", "factor_ok", "fecha_ok", "codigo_ok"]
    c["conflicto"] = (c[pruebas] == False).any(axis=1)  # noqa: E712  (NaN no es conflicto)
    return c

Con las tres funciones se arman los candidatos de las tres tareas: OEF→XM (¿el proyecto ya está en XM?), OEF→UPME (¿está en el registro?) y UPME→XM (¿el proyecto del registro ya opera?). En OEF→XM solo entran los códigos `PROG`: un elemento con código de planta no puede ser otra planta.

In [25]:
bases = leer("homologacion_bases.csv", ["fecha", "fecha_alta"], dtype={"id": str})
bases["nombre_norm"] = bases["nombre"].map(normalizar)
print(bases["fuente"].value_counts().to_string())

TAREAS = {"oef_xm": ("oef", "xm"), "oef_upme": ("oef", "upme"), "upme_xm": ("upme", "xm")}
partes = []
for tarea, (a, b) in TAREAS.items():
    izq = bases[bases["fuente"] == a]
    if tarea == "oef_xm":
        izq = izq[izq["id"].str.startswith("PROG")]
    der = bases[bases["fuente"] == b]
    partes.append(atributos(candidatos(izq.reset_index(drop=True), der.reset_index(drop=True))).assign(tarea=tarea))
cand = pd.concat(partes, ignore_index=True)
print(f"\n{len(cand)} pares candidatos; {cand['conflicto'].sum()} con algún conflicto de atributos")
cand.groupby("tarea").agg(pares=("conflicto", "size"), con_conflicto=("conflicto", "sum"))

fuente
xm      2499
upme      52
oef       49



429 pares candidatos; 319 con algún conflicto de atributos


,pares,con_conflicto
tarea,,
oef_upme,147,57
oef_xm,126,110
upme_xm,156,152


**Lectura.** Las bases tienen 49 registros con OEF, 52 proyectos de la UPME y 2.499 plantas de XM. Con los tres mejores candidatos de cada registro quedan 429 pares, y 319 tienen algún conflicto de atributos: la mayoría de los candidatos por nombre no son la misma planta. En UPME→XM chocan 152 de 156: los atributos descartan casi todos los candidatos que propone el nombre.

**Escoger el umbral.** Un par se acepta si es el mejor candidato de su registro y su puntaje alcanza el umbral. Para escoger el umbral hace falta saber cuáles pares son de verdad la misma planta: el paquete trae 205 pares revisados uno a uno con su evidencia (agente, capacidad, etapa y fechas), 17 de ellos parejas verdaderas. Para cada umbral se mide:
- **Precisión:** de los pares aceptados, cuántos son verdaderos.
- **Exhaustividad:** de las parejas verdaderas, cuántas se encuentran.
- **F1:** la media armónica de las dos.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `Series.where(condicion)` | Deja el valor donde se cumple la condición y faltante donde no | `~conflicto` | La regla "con atributos" no deja competir a un par con conflicto |
| `groupby(...).transform("max")` | Le da a cada fila el máximo de su grupo | El registro de la izquierda | Para saber si un par es el mejor candidato de su registro |
| `merge(..., how="right", validate="one_to_one")` | Une las predicciones con las etiquetas, conservando todas las etiquetas | La llave tarea, id_izq, id_der | Una pareja verdadera que no quedó entre los candidatos cuenta como no encontrada |
| `curva(c, etiquetas)` y `elegir_umbral(cur, precision_min)` (propias) | Miden precisión, exhaustividad y F1 por umbral, y escogen el umbral | Umbrales de 50 a 100, de 2 en 2; `precision_min=0.9` | Entre los umbrales con al menos 90% de precisión, el de mayor F1 |

In [26]:
def curva(c, etiquetas, medidas=("token_set_ratio", "wratio"), umbrales=range(50, 101, 2)):
    """Precisión, exhaustividad y F1 de cada medida y umbral contra los pares revisados, sin y con atributos."""
    llave = ["tarea", "id_izq", "id_der"]
    filas = []
    for medida in medidas:
        for regla, ok in [("solo puntaje", pd.Series(True, index=c.index)), ("con atributos", ~c["conflicto"])]:
            s = c[medida].where(ok)
            mejor = s == s.groupby([c["tarea"], c["id_izq"]]).transform("max")
            base = c[llave].assign(s=s, mejor=mejor).merge(etiquetas, on=llave, how="right", validate="one_to_one")
            for u in umbrales:
                pred = base["mejor"].eq(True) & (base["s"] >= u)
                vp = int((pred & (base["es_misma"] == 1)).sum())
                fp = int((pred & (base["es_misma"] == 0)).sum())
                fn = int((~pred & (base["es_misma"] == 1)).sum())
                prec = vp / (vp + fp) if vp + fp else np.nan
                exh = vp / (vp + fn) if vp + fn else np.nan
                f1 = 2 * prec * exh / (prec + exh) if prec and exh else 0.0
                filas.append((medida, regla, u, vp, fp, fn, prec, exh, f1))
    return pd.DataFrame(filas, columns=["medida", "regla", "umbral", "vp", "fp", "fn", "precision", "exhaustividad", "f1"])


def elegir_umbral(cur, precision_min=0.9):
    """Regla con atributos: entre los umbrales con precisión >= precision_min, el de mayor F1. Si hay empate,
    la medida que lo sostiene en más umbrales y, dentro de ella, el umbral más bajo."""
    ok = cur[(cur["regla"] == "con atributos") & (cur["precision"] >= precision_min)]
    top = ok[np.isclose(ok["f1"], ok["f1"].max())]
    estable = top.groupby("medida")["umbral"].count().sort_values(ascending=False).index[0]
    return top[top["medida"] == estable].sort_values("umbral").iloc[0]

In [27]:
etiquetas = leer("homologacion_etiquetas.csv", dtype={"id_izq": str, "id_der": str})
print(f"{len(etiquetas)} pares revisados, {int(etiquetas['es_misma'].sum())} parejas verdaderas")
cur = curva(cand, etiquetas)
elegido = elegir_umbral(cur)
print(f"Umbral elegido: {elegido['medida']} >= {elegido['umbral']:.0f} (precisión {elegido['precision']:.2f}, "
      f"exhaustividad {elegido['exhaustividad']:.2f}, F1 {elegido['f1']:.2f})")
assert (elegido["medida"], elegido["umbral"]) == ("token_set_ratio", 86), "no es el umbral que usa el pipeline"
cur[cur["umbral"].isin([70, 86, 100])].pivot_table(index=["medida", "regla"], columns="umbral",
                                                   values=["precision", "exhaustividad"]).round(2)

205 pares revisados, 17 parejas verdaderas
Umbral elegido: token_set_ratio >= 86 (precisión 0.92, exhaustividad 0.71, F1 0.80)


exhaustividad             precision            
umbral                                  70    86    100       70    86    100
medida          regla                                                        
token_set_ratio con atributos          0.71  0.71  0.71      0.63  0.92  0.92
                solo puntaje           0.71  0.71  0.71      0.14  0.19  0.23
wratio          con atributos          0.71  0.71  0.53      0.60  0.92  1.00
                solo puntaje           0.71  0.71  0.53      0.13  0.19  0.75

La figura muestra las curvas de la medida elegida, sin y con la revisión de atributos.

In [28]:
d = cur[cur["medida"] == elegido["medida"]].melt(id_vars=["regla", "umbral"], value_vars=["precision", "exhaustividad", "f1"],
                                                   var_name="métrica", value_name="valor")
fig = px.line(d, x="umbral", y="valor", color="métrica", facet_col="regla", markers=True,
              category_orders={"regla": ["solo puntaje", "con atributos"]},
              labels={"umbral": "Umbral de puntaje (0-100)", "valor": "Proporción", "métrica": "Métrica"},
              title=f"Homologación: precisión, exhaustividad y F1 según el umbral ({elegido['medida']})")
fig.add_vline(x=elegido["umbral"], line_dash="dash", line_color="gray", col=2,
              annotation_text=f"umbral elegido: {elegido['umbral']:.0f}")
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("regla=", "Regla: ")))
fig.update_layout(height=420, legend=dict(orientation="h", y=-0.25))
fig.show()

**Lectura.** 
- **Solo con el puntaje, la precisión es baja.** Con `token_set_ratio` no pasa de 0,23; con `WRatio` llega a 0,75 en el umbral 100, pero encuentra apenas 9 de las 17 parejas (exhaustividad de 0,53). La causa son los cientos de autogeneradores pequeños de XM, cuyos nombres comparten palabras con las sociedades de los proyectos.
- **Con los atributos, la precisión sube a 0,92 desde el umbral 86.** La exhaustividad se queda en 0,71 en todos los umbrales, porque cinco parejas verdaderas no se parecen por nombre. En el pipeline, una regla adicional (la única planta del mismo agente) encuentra tres de ellas.
- **El umbral elegido es `token_set_ratio` ≥ 86,** con F1 de 0,80. Entre 70 y 86 queda la zona gris, que revisa una persona: son los dos umbrales de Fellegi y Sunter (sección 1).

**De la homologación a los retrasos.** Con las plantas emparejadas se puede comparar el compromiso de entrada de cada proyecto (el inicio de la OEF que se le asignó como planta nueva) con su entrada real. La subasta de 2019 es la única con plantas nuevas cuyo compromiso ya venció para todas: es la evidencia para modelar los retrasos de los proyectos que faltan.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `Series.str.contains(patron)` | Marca los textos que contienen un patrón | `"TERCERA SUBASTA"` y `"Nuevas"` | La subasta de 2019, plantas nuevas |
| `np.select(condiciones, valores, default)` | Elige un valor según la primera condición que se cumpla | Entró tarde (más de un mes), entró a tiempo, no ha entrado | Tres grupos excluyentes |
| `np.ceil(x)` | Redondea hacia arriba | El retraso medio en meses / 12 | Un retraso de 21 meses ocupa dos períodos del cargo |

In [29]:
ret = leer("retrasos_subastas.csv", ["compromiso", "fecha_entrada"])
s2019 = ret[ret["mecanismo"].str.contains("TERCERA SUBASTA") & ret["mecanismo"].str.contains("Nuevas")].copy()
entro = s2019["estado_entrada"].isin(["en operación", "en pruebas"])
s2019["grupo"] = np.select([entro & (s2019["retraso_meses"] > 1), entro], ["entró tarde", "entró a tiempo"], "no ha entrado")
resumen_2019 = s2019.groupby("grupo").agg(plantas=("nombre", "size"), oef_gwh_dia=("oef_registrada_gwh_dia", "sum"))
resumen_2019["fraccion_oef"] = resumen_2019["oef_gwh_dia"] / resumen_2019["oef_gwh_dia"].sum()
tarde = s2019.loc[s2019["grupo"] == "entró tarde", "retraso_meses"]

# El modelo de retrasos del caso base: fracciones de OEF redondeadas a dos decimales y el retraso en períodos
MODELO_RETRASOS = {"a_tiempo": round(resumen_2019.loc["entró a tiempo", "fraccion_oef"], 2),
                   "tarde": round(resumen_2019.loc["entró tarde", "fraccion_oef"], 2),
                   "periodos_retraso": int(np.ceil(tarde.mean() / 12))}
print(f"Retrasos de las que entraron tarde: {', '.join(f'{m:.1f}' for m in tarde)} meses (media {tarde.mean():.1f})")
print("Modelo de retrasos:", MODELO_RETRASOS)
display(s2019[["nombre", "compromiso", "fecha_entrada", "estado_entrada", "retraso_meses", "oef_registrada_gwh_dia"]])
resumen_2019.round(3)

Retrasos de las que entraron tarde: 27.6, 18.8, 15.7 meses (media 20.7)
Modelo de retrasos: {'a_tiempo': np.float64(0.53), 'tarde': np.float64(0.19), 'periodos_retraso': 2}


,nombre,compromiso,fecha_entrada,estado_entrada,retraso_meses,oef_registrada_gwh_dia
0,TERMOCARIBE III 1,2021-12-01,2024-03-21,en operación,27.6,0.889470
1,LATAM SOLAR LA LOMA,2022-12-01,2024-06-24,en operación,18.8,0.525902
2,TESORITO,2022-12-01,2022-09-13,en operación,-2.6,4.572488
3,EL PASO,2022-12-01,2024-03-23,en operación,15.7,0.237644
55,ENEL COLOMBIA SA ESP,2021-12-01,NaT,no ha entrado (vencido),59.1,0.781183
56,FUTURA - PARQUE ALPHA,2022-12-01,NaT,no ha entrado (vencido),47.1,0.153058
57,FUTURA - APOTOLORRU,2022-12-01,NaT,no ha entrado (vencido),47.1,0.446234
58,FUTURA - PARQUE BETA,2022-12-01,NaT,no ha entrado (vencido),47.1,0.202152
59,FUTURA - CASA ELECTRICA,2022-12-01,NaT,no ha entrado (vencido),47.1,0.890679


,plantas,oef_gwh_dia,fraccion_oef
grupo,,,
entró a tiempo,1,4.572,0.526
entró tarde,3,1.653,0.190
no ha entrado,5,2.473,0.284


**Lectura.** De las nueve plantas nuevas de la subasta de 2019, solo Tesorito entró a tiempo, y de hecho unos tres meses antes. Tres entraron tarde, entre 16 y 28 meses: El Paso, La Loma y Termocaribe III. Cinco no han entrado, con 47 meses de retraso o más: Windpeshi (de Enel) y los cuatro parques eólicos de La Guajira (Alpha, Beta, Apotolorru y Casa Eléctrica). Medido por OEF: 53% a tiempo, 19% tarde y 28% sin entrar. Las que entraron tarde se demoraron 20,7 meses en promedio, que ocupan dos períodos del cargo. Ese es el modelo de retrasos del caso base; reemplaza el "20% de proyectos retrasados" que suponía la propuesta sin fuente.

### 3.2 Costos y bloques candidatos

Para escoger la expansión de mínimo costo hay que comparar tecnologías distintas con una misma vara: el costo anual de cada MWh de energía firme. Aquí se calcula para los seis bloques candidatos.

El modelo de expansión escoge entre **bloques**: una tecnología con un tamaño fijo (100 MW de ciclo abierto, por ejemplo), que puede entrar desde un período mínimo y hasta un cupo. Para compararlos hace falta llevar la inversión a un costo anual. El **factor de recuperación de capital** (CRF) convierte una inversión en una cuota anual con la tasa de descuento $r$ y la vida útil $n$:

$$\text{CRF}(r, n) = \frac{r(1+r)^n}{(1+r)^n - 1}, \qquad C_b = \text{MW}_b \big[\text{CAPEX}_b \cdot \text{CRF}(r, n_b) + \text{O\&M fijo}_b\big] + \text{costo variable}_b$$

El costo variable es el de la energía que se espera que la planta genere (combustible y O&M variable). La **energía firme** de un bloque es $\text{MW}_b \times 8.760 \text{ h} \times f_b$, con el factor firme $f_b$ del PIEG (Tabla 11). Dividir el costo anual por la energía firme da el **costo anual por MWh firme**: lo que cuesta cubrir 1 MWh de brecha durante un año con esa tecnología.

**De dónde salen los costos.** El paquete trae, por tecnología y escenario de costo (bajo, central y alto), los parámetros que el pipeline escogió con una jerarquía de fuentes: el Catálogo Tecnológico Colombiano de la UPME primero, el PIEG 2025-2039 después y, donde ninguno trae la tecnología, CNE Chile, EPE Brasil e IRENA, todo en USD de 2025. El Catálogo no trae tres tecnologías del modelo: la solar a gran escala en tierra, los motores diésel y la cogeneración con bagazo. La tasa de descuento es la del PIEG: 10% real.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `crf(r, n)` (propia) | Factor de recuperación de capital | $r$ = 10% y la vida útil de cada tecnología | Anualiza la inversión con la tasa del PIEG |
| `bloques(escenario_costos, r, factor_firme, cupos, retraso_periodos, objetivo)` (propia) | Arma un bloque por tecnología con su costo anual, su energía firme y su primer período factible | Escenario central, factor del PIEG y costo total (fijo + variable) | Los demás valores son los casos de sensibilidad de la sección 3.5 |
| `Series.map(diccionario o función)` | Traduce cada valor | El grupo de cupo → MW; el período → su posición | Une cada bloque con su cupo y su primer período |

In [30]:
TASA = 0.10   # tasa de descuento real del PIEG 2025-2039
# MW que pueden entrar en el horizonte por grupo; el gas es un cupo compartido por ciclo abierto y combinado.
# Bagazo: capacidad de los ingenios para vender excedentes (Asocaña); eólica: JK1, JK2 y Windpeshi; diésel: sin proyectos
CUPOS = {"solar_fv": 7918, "eolica_terrestre": 464, "gas": 1400, "diesel_motores": 0, "biomasa_bagazo": 40}
candidatos_costos = leer("candidatos_costos.csv")


def crf(r, n):
    """Factor de recuperación de capital: r(1+r)^n / ((1+r)^n − 1). Anualiza una inversión."""
    return r * (1 + r) ** n / ((1 + r) ** n - 1)

In [31]:
def bloques(escenario_costos="central", r=TASA, factor_firme="pieg", cupos=None, retraso_periodos=0,
            objetivo="costo_total"):
    """Un bloque por tecnología candidata, con su costo anual (MUSD), su factor firme y su primer período.

    escenario_costos: "bajo", "central" o "alto". factor_firme: "pieg" (Tabla 11) u "observado" (XM).
    cupos: cambios a CUPOS. retraso_periodos: corre el primer período factible. objetivo: "costo_total"
    (fijo + variable) o "solo_fijo" (la lógica del cargo, que paga la firmeza y no la energía).
    """
    c = candidatos_costos[candidatos_costos["escenario_costos"] == escenario_costos].copy()
    c["costo_fijo_musd_ano"] = c["tamano_mw"] * (c["capex_usd_kw"] * crf(r, c["vida_util"]) + c["om_fijo_usd_kw_ano"]) / 1e3
    c["costo_anual_musd"] = c["costo_fijo_musd_ano"] + (c["costo_variable_musd_ano"] if objetivo == "costo_total" else 0.0)
    if factor_firme == "observado":
        c["factor_firme"] = c["factor_firme_observado"]
    c["k_min"] = c["primer_periodo"].map(PERIODOS.index) + retraso_periodos
    c["cupo_mw"] = c["grupo_cupo"].map({**CUPOS, **(cupos or {})}).astype(float)
    c["costo_firme_usd_mwh"] = c["costo_anual_musd"] / (c["tamano_mw"] * 8.76 * c["factor_firme"] / 1e3)
    return c.reset_index(drop=True)


bloques().set_index("tecnologia")[["tamano_mw", "primer_periodo", "cupo_mw", "factor_firme", "costo_fijo_musd_ano",
                                   "costo_variable_musd_ano", "costo_firme_usd_mwh"]].sort_values("costo_firme_usd_mwh").round(2)

,tamano_mw,primer_periodo,cupo_mw,factor_firme,costo_fijo_musd_ano,costo_variable_musd_ano,costo_firme_usd_mwh
tecnologia,,,,,,,
biomasa_bagazo,20,2028-2029,40.0,0.54,1.94,0.00,20.54
gas_ciclo_combinado,300,2029-2030,1400.0,0.90,45.09,46.84,38.87
diesel_motores,20,2027-2028,0.0,0.88,1.33,5.96,47.24
gas_ciclo_abierto,100,2028-2029,1400.0,0.90,13.27,27.14,51.26
solar_fv,100,2027-2028,7918.0,0.13,10.84,0.00,95.19
eolica_terrestre,100,2029-2030,464.0,0.14,17.84,0.00,145.43


**Lectura.** 
- **Costo por MWh firme:** bagazo 21 USD, ciclo combinado 39, motores diésel 47, ciclo abierto 51, solar 95 y eólica 145. La solar y la eólica son caras por MWh firme porque el PIEG les reconoce poca energía firme: un bloque solar de 100 MW aporta unos 114 GWh firmes al año, frente a 788 de un ciclo abierto del mismo tamaño.
- **El calendario pesa más que el costo:** en 2027-2028 solo podrían llegar la solar y el diésel, que no tiene cupo; el ciclo abierto y el bagazo, desde 2028-2029, y el ciclo combinado y la eólica, en 2029-2030.
- **Cuidado con esta métrica:** ignora el valor de la energía que producen las renovables, que desplaza generación térmica. La sección 3.5 prueba el objetivo de solo costo fijo.

### 3.3 Balance de energía firme

La brecha de cada período $t$ es la demanda menos la oferta firme:

$$\text{Brecha}_t = \text{Demanda}_t - \Big(\underbrace{\text{ENFICC}_t}_{\text{plantas que operan}} + \underbrace{\textstyle\sum_m \text{MW}_m \cdot 24 \cdot f_m}_{\text{menores sin ENFICC}} + S_t + \sum_j \text{OEF}_{j,t} \cdot e_{j,t}\Big) \times \text{días}_t$$

Cada término tiene una regla:
- **Plantas con ENFICC:** cuenta su ENFICC de octubre de 2026 (el mes de referencia) si están en operación o en pruebas. Las registradas sin pruebas no generaron en doce meses y no suman.
- **Menores sin ENFICC:** las plantas pequeñas no despachadas centralmente no tienen ENFICC propia. Se cuentan con su capacidad × 24 h × el factor firme del PIEG de su tecnología ($f_m$), salvo las que probablemente ya son generación distribuida, que la UPME resta de la demanda: nombre que empieza por "GD", "MINIGRANJA" o solar de 1 MW o menos.
- **Retiros:** Termoyopal 2 sale el 1 de diciembre de 2027 (PIEG, Tabla 6); es la única planta del cronograma de salida sin OEF para 2029-2030.
- **Entradas sin subasta ($S_t$):** las PCH y la solar de 1 a 20 MW entran cada año sin OEF, al ritmo observado en XM: un año de entradas en 2027-2028, dos en 2028-2029 y tres en 2029-2030.
- **Proyectos nuevos:** su OEF de cada período por $e_{j,t}$, la fracción que se espera que esté operando. Sin retrasos, $e = 1$ desde su período de entrada. Con retrasos, el modelo de 3.1: el 53% entra a tiempo, el 19% dos períodos tarde y el resto no entra; un proyecto con el compromiso ya vencido no entra.

| Escenario | Demanda de la UPME | Proyectos nuevos |
|---|---|---|
| Base | Media | Con retrasos |
| Estrés | Alta (borde superior de la banda de 68%) | Con retrasos |
| Optimista | Media | Sin retrasos |

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `Series.isin(lista)` | Marca los valores que están en la lista | Situación "en operación" o "en pruebas" | Solo esas plantas generan |
| `Series.str.startswith(texto)` | Marca los textos que empiezan así | `"GD "` | Así se nombran las plantas de generación distribuida |
| `parque_firme(plantas)` (propia) | Energía firme de cada planta existente con las reglas de arriba | Las plantas del mes de referencia | Cada planta queda con su motivo si no suma |

In [32]:
FACTORES_MENORES = {"solar": 0.13, "hidraulica": 0.15, "eolica": 0.14, "biomasa": 0.54, "termica": 0.90}  # PIEG
GD_MAX_MW = 1.0                    # la generación distribuida es de 1 MW o menos (CREG 174 de 2021)
RETIROS = {"TYP2": "2027-12-01"}   # Termoyopal 2 (PIEG 2025-2039, Tabla 6)
plantas = leer("xm_plantas_mes_referencia.csv", dtype={"codigo": str})


def parque_firme(plantas):
    """Energía firme diaria (GWh-día) de cada planta existente, con el motivo si no suma."""
    p = plantas.copy()
    con_enficc = p["enficc_gwh_dia"] > 0
    opera = p["situacion"].isin(["en operación", "en pruebas"])
    nombre = p["nombre"].fillna("").str.upper()
    gd = ~con_enficc & (nombre.str.startswith("GD ") | nombre.str.contains("MINIGRANJA")
                        | ((p["tecnologia"] == "solar") & (p["capacidad_mw"] <= GD_MAX_MW)))
    firme_menor = (p["capacidad_mw"] * 24 * p["tecnologia"].map(FACTORES_MENORES) / 1e3).fillna(0)
    p["fuente"] = np.where(con_enficc, "ENFICC", "menor (factor firme)")
    p["motivo"] = np.select([con_enficc & ~opera, gd], ["no opera", "probable generación distribuida"], "suma")
    p["firme_gwh_dia"] = np.select([con_enficc & opera, ~con_enficc & ~gd], [p["enficc_gwh_dia"], firme_menor], 0.0)
    return p


parque = parque_firme(plantas)
parque.groupby(["fuente", "motivo"]).agg(plantas=("codigo", "size"), firme_gwh_dia=("firme_gwh_dia", "sum"),
                                         enficc_gwh_dia=("enficc_gwh_dia", "sum")).round(2)

plantas  firme_gwh_dia  enficc_gwh_dia
fuente               motivo                                                                 
ENFICC               no opera                              17           0.00            9.99
                     suma                                 138         237.69          237.69
menor (factor firme) probable generación distribuida      233           0.00            0.00
                     suma                                 243          13.65            0.00

**Lectura.** El parque que opera aporta 237,7 GWh-día de ENFICC en 138 plantas. Otras 17 unidades, con 10,0 GWh-día de ENFICC, no suman: están registradas sin pruebas o fuera del listado de XM, y ninguna generó en doce meses. Las menores suman 13,65 GWh-día en 243 plantas; otras 233, de 1 MW o menos o con nombre de generación distribuida, quedan por fuera para no contarlas dos veces. En total, el parque existente garantiza unos 251,3 GWh-día.

Ahora los proyectos nuevos. `fraccion_entrada()` aplica el modelo de retrasos de la sección 3.1 a un proyecto, y la suma sobre todos da la OEF nueva que se espera en cada período.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `fraccion_entrada(periodo_entrada, vencido, retrasos)` (propia) | Fracción esperada de la OEF de un proyecto que opera en cada período | El período en que empieza su OEF, si ya venció y si el escenario lleva retrasos | Con retrasos: 53% desde su período y 19% más dos períodos después |
| `np.arange(n)` | Los enteros de 0 a n − 1 | Las posiciones de los cuatro períodos | Para comparar cada período con el de entrada |
| `DataFrame.to_numpy()` | Pasa la tabla a una matriz de NumPy | La OEF de los proyectos por período | El cálculo es una suma de productos, más rápida sobre matrices |

In [33]:
nuevos = leer("proyectos_nuevos_oef.csv")
COLUMNAS_OEF = [f"oef_{p.replace('-', '_')}" for p in PERIODOS]


def fraccion_entrada(periodo_entrada, vencido, retrasos, modelo=MODELO_RETRASOS):
    """Fracción esperada de la OEF de un proyecto que está operando en cada período del horizonte."""
    k, t = PERIODOS.index(periodo_entrada), np.arange(len(PERIODOS))
    if not retrasos:
        return (t >= k).astype(float)
    if vencido:                                   # ya pasó su compromiso sin entrar: no entra
        return np.zeros(len(PERIODOS))
    return modelo["a_tiempo"] * (t >= k) + modelo["tarde"] * (t >= k + modelo["periodos_retraso"])


def oferta_nuevos(nuevos, retrasos):
    """OEF esperada de los proyectos nuevos en cada período (GWh-día)."""
    e = np.array([fraccion_entrada(p, v, retrasos) for p, v in zip(nuevos["periodo_entrada"], nuevos["vencido"])])
    return pd.Series((nuevos[COLUMNAS_OEF].to_numpy() * e).sum(axis=0), index=PERIODOS)


def solar_comprometida(nuevos, retrasos):
    """MW solares con OEF que se espera que entren en cada período (para el tope de construcción solar)."""
    s = nuevos[nuevos["tecnologia"] == "solar"]
    e = np.array([fraccion_entrada(p, v, retrasos) for p, v in zip(s["periodo_entrada"], s["vencido"])])
    entra = np.diff(e, axis=1, prepend=0)          # lo que entra en cada período, no lo acumulado
    return pd.Series((entra * s[["mw"]].to_numpy()).sum(axis=0), index=PERIODOS)


print(f"{len(nuevos)} proyectos con OEF que no operan; por período de entrada:")
pd.DataFrame({"con retrasos": oferta_nuevos(nuevos, True), "sin retrasos": oferta_nuevos(nuevos, False)}).round(2)

41 proyectos con OEF que no operan; por período de entrada:


,con retrasos,sin retrasos
2026-2027,0.00,0.00
2027-2028,4.03,7.61
2028-2029,4.34,8.18
2029-2030,11.02,17.97


**Lectura.** Los 41 proyectos con OEF que no operan aportarían 18,0 GWh-día en 2029-2030 si todos llegaran a tiempo, pero solo 11,0 con los retrasos de la subasta de 2019: un 39% menos. En 2027-2028 la diferencia es aún mayor (4,0 frente a 7,6), porque en su primer período solo cuenta el 53% de cada proyecto.

Con todas las piezas, `oferta_firme()` arma la oferta de cada período y `balance()` la resta de la demanda de la sección 2.5.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `oferta_firme(retrasos)` (propia) | Energía firme de cada período (GWh), por componente | Con o sin retrasos | Las dos ofertas de los tres escenarios |
| `balance(escenario)` (propia) | Demanda, oferta y brecha de cada período de un escenario | `"base"`, `"estres"` u `"optimista"` | Cada escenario combina una demanda con una oferta |
| `pd.concat(tablas, ignore_index=True)` | Apila tablas una debajo de otra | Los tres escenarios | Una sola tabla para comparar y graficar |

In [34]:
ESCENARIOS = {"base": ("media", True), "estres": ("alta", True), "optimista": ("media", False)}
dias = demanda.drop_duplicates("periodo").set_index("periodo")["dias"].reindex(PERIODOS)
entradas = leer("entradas_sin_subasta.csv")
sin_subasta_dia = (entradas["mw_ano"] * entradas["tecnologia"].map(FACTORES_MENORES) * 24 / 1e3).sum()  # por año
retiros_dia = pd.Series([sum(parque.loc[parque["codigo"] == c, "firme_gwh_dia"].sum()
                             for c, fecha in RETIROS.items() if f"{p[:4]}-12-01" >= fecha) for p in PERIODOS], index=PERIODOS)


def oferta_firme(retrasos):
    """Energía firme de cada período (GWh): parque con ENFICC sin retiros, menores, entradas sin subasta y nuevos."""
    base = parque.groupby("fuente")["firme_gwh_dia"].sum()
    return pd.DataFrame({"enficc_gwh": (base["ENFICC"] - retiros_dia) * dias,
                         "menores_gwh": base["menor (factor firme)"] * dias,
                         "sin_subasta_gwh": sin_subasta_dia * np.arange(len(PERIODOS)) * dias,
                         "nuevas_gwh": oferta_nuevos(nuevos, retrasos) * dias})


def balance(escenario):
    """Demanda, oferta firme por componente y brecha de cada período de un escenario."""
    demanda_esc, retrasos = ESCENARIOS[escenario]
    t = oferta_firme(retrasos)
    t["oferta_firme_gwh"] = t.sum(axis=1)
    t["demanda_gwh"] = demanda.query("escenario_demanda == @demanda_esc").set_index("periodo")["demanda_gwh"]
    t["brecha_gwh"] = t["demanda_gwh"] - t["oferta_firme_gwh"]
    t["brecha_pct"] = 100 * t["brecha_gwh"] / t["demanda_gwh"]
    return t.assign(escenario=escenario, dias=dias).rename_axis("periodo").reset_index()

In [35]:
balances = pd.concat([balance(e) for e in ESCENARIOS], ignore_index=True)
pipeline = leer("pipeline_brechas_escenarios.csv").query("definicion == @DEFINICION")
comprobar(balances.set_index(["escenario", "periodo"])["brecha_gwh"],
          pipeline.set_index(["escenario", "periodo"])["brecha_firme_gwh"], "Brecha firme (GWh), 3 escenarios × 4 períodos")
balances.pivot(index="periodo", columns="escenario", values=["brecha_gwh", "brecha_pct"]).round(1)

✓ Brecha firme (GWh), 3 escenarios × 4 períodos: coincide con el pipeline (diferencia máxima 1.8e-06)


brecha_gwh                   brecha_pct                 
escenario       base  estres optimista       base estres optimista
periodo                                                           
2026-2027     1770.0  4493.1    1770.0        1.9    4.7       1.9
2027-2028     2808.6  6531.8    1499.3        2.9    6.5       1.6
2028-2029     5167.8  9740.3    3764.1        5.2    9.4       3.8
2029-2030     4584.9  9933.4    2048.8        4.6    9.4       2.0

**Lectura.** La brecha coincide con la del pipeline en los doce casos; la diferencia máxima es de una millonésima de GWh, redondeo.
- **Base (demanda media, con retrasos):** falta energía firme desde 2026-2027: 1.770 GWh (1,9% de la demanda), sin proyectos nuevos ese período. El máximo llega en 2028-2029: 5.168 GWh (5,2%), el año sin subasta y ya sin Termoyopal 2. En 2029-2030 quedan 4.585 GWh (4,6%).
- **Optimista (sin retrasos):** baja a 1.499 GWh en 2027-2028 y a 2.049 en 2029-2030, pero no desaparece en ningún período.
- **Estrés (demanda alta, con retrasos):** falta desde el primer período y crece hasta 9.933 GWh en 2029-2030 (9,4%).

**El año normal, como contraste.** La ENFICC es la energía que el parque garantiza con la hidrología crítica, así que el balance de arriba es el del año seco, el de El Niño. Los profesores recomendaron, al evaluar la propuesta, calcular también un año de hidrología normal. Para eso se mide cuánto generaron las hidráulicas con ENFICC frente a su propia ENFICC en cada período del cargo desde 2010-2011 (la generación por planta empieza en 2010), y cada período se clasifica por sus aportes, con los umbrales de la sección 2.5: seco por debajo del 80% de la media histórica, normal entre 80% y 110% y húmedo por encima.

En el año normal, la hidráulica con ENFICC aporta su ENFICC multiplicada por la mediana de los períodos normales; lo demás queda igual. Es una estimación conservadora: en un año normal la generación hidráulica la limita la demanda, no el agua, así que el agua que sobra no se ve en la generación.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `pd.cut(serie, bins, labels)` | Clasifica valores en intervalos | Cortes en 80% y 110% de los aportes medios | Los mismos umbrales de la sección 2.5 |
| `groupby(...).agg([...])` | Resume por grupo con varias funciones | Número, mínimo, mediana y máximo por condición | Muestra cuánto varía la relación en cada condición, no solo su centro |
| `Series.median()` | La mediana | Los períodos normales | Con seis períodos, un período atípico mueve menos la mediana que la media |
| `balance_normal(escenario)` (propia) | Brecha del año normal | `"base"` o `"estres"` | Suma a la oferta la energía extra de la hidráulica y deja igual lo demás |

In [36]:
hidro = leer("xm_hidraulica_periodos.csv")
hidro["aportes_pct"] = 100 * hidro["aportes_gwh"] / hidro["aportes_media_hist_gwh"]
hidro["condicion"] = pd.cut(hidro["aportes_pct"], [-np.inf, 80, 110, np.inf], labels=["Seca", "Normal", "Húmeda"])
hidro["generacion_sobre_enficc"] = hidro["generacion_hidraulica_gwh"] / hidro["dias"] / hidro["enficc_hidraulica_gwh_dia"]
FACTOR_ANO_NORMAL = hidro.loc[hidro["condicion"] == "Normal", "generacion_sobre_enficc"].median()
print(f"{len(hidro)} períodos completos desde 2010-2011; en los normales, la hidráulica generó "
      f"{FACTOR_ANO_NORMAL:.2f} veces su ENFICC (mediana)")
display(hidro.set_index("periodo")[["aportes_pct", "condicion", "generacion_sobre_enficc"]].round(2))
hidro.groupby("condicion", observed=True)["generacion_sobre_enficc"].agg(["count", "min", "median", "max"]).round(2)

14 períodos completos desde 2010-2011; en los normales, la hidráulica generó 1.26 veces su ENFICC (mediana)


,aportes_pct,condicion,generacion_sobre_enficc
periodo,,,
2010-2011,133.18,Húmeda,1.48
2011-2012,111.83,Húmeda,1.39
2012-2013,88.38,Normal,1.22
2013-2014,93.19,Normal,1.22
2015-2016,79.17,Seca,1.05
2016-2017,98.56,Normal,1.32
2017-2018,103.94,Normal,1.30
2018-2019,86.75,Normal,1.06
2019-2020,79.50,Seca,1.04


,count,min,median,max
condicion,,,,
Seca,3,1.04,1.05,1.20
Normal,6,1.06,1.26,1.35
Húmeda,5,1.28,1.40,1.48


In [37]:
ENFICC_HIDRAULICA = parque.loc[(parque["fuente"] == "ENFICC") & (parque["tecnologia"] == "hidraulica"), "firme_gwh_dia"].sum()


def balance_normal(escenario):
    """Brecha del año de hidrología normal (GWh por período): la hidráulica con ENFICC aporta FACTOR_ANO_NORMAL
    veces su ENFICC y lo demás queda como en el balance firme."""
    t = balance(escenario).set_index("periodo")
    return t["brecha_gwh"] - ENFICC_HIDRAULICA * (FACTOR_ANO_NORMAL - 1) * t["dias"]


print(f"ENFICC hidráulica del parque: {ENFICC_HIDRAULICA:.1f} GWh-día; en el año normal aporta "
      f"{ENFICC_HIDRAULICA * (FACTOR_ANO_NORMAL - 1):.1f} GWh-día más")
nino = balances.pivot(index="periodo", columns="escenario", values="brecha_gwh")[["base", "estres"]]
ano_normal = pd.concat({"El Niño (energía firme)": nino,
                        "Año normal": pd.DataFrame({e: balance_normal(e) for e in ["base", "estres"]})}, axis=1)
ano_normal.round(0)

ENFICC hidráulica del parque: 113.4 GWh-día; en el año normal aporta 29.7 GWh-día más


El Niño (energía firme)         Año normal        
escenario                    base  estres       base  estres
periodo                                                     
2026-2027                  1770.0  4493.0    -9080.0 -6357.0
2027-2028                  2809.0  6532.0    -8071.0 -4348.0
2028-2029                  5168.0  9740.0    -5682.0 -1109.0
2029-2030                  4585.0  9933.0    -6265.0  -916.0

**Lectura.** La hidráulica generó más que su ENFICC en todos los períodos observados, también en los secos (mediana de 1,05), así que la ENFICC describe bien lo que entrega en un año seco. En los seis períodos normales generó entre 1,06 y 1,35 veces su ENFICC, con mediana de 1,26; en los húmedos, 1,40. Con ese factor, las hidráulicas con ENFICC del parque (113,4 GWh-día) aportan 29,7 GWh-día más en un año normal, unos 10.850 GWh por período.

En un año normal no falta energía: en el base sobran entre 5.682 GWh (2028-2029) y 9.080 GWh (2026-2027). Con la demanda alta, el margen se estrecha: sobran 1.109 GWh en 2028-2029 y 916 en 2029-2030, alrededor del 1% de la demanda. La respuesta a cuánta energía falta y para cuándo depende de la hidrología: con El Niño falta desde 2026-2027; en un año normal no falta. Por eso la expansión se dimensiona para el año seco, que es el que define el riesgo.

### 3.4 Expansión de mínimo costo (MILP)

Con la brecha de la sección 3.3 y los bloques de la 3.2, un modelo de optimización con variables enteras escoge qué construir, cuándo y a qué costo.

Un **MILP** (*mixed-integer linear program*, programa lineal entero mixto) es un problema de optimización con objetivo y restricciones lineales en el que algunas variables solo pueden tomar valores enteros. Aquí las enteras son los bloques de cada tecnología que entran en cada período (no se construye media planta), y la continua es la energía que ningún bloque alcanza a cubrir a tiempo.

$$\min_{y,\,u}\; \sum_{t} \delta_t \Big[ \sum_{b} C_b\, x_{b,t} + \pi\, u_t \Big] \quad \text{sujeto a} \quad
\begin{aligned}
x_{b,t} &= \textstyle\sum_{\tau \le t} y_{b,\tau} && \text{bloques en operación}\\
\textstyle\sum_b F_{b,t}\, x_{b,t} + u_t &\ge G_t && \text{brecha firme (sección 3.3)}\\
y_{b,t} &= 0 \ \text{ si } t < t^{\min}_b && \text{primer período factible}\\
\textstyle\sum_{b \in g} \text{MW}_b\, x_{b,T} &\le \text{cupo}_g && \text{cupo por grupo}\\
\text{MW}_s\, y_{s,t} &\le \text{tope}_t && \text{tope de construcción solar}\\
y_{b,t} \in \mathbb{Z}_{\ge 0},\ u_t &\ge 0
\end{aligned}$$

| Símbolo | Qué es | De dónde sale |
|---|---|---|
| $y_{b,t}$ | Bloques de la tecnología $b$ que entran en el período $t$ (entero) | Variable |
| $u_t$ | Energía que ningún bloque cubre (GWh) | Variable |
| $C_b$ | Costo anual del bloque (MUSD) | Sección 3.2 |
| $F_{b,t}$ | Energía firme del bloque en el período: MW × horas × factor firme | Sección 3.2 |
| $G_t$ | Brecha firme | Sección 3.3 |
| $\delta_t$ | Descuento $(1+r)^{-t}$, con $t = 0$ en 2026-2027 y $r$ = 10% | PIEG |
| $\pi$ | Penalidad de la energía no cubierta: el costo de racionamiento de la UPME, primer tramo | UPME (octubre de 2026) |
| $\text{tope}_t$ | 1.400 MW/año de solar, el récord observado en XM, menos los MW solares con OEF que se espera que entren en $t$ | XM; sección 3.3 |

Dos detalles de implementación. $x$ no es una variable: se reemplaza por la suma acumulada de $y$, así que quedan 24 enteras ($6$ tecnologías × $4$ períodos) y 4 continuas. Y los costos son anuales: un bloque que entra en 2029-2030 paga solo un año de su anualidad dentro del horizonte, lo que lo compara bien con uno que entra antes.

**La penalidad.** El costo de racionamiento de la UPME viene en pesos por kWh. Se lleva a dólares con la TRM del mes de referencia y a USD de 2025 con el CPI de EE. UU.:

In [38]:
meta = json.loads((CARPETA / "metadatos.json").read_text(encoding="utf-8"))
m = meta["moneda"]
PENALIDADES = {k: v["cop_kwh"] * 1000 / m["trm"] * m["factor_cpi"] for k, v in meta["penalidades"].items()}
for criterio, usd in PENALIDADES.items():
    print(f"{criterio}: {meta['penalidades'][criterio]['cop_kwh']:,.1f} COP/kWh → {usd:,.0f} USD de 2025/MWh")
TOPE_SOLAR = 1400
SOLAR_COMPROMETIDA = {r: solar_comprometida(nuevos, r) for r in (True, False)}
pd.DataFrame({"solar con OEF, con retrasos (MW)": SOLAR_COMPROMETIDA[True],
              "lo que el tope deja, con retrasos (MW)": (TOPE_SOLAR - SOLAR_COMPROMETIDA[True]).clip(lower=0)}).round(0)

costo_racionamiento_upme: 2,572.5 COP/kWh → 770 USD de 2025/MWh
precio_escasez_superior: 1,463.5 COP/kWh → 438 USD de 2025/MWh
precio_escasez_inferior: 327.2 COP/kWh → 98 USD de 2025/MWh


,"solar con OEF, con retrasos (MW)","lo que el tope deja, con retrasos (MW)"
2026-2027,0.0,1400.0
2027-2028,1486.0,0.0
2028-2029,238.0,1162.0
2029-2030,1295.0,105.0


**Lectura.** La penalidad es de 770 USD de 2025 por MWh: lo que cuesta racionar el primer 1,5% de la demanda, el tramo más barato del costo de racionamiento de la UPME. Los precios de escasez superior (438) e inferior (98) quedan para la sección 3.5. El tope solar no deja espacio en 2027-2028: los proyectos solares con OEF que se espera que entren ese período suman unos 1.486 MW con retrasos, más que los 1.400 del tope. En 2028-2029 quedan libres unos 1.162 MW, y en 2029-2030, 105.

El modelo se arma en dos funciones: `restricciones()` escribe cada fila de la matriz $A$ con sus límites, y `armar_milp()` agrega el vector de costos, las cotas y la integralidad.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `np.outer(a, b)` | Producto exterior: una matriz con todos los productos $a_i b_j$ | MW × factor firme de cada bloque y horas de cada período | Da de una vez la energía firme $F_{b,t}$ de cada bloque en cada período |
| `np.triu(np.ones((n, n))) @ delta` | Suma, para cada período, los descuentos de ese período en adelante | La matriz triangular superior | Un bloque que entra en τ paga su costo en todos los períodos desde τ |
| `np.r_[a, b]` | Concatena vectores | Costos de las $y$ y de las $u$ | El vector de costos sigue el orden de las variables |
| `groupby("grupo_cupo", sort=False)` | Agrupa conservando el orden de aparición | Los grupos de cupo | El ciclo abierto y el combinado comparten el cupo de gas |
| `restricciones(...)` y `armar_milp(...)` (propias) | Arman el problema | Los bloques, la brecha, los días, la tasa, la penalidad y el tope | Separan la matriz del resto para que cada pieza se lea sola |

In [39]:
def restricciones(bl, F, mw, brecha, tope_solar):
    """Filas de A con sus límites lb y ub: brecha firme de cada período, cupo de cada grupo y tope solar."""
    nb, nt = F.shape
    n = nb * nt + nt                                  # variables: y[b, t] en la posición b·nt + t, y después u[t]
    A, lb, ub, filas = [], [], [], []
    for t in range(nt):                               # energía firme nueva + energía no cubierta ≥ brecha
        a = np.zeros(n)
        for k in range(nb):
            a[k * nt: k * nt + t + 1] = F[k, t]       # un bloque que entró en τ ≤ t está operando en t
        a[nb * nt + t] = 1.0
        A.append(a), lb.append(brecha[t]), ub.append(np.inf), filas.append(f"firme {PERIODOS[t]}")
    for grupo, sub in bl.groupby("grupo_cupo", sort=False):   # MW totales del grupo ≤ cupo
        a = np.zeros(n)
        for k in sub.index:
            a[k * nt:(k + 1) * nt] = mw[k]
        A.append(a), lb.append(-np.inf), ub.append(float(sub["cupo_mw"].iloc[0])), filas.append(f"cupo {grupo}")
    k = int(np.flatnonzero(bl["tecnologia"] == "solar_fv")[0])
    for t in range(nt):                               # MW solares que entran en t ≤ lo que deja el tope
        a = np.zeros(n)
        a[k * nt + t] = mw[k]
        A.append(a), lb.append(-np.inf), ub.append(float(tope_solar.iloc[t])), filas.append(f"tope solar {PERIODOS[t]}")
    return np.array(A), np.array(lb), np.array(ub), filas

In [40]:
def armar_milp(bl, brecha, dias, r, pi, tope_solar):
    """Arma el MILP: costos c, restricciones lb ≤ A·v ≤ ub, cotas e integralidad, con v = (y[b, t], u[t]).

    bl: bloques de la sección 3.2. brecha y dias: por período. r: tasa de descuento. pi: penalidad (USD/MWh).
    tope_solar: MW solares nuevos permitidos en cada período.
    """
    nb, nt = len(bl), len(PERIODOS)
    delta = (1 + r) ** -np.arange(nt, dtype=float)
    mw = bl["tamano_mw"].to_numpy(float)
    F = np.outer(mw * bl["factor_firme"].to_numpy(float), np.asarray(dias, float) * 24) / 1e3   # GWh firmes
    C = bl["costo_anual_musd"].to_numpy(float)
    c = np.r_[(C[:, None] * (np.triu(np.ones((nt, nt))) @ delta)[None, :]).ravel(), delta * pi / 1e3]
    A, lb, ub, filas = restricciones(bl, F, mw, np.asarray(brecha, float), tope_solar)
    alto = np.r_[np.repeat(np.floor(bl["cupo_mw"].to_numpy() / mw), nt), np.full(nt, np.inf)]
    alto[:nb * nt] *= (np.arange(nt)[None, :] >= bl["k_min"].to_numpy()[:, None]).ravel()   # antes de t_min: 0
    return dict(c=c, A=A, lb=lb, ub=ub, cotas=Bounds(0, alto), enteras=np.r_[np.ones(nb * nt), np.zeros(nt)],
                F=F, C=C, delta=delta, filas=filas, pi=pi)

`resolver()` llama a `scipy.optimize.milp`, que usa el solver HiGHS.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `scipy.optimize.milp(c, constraints, integrality, bounds, options)` | Minimiza c·v sujeto a lb ≤ A·v ≤ ub, con enteras las variables marcadas en `integrality` | `LinearConstraint(A, lb, ub)`; `integrality` = 1 en las $y$ y 0 en las $u$; `options={"mip_rel_gap": 1e-9}` | Con 28 variables, HiGHS encuentra el óptimo exacto en menos de un segundo: no hace falta aceptar una brecha de optimalidad |
| `LinearConstraint(A, lb, ub)` y `Bounds(lb, ub)` | Empaquetan las restricciones y las cotas | La matriz y los vectores de `armar_milp()` | Es el formato que pide `milp` |
| `resolver(pr, relajado)` (propia) | Resuelve el MILP o su relajación lineal | `relajado=True` quita la integralidad | La relajación sirve para validar: su costo nunca puede superar al del MILP |

In [41]:
def resolver(pr, relajado=False):
    """Resuelve el MILP con scipy.optimize.milp (HiGHS); con `relajado`, sin integralidad. Se detiene si falla."""
    res = milp(pr["c"], constraints=LinearConstraint(pr["A"], pr["lb"], pr["ub"]), bounds=pr["cotas"],
               integrality=np.zeros_like(pr["enteras"]) if relajado else pr["enteras"],
               options={"mip_rel_gap": 1e-9})
    if not res.success:
        raise RuntimeError(f"El MILP no se resolvió: {res.message}")
    return res


def brecha_de(escenario):
    """Brecha firme (GWh) y días de cada período de un escenario, de la sección 3.3."""
    return balances.query("escenario == @escenario").set_index("periodo")[["brecha_gwh", "dias"]].reindex(PERIODOS)

**Un caso paso a paso: el estrés** (demanda alta, con retrasos), el escenario más exigente. Primero se arma el problema y se mira la matriz: cada fila es una restricción y cada columna una variable. Se muestran las columnas del ciclo abierto, la solar y la energía no cubierta; las demás tecnologías tienen la misma estructura.

In [42]:
bl = bloques()
b = brecha_de("estres")
pr = armar_milp(bl, b["brecha_gwh"], b["dias"], TASA, PENALIDADES["costo_racionamiento_upme"],
                (TOPE_SOLAR - SOLAR_COMPROMETIDA[True]).clip(lower=0))
corto = [p[2:4] + "-" + p[7:] for p in PERIODOS]                       # "2026-2027" → "26-27"
columnas = [f"y {t} {p}" for t in bl["tecnologia"] for p in corto] + [f"u {p}" for p in corto]
print(f"{pr['A'].shape[0]} restricciones y {pr['A'].shape[1]} variables ({int(pr['enteras'].sum())} enteras)")
pd.DataFrame(pr["A"], index=pr["filas"], columns=columnas).filter(regex=r"^(y gas_ciclo_abierto|y solar_fv|u) ").round(0)

13 restricciones y 28 variables (24 enteras)


,y gas_ciclo_abierto 26-27,y gas_ciclo_abierto 27-28,y gas_ciclo_abierto 28-29,y gas_ciclo_abierto 29-30,y solar_fv 26-27,y solar_fv 27-28,y solar_fv 28-29,y solar_fv 29-30,u 26-27,u 27-28,u 28-29,u 29-30
firme 2026-2027,788.0,0.0,0.0,0.0,114.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
firme 2027-2028,791.0,791.0,0.0,0.0,114.0,114.0,0.0,0.0,0.0,1.0,0.0,0.0
firme 2028-2029,788.0,788.0,788.0,0.0,114.0,114.0,114.0,0.0,0.0,0.0,1.0,0.0
firme 2029-2030,788.0,788.0,788.0,788.0,114.0,114.0,114.0,114.0,0.0,0.0,0.0,1.0
cupo biomasa_bagazo,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
cupo diesel_motores,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
cupo eolica_terrestre,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
cupo gas,100.0,100.0,100.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
cupo solar_fv,0.0,0.0,0.0,0.0,100.0,100.0,100.0,100.0,0.0,0.0,0.0,0.0
tope solar 2026-2027,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [43]:
res = resolver(pr)
nb, nt = len(bl), len(PERIODOS)
y = np.round(res.x[:nb * nt]).reshape(nb, nt)      # bloques que entran en cada período
x, u = y.cumsum(axis=1), res.x[nb * nt:]            # bloques en operación y energía no cubierta
print(f"{res.message} Costo en valor presente: {res.fun:,.1f} MUSD de 2025")
mw = pd.DataFrame(x * bl["tamano_mw"].to_numpy()[:, None], index=bl["tecnologia"], columns=PERIODOS)
display(mw[mw.sum(axis=1) > 0].rename_axis("MW en operación"))
pd.DataFrame({"brecha firme (GWh)": b["brecha_gwh"], "energía firme nueva (GWh)": (x * pr["F"]).sum(axis=0),
              "energía no cubierta (GWh)": u}).round(0)

Optimization terminated successfully. (HiGHS Status 7: Optimal) Costo en valor presente: 8,841.5 MUSD de 2025


,2026-2027,2027-2028,2028-2029,2029-2030
MW en operación,,,,
biomasa_bagazo,0.0,0.0,40.0,40.0
eolica_terrestre,0.0,0.0,0.0,100.0
gas_ciclo_abierto,0.0,0.0,1200.0,1200.0
solar_fv,0.0,0.0,100.0,200.0


,brecha firme (GWh),energía firme nueva (GWh),energía no cubierta (GWh)
periodo,,,
2026-2027,4493.0,0.0,4493.0
2027-2028,6532.0,0.0,6532.0
2028-2029,9740.0,9764.0,0.0
2029-2030,9933.0,10000.0,0.0


**Lectura.** 
- **2026-2027 y 2027-2028:** no llega ninguna tecnología, así que la brecha queda sin cubrir: 4.493 y 6.532 GWh. En 2027-2028 solo podría entrar la solar, pero los proyectos con OEF ya llenan el tope, y el diésel no tiene cupo.
- **2028-2029:** entran 12 bloques de ciclo abierto (1.200 MW), 40 MW de bagazo, que llega a su cupo, y 100 MW solares en el espacio que deja el tope: 9.764 GWh firmes para una brecha de 9.740.
- **2029-2030:** la brecha sube a 9.933 y lo construido cubre 9.764: faltan 169 GWh. Entran 100 MW eólicos y 100 solares; otro bloque de ciclo abierto traería 788 GWh, casi cinco veces lo que falta, y costaría más.
- **Costo:** 8.842 MUSD en valor presente, casi todo penalidad de la energía que no alcanza a cubrirse a tiempo. Los bloques cuestan 810.

**Los tres escenarios.** `correr()` repite lo anterior para cualquier escenario y caso de sensibilidad, y resume la solución: el costo de los bloques en valor presente, el costo total con la penalidad, la energía no cubierta y los MW de cada tecnología al final del horizonte. También resuelve la relajación lineal, para validar.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `correr(escenario, caso, **cambios)` (propia) | Resuelve un escenario con los cambios de un caso y resume la solución | `caso="Central"`; `brecha_extra` suma GWh a la brecha (sección 5 y ejercicio 2) | Una sola función para los escenarios, las sensibilidades y los ejercicios |
| `dict \| dict` | Une dos diccionarios; el de la derecha manda | Los cambios del caso y los que se pasen | Así un caso puede modificarse sin reescribirlo |

In [44]:
CASOS = {"Central": {}}   # la sección 3.5 agrega los casos de sensibilidad


def correr(escenario, caso="Central", **cambios):
    """Resuelve el MILP de un escenario con los cambios de un caso de sensibilidad y resume la solución.

    Devuelve (resumen, detalle): el resumen tiene los costos en valor presente (MUSD de 2025), la energía
    no cubierta (GWh) y los MW de cada tecnología en 2029-2030; el detalle, el problema y la solución.
    """
    cambios = CASOS.get(caso, {}) | cambios
    r = cambios.get("r", TASA)
    bl = bloques(cambios.get("escenario_costos", "central"), r, cambios.get("factor_firme", "pieg"),
                 cambios.get("cupo"), cambios.get("retraso_periodos", 0), cambios.get("objetivo", "costo_total"))
    pi = PENALIDADES[cambios.get("criterio_penalidad", "costo_racionamiento_upme")] * cambios.get("factor_penalidad", 1)
    b = brecha_de(escenario)
    tope = (cambios.get("tope_solar", TOPE_SOLAR) - SOLAR_COMPROMETIDA[ESCENARIOS[escenario][1]]).clip(lower=0)
    pr = armar_milp(bl, b["brecha_gwh"] + cambios.get("brecha_extra", 0), b["dias"], r, pi, tope)
    res, lp = resolver(pr), resolver(pr, relajado=True)
    nb, nt = len(bl), len(PERIODOS)
    x = np.round(res.x[:nb * nt]).reshape(nb, nt).cumsum(axis=1)
    u = res.x[nb * nt:]
    resumen = {"escenario": escenario, "caso": caso, "costo_bloques_vp_musd": float(pr["delta"] @ (pr["C"] @ x)),
               "costo_total_vp_musd": res.fun, "objetivo_lp_musd": lp.fun, "no_cubierta_gwh": u.sum()}
    resumen |= {f"mw_{t}": x[k, -1] * bl.at[k, "tamano_mw"] for k, t in enumerate(bl["tecnologia"])}
    return resumen, {"pr": pr, "x": x, "u": u, "bl": bl}

In [45]:
corridas = {e: correr(e) for e in ESCENARIOS}
central = pd.DataFrame([resumen for resumen, _ in corridas.values()]).set_index("escenario")
pipeline_exp = (leer("pipeline_expansion_resumen.csv").query("definicion == @DEFINICION")
                .replace({"caso": {"Factores firmes observados (Fase 2)": "Factores firmes observados en XM"}})
                .set_index(["escenario", "caso"]))
cols = ["costo_bloques_vp_musd", "costo_total_vp_musd", "no_cubierta_gwh"] + [c for c in central if c.startswith("mw_")]
comprobar(central[cols], pipeline_exp.xs("Central", level="caso")[cols], "MILP, caso central de los 3 escenarios")
brecha_lp = 100 * (central["costo_total_vp_musd"] - central["objetivo_lp_musd"]) / central["costo_total_vp_musd"]
print("Relajación lineal: su costo es menor que el del MILP en", ", ".join(f"{e} {v:.1f}%" for e, v in brecha_lp.items()))
central[cols].round(1)

✓ MILP, caso central de los 3 escenarios: coincide con el pipeline (diferencia máxima 2.9e-06)
Relajación lineal: su costo es menor que el del MILP en base 0.7%, estres 0.3%, optimista 0.9%


,costo_bloques_vp_musd,costo_total_vp_musd,no_cubierta_gwh,mw_biomasa_bagazo,mw_diesel_motores,mw_eolica_terrestre,mw_gas_ciclo_abierto,mw_gas_ciclo_combinado,mw_solar_fv
escenario,,,,,,,,,
base,422.9,3764.7,4599.0,40.0,0.0,0.0,600.0,0.0,200.0
estres,809.9,8841.5,11024.9,40.0,0.0,100.0,1200.0,0.0,200.0
optimista,318.8,2731.1,3269.3,0.0,0.0,0.0,500.0,0.0,0.0


**Lectura.** Los tres escenarios coinciden con el pipeline.
- **Base:** 600 MW de ciclo abierto, 200 MW solares y 40 MW de bagazo, desde 2028-2029. Los bloques cuestan 423 MUSD en valor presente. Quedan sin cubrir 4.599 GWh, casi todos en 2026-2028, cuando nada alcanza a llegar.
- **Estrés:** 1.200 MW de ciclo abierto, 200 solares, 100 eólicos y 40 de bagazo, por 810 MUSD; 11.025 GWh sin cubrir.
- **Optimista:** 500 MW de ciclo abierto, por 319 MUSD; 3.269 GWh sin cubrir.
- **Validación:** la relajación lineal cuesta entre 0,3% y 0,9% menos que el MILP, como debe ser: nunca puede costar más. La diferencia es pequeña porque las brechas son grandes frente al tamaño de un bloque.

### 3.5 Sensibilidades

Cada caso cambia un supuesto frente al central y vuelve a resolver los tres escenarios. Los casos prueban cada decisión del modelo: costos, tasa de descuento, cupos, factores firmes (los del PIEG o las medianas observadas en XM, que calculó el pipeline), plazos, el tope solar, la penalidad y el objetivo.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `dict.update(otro)` | Agrega las entradas de otro diccionario | Los 16 casos de sensibilidad | `correr()` los lee de `CASOS` |
| `DataFrame.xs(clave, level)` | Toma una sección de un índice de varios niveles | Un escenario | Para ver los 17 casos de un escenario |

In [46]:
CASOS.update({
    "Costos bajos": {"escenario_costos": "bajo"}, "Costos altos": {"escenario_costos": "alto"},
    "Tasa de descuento 8%": {"r": 0.08}, "Tasa de descuento 12%": {"r": 0.12},
    "Sin eólica nueva": {"cupo": {"eolica_terrestre": 0}},
    "Factores firmes observados en XM": {"factor_firme": "observado"},
    "Plazos un período más largos": {"retraso_periodos": 1},
    "Tope solar del PIEG, referencia (2.315 MW/año)": {"tope_solar": 2315},
    "Tope solar del PIEG, máximo (4.325 MW/año)": {"tope_solar": 4325},
    "Diésel con cupo de 200 MW (supuesto)": {"cupo": {"diesel_motores": 200}},
    "Sin bagazo": {"cupo": {"biomasa_bagazo": 0}}, "Bagazo con cupo de 70 MW (Asocaña)": {"cupo": {"biomasa_bagazo": 70}},
    "Penalidad: precio de escasez superior": {"criterio_penalidad": "precio_escasez_superior"},
    "Penalidad: precio de escasez inferior": {"criterio_penalidad": "precio_escasez_inferior"},
    "Penalidad sin tope (10 veces)": {"factor_penalidad": 10},
    "Solo costo fijo (lógica del cargo)": {"objetivo": "solo_fijo"}})

sens = pd.DataFrame([correr(e, caso)[0] for e in ESCENARIOS for caso in CASOS]).set_index(["escenario", "caso"])
comprobar(sens[cols], pipeline_exp[cols], f"MILP, {len(sens)} corridas (3 escenarios × {len(CASOS)} casos)")
sens.xs("estres", level="escenario")[["costo_bloques_vp_musd", "costo_total_vp_musd", "no_cubierta_gwh"]].round(0)

✓ MILP, 51 corridas (3 escenarios × 17 casos): coincide con el pipeline (diferencia máxima 6.7e-06)


,costo_bloques_vp_musd,costo_total_vp_musd,no_cubierta_gwh
caso,,,
Central,810.0,8842.0,11025.0
Costos bajos,765.0,8797.0,11025.0
Costos altos,1245.0,9277.0,11025.0
Tasa de descuento 8%,806.0,8923.0,11025.0
Tasa de descuento 12%,814.0,8764.0,11025.0
Sin eólica nueva,812.0,8844.0,11025.0
Factores firmes observados en XM,811.0,8842.0,11025.0
Plazos un período más largos,524.0,13956.0,19513.0
"Tope solar del PIEG, referencia (2.315 MW/año)",939.0,8332.0,10111.0


**Lectura.** Las 51 corridas coinciden con el pipeline. En el estrés:
- **Los plazos son lo que más pesa.** Si cada tecnología llega un período después, quedan 19.513 GWh sin cubrir y el costo total sube 58%, a 13.956 MUSD.
- **Antes de 2028 solo hay dos palancas.** Un cupo de diésel de 200 MW, un caso hipotético porque no hay proyectos en curso, deja 9.479 GWh sin cubrir; con el tope solar más alto del PIEG (4.325 MW/año) quedan 7.828.
- **Los costos y la tasa de descuento** mueven el costo de los bloques entre 765 y 1.245 MUSD, sin cambiar la energía sin cubrir.
- **La penalidad** cambia sobre todo el costo total: con el precio de escasez superior el portafolio no cambia, y con el inferior casi no cambia (sale la eólica y quedan 260 GWh más sin cubrir).

**Lo que sigue.** El método ya dio sus cifras: la brecha de cada escenario y período, con El Niño y con un año normal, el portafolio de mínimo costo y su sensibilidad a cada supuesto. La sección 4 las muestra en figuras y responde la pregunta.

<a id="resultados" name="resultados"></a>
## 4. Resultados y visualización

La sección 3 calculó la brecha y la expansión. Aquí se muestran en siete figuras interactivas y dos animaciones, de la demanda a la expansión, y al final se responde la pregunta.

Al pasar el cursor sobre una figura se ven sus valores. Las animaciones muestran cómo se forman los resultados en el tiempo: la brecha del caso base mes a mes (4.2) y la cobertura de los tres escenarios período a período (4.6). La primera celda fija los nombres y colores que comparten.

In [47]:
NOMBRES_ESCENARIO = {"base": "Base: demanda media, con retrasos", "estres": "Estrés: demanda alta, con retrasos",
                     "optimista": "Optimista: demanda media, sin retrasos"}
COLORES_ESCENARIO = {"base": "#2a78d6", "estres": "#eb6834", "optimista": "#1baf7a"}
NOMBRES_TECNOLOGIA = {"solar_fv": "Solar FV", "eolica_terrestre": "Eólica", "biomasa_bagazo": "Bagazo",
                      "gas_ciclo_abierto": "Gas, ciclo abierto", "gas_ciclo_combinado": "Gas, ciclo combinado",
                      "diesel_motores": "Motores diésel"}
COLORES_TECNOLOGIA = {"Solar FV": "#eda100", "Eólica": "#1baf7a", "Bagazo": "#008300", "Gas, ciclo abierto": "#eb6834",
                      "Gas, ciclo combinado": "#6250d6", "Motores diésel": "#e87ba4"}
LEYENDA = dict(orientation="h", yanchor="top", y=-0.22, x=0, title=None)

### 4.1 La demanda: observada y proyectada

La demanda observada de XM desde 2018 y la proyección de la UPME para la demanda total, con su banda de 68%. El borde superior de la banda es la demanda del escenario de estrés.

In [48]:
obs = sistema[sistema["mes_completo"] & (sistema["mes"] >= "2018-01-01")]
upme = (demanda_upme.query("definicion == @DEFINICION and mes <= '2030-12-01'")
                    .pivot(index="mes", columns="escenario", values="energia_gwh"))
fig = go.Figure()
fig.add_trace(go.Scatter(x=upme.index, y=upme["ic68_sup"], line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=upme.index, y=upme["ic68_inf"], fill="tonexty", fillcolor="rgba(42,120,214,0.18)",
                         line=dict(width=0), name="UPME, banda de 68%"))
fig.add_trace(go.Scatter(x=upme.index, y=upme["medio"], name="UPME, escenario medio", line=dict(color="#2a78d6", width=2)))
fig.add_trace(go.Scatter(x=obs["mes"], y=obs["demanda_sin_gwh"], name="Observada (XM, DemaSIN)",
                         line=dict(color="#1f1f1f", width=2)))
fig.add_vrect(x0="2026-12-01", x1="2030-11-30", fillcolor="#e9e7e2", opacity=0.5, layer="below", line_width=0,
              annotation_text="Horizonte: dic-2026 a nov-2030", annotation_position="top left")
fig.update_layout(title="Demanda mensual de energía del SIN: observada y proyectada (demanda total de la UPME)",
                  xaxis_title="Mes", yaxis_title="GWh por mes", height=440, legend=LEYENDA, hovermode="x unified")
fig.show()

**Lectura.** La demanda observada creció de 66.175 GWh en 2015 a 84.049 en 2025, un 27% (2,4% al año), con una sola caída: la de 2020, por la pandemia (−2,1% frente a 2019). La proyección media de la demanda total sube más rápido: 100.946 GWh en 2030, un 20% más que en 2025 (3,7% al año), porque suma los grandes consumidores especiales y la movilidad eléctrica que todavía no se ven en la serie observada. En 2030 la banda de 68% va de 95.583 a 106.358 GWh, cerca de ±5% alrededor del medio. Por eso el escenario medio no es optimista y la demanda alta es de estrés, no la base (sección 2.5).

### 4.2 La brecha de energía firme

La brecha de cada período en los tres escenarios de la sección 3.3. Los marcadores vacíos son la brecha del año de hidrología normal, y los rombos, las cifras del Boletín 364 de XM para cada período, tomadas de la prensa.

In [49]:
ref = leer("referencias_suficiencia.csv").query("concepto == 'brecha_firme' and fuente == 'XM' and periodo in @PERIODOS")
fig = px.bar(balances.assign(escenario=balances["escenario"].map(NOMBRES_ESCENARIO)), x="periodo", y="brecha_gwh",
             color="escenario", barmode="group", color_discrete_map={NOMBRES_ESCENARIO[k]: v for k, v in COLORES_ESCENARIO.items()},
             custom_data=["brecha_pct"], labels={"periodo": "Período del Cargo por Confiabilidad (diciembre a noviembre)",
                                                "brecha_gwh": "GWh por período (positivo = falta)", "escenario": "Escenario"},
             title="Brecha de energía firme por período: demanda menos oferta firme")
fig.update_traces(hovertemplate="%{y:,.0f} GWh (%{customdata[0]:.1f}% de la demanda)<extra></extra>")
fig.add_scatter(x=ref["periodo"], y=ref["valor"], mode="markers", name="XM, Boletín 364 (prensa)",
                marker=dict(symbol="diamond", size=12, color="#1f1f1f"))
for e, simbolo in [("base", "circle-open"), ("estres", "square-open")]:            # el año de hidrología normal
    fig.add_scatter(x=PERIODOS, y=ano_normal[("Año normal", e)], mode="markers",
                    name=f"Año normal, {NOMBRES_ESCENARIO[e].split(':')[0].lower()}",
                    marker=dict(symbol=simbolo, size=11, color=COLORES_ESCENARIO[e], line=dict(width=2)))
fig.update_layout(height=440, legend=LEYENDA, separators=",.")
fig.show()

**Lectura.** La brecha crece hasta 2028-2029 en el base y hasta 2029-2030 en el estrés, y no se cierra en ningún escenario: ni siquiera el optimista, sin retrasos, alcanza la demanda media. El Boletín 364 de XM estima 7.669 GWh para 2026-2027, más que el propio estrés (4.493). La diferencia es de medición: XM no les da a las plantas menores un factor firme, cuenta la ENFICC de unidades que no operan y usa la ENFICC verificada de SIMEM. En el pipeline, esas tres diferencias explican la mayor parte de la distancia; queda sin explicar el 1,6% de la demanda. Los marcadores vacíos son el año de hidrología normal: ahí la brecha es negativa en todos los períodos, es decir, sobra energía firme, aun con la demanda alta.

#### En movimiento: la brecha del caso base, mes a mes

La figura anterior suma cada período en una barra. Esta animación muestra cómo se forma esa suma: la demanda media de cada mes, por día, frente a la oferta firme diaria del caso base (sección 3.3), de diciembre de 2026 a noviembre de 2030.

El texto lleva la cuenta de la brecha acumulada del período, que vuelve a cero cada diciembre. Se reproduce con el botón ▶ o se recorre con el control deslizante.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `px.line(..., animation_frame=)` | Dibuja una figura por cada valor de la columna y las une en una animación, con botón y control deslizante | `animation_frame="cuadro"`; `range_x` y `range_y` fijos | Cada cuadro trae los meses hasta ese mes, así que la línea crece; con los ejes fijos, la escala no salta de un cuadro a otro |
| `go.Scatter(mode="text")` | Una traza que solo muestra texto | Una traza por cuadro, añadida a `fig.frames` | El texto cambia con cada cuadro, igual que las líneas |
| `layout.updatemenus` | Los botones de la animación | La duración de cada cuadro | Un mes cada 0,15 segundos: los 48 meses en unos 7 segundos |

In [50]:
# Los 48 meses del horizonte: demanda media y oferta firme del caso base, por día
meses = demanda_upme.query("definicion == @DEFINICION and escenario == 'medio'").copy()
meses["periodo"], meses["dias"] = periodo_cxc(meses["mes"]), meses["mes"].dt.days_in_month
meses = meses[meses["periodo"].isin(PERIODOS)].sort_values("mes").reset_index(drop=True)
firme = balance("base").set_index("periodo")
meses["demanda_dia"] = meses["energia_gwh"] / meses["dias"]
meses["firme_dia"] = meses["periodo"].map(firme["oferta_firme_gwh"] / firme["dias"])
meses["acumulado_gwh"] = ((meses["demanda_dia"] - meses["firme_dia"]) * meses["dias"]).groupby(meses["periodo"]).cumsum()
cierre = meses.groupby("periodo")["acumulado_gwh"].last()
assert (cierre - firme["brecha_gwh"]).abs().max() < 0.5, "la suma de los meses no da la brecha del período"
print("Brecha acumulada a noviembre, igual a la de la sección 3.3 (GWh):", cierre.round(0).to_dict())

cuadros = pd.concat([meses.iloc[:i + 1].assign(cuadro=f"{m:%Y-%m}") for i, m in enumerate(meses["mes"])])
largo = cuadros.melt(id_vars=["cuadro", "mes"], value_vars=["demanda_dia", "firme_dia"], var_name="serie", value_name="gwh_dia")
SERIES = {"demanda_dia": "Demanda media de la UPME", "firme_dia": "Oferta firme del caso base"}
largo["serie"] = largo["serie"].map(SERIES)


def rotulo(fila):
    """Texto de un cuadro: brecha acumulada del período hasta ese mes."""
    palabra = "faltan" if fila.acumulado_gwh > 0 else "sobran"
    gwh = f"{abs(fila.acumulado_gwh):,.0f}".replace(",", ".")
    return f"Período {fila.periodo}, hasta {fila.mes:%Y-%m}: {palabra} {gwh} GWh firmes"

Brecha acumulada a noviembre, igual a la de la sección 3.3 (GWh): {'2026-2027': 1770.0, '2027-2028': 2809.0, '2028-2029': 5168.0, '2029-2030': 4585.0}


In [51]:
rango_y = [meses[["demanda_dia", "firme_dia"]].min().min() * 0.97, meses[["demanda_dia", "firme_dia"]].max().max() * 1.04]
fig = px.line(largo, x="mes", y="gwh_dia", color="serie", animation_frame="cuadro", markers=True,
              color_discrete_map={SERIES["demanda_dia"]: "#2a78d6", SERIES["firme_dia"]: "#1f1f1f"},
              range_x=[meses["mes"].min() - pd.Timedelta(days=20), meses["mes"].max() + pd.Timedelta(days=20)],
              range_y=rango_y, labels={"mes": "Mes", "gwh_dia": "GWh por día", "serie": "Serie", "cuadro": "Mes"},
              title="La brecha del caso base, mes a mes: demanda media frente a oferta firme (GWh por día)")
texto = dict(mode="text", x=[meses["mes"].min()], y=[rango_y[1] * 0.995], textposition="bottom right",
             showlegend=False, hoverinfo="skip", textfont=dict(size=14, color="#1f1f1f"))
fig.add_trace(go.Scatter(text=[rotulo(meses.iloc[0])], **texto))
for cuadro, (_, fila) in zip(fig.frames, meses.iterrows()):
    cuadro.data += (go.Scatter(text=[rotulo(fila)], **texto),)
fig.update_traces(hovertemplate="%{x|%Y-%m}: %{y:,.1f} GWh por día<extra></extra>", selector=dict(mode="lines+markers"))
for cuadro in fig.frames:
    for traza in cuadro.data[:2]:
        traza.hovertemplate = "%{x|%Y-%m}: %{y:,.1f} GWh por día<extra></extra>"
fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 150
fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 0
fig.layout.sliders[0].currentvalue.prefix = "Mes: "
fig.update_layout(height=540, legend=dict(LEYENDA, y=1.0, yanchor="bottom"), margin=dict(t=110), xaxis_title=None,
                  xaxis_tickformat="%Y-%m", separators=",.")       # leyenda arriba: abajo van los controles
fig.show()

**Lectura.** La brecha no sale de unos pocos meses de demanda alta: se acumula casi todo el año. En 2026-2027 la oferta física del caso base (parque que opera, menores, entradas sin subasta y proyectos con OEF con retrasos) es de 251,3 GWh por día, y la demanda media pasa de ella en 11 de los 12 meses; solo enero, el mes de menor demanda en todos los períodos, queda por debajo (246,4). Lo mismo ocurre en 2027-2028, y desde 2028-2029 la demanda supera la oferta física en todos los meses. En 2029-2030 la oferta física sube a 263,5 GWh por día con la entrada de proyectos con OEF, y la brecha baja, pero la demanda la sigue superando en todos los meses: va de 265,5 GWh por día en enero a 283,7 en febrero. Cada noviembre la cuenta llega exactamente a la brecha de la figura anterior (1.770, 2.809, 5.168 y 4.585 GWh): las dos vistas salen de los mismos datos.

### 4.3 De dónde sale la energía firme

La oferta firme del caso base por componente, frente a las dos demandas. Muestra qué parte de la oferta es el parque que ya existe y qué parte depende de que lleguen los proyectos nuevos.

In [52]:
base = balance("base").set_index("periodo")
componentes = {"enficc_gwh": "Plantas con ENFICC", "menores_gwh": "Menores (factor firme)",
               "sin_subasta_gwh": "Entradas sin subasta", "nuevas_gwh": "Proyectos con OEF (con retrasos)"}
colores = ["#2a78d6", "#9cc3ec", "#008300", "#eda100"]
fig = go.Figure()
for (col, nombre), color in zip(componentes.items(), colores):
    fig.add_trace(go.Bar(x=PERIODOS, y=base[col], name=nombre, marker_color=color,
                         hovertemplate=f"{nombre}: %{{y:,.0f}} GWh<extra></extra>"))
for escenario, estilo in [("media", "solid"), ("alta", "dash")]:
    y = demanda.query("escenario_demanda == @escenario").set_index("periodo").loc[PERIODOS, "demanda_gwh"]
    fig.add_trace(go.Scatter(x=PERIODOS, y=y, name=f"Demanda {escenario}", mode="lines+markers",
                             line=dict(color="#1f1f1f", dash=estilo, width=2)))
fig.update_layout(barmode="stack", title="Oferta firme del caso base por componente frente a la demanda",
                  xaxis_title="Período del Cargo por Confiabilidad", yaxis_title="GWh por período",
                  yaxis_range=[80_000, None], height=460, legend=LEYENDA, separators=",.")
fig.show()

**Lectura.** El eje vertical empieza en 80.000 GWh para que se vean las diferencias. El parque que ya existe pone casi toda la energía firme: unos 86.700 GWh de ENFICC y 5.000 de las menores por período. Lo que crece son los proyectos con OEF (hasta unos 4.000 GWh en 2029-2030, con retrasos) y las entradas sin subasta (unos 600). Pero la demanda media crece más rápido: de 93.508 a 100.759 GWh, unos 7.250 GWh en cuatro períodos, frente a unos 4.400 de oferta nueva. La demanda alta queda todavía más lejos.

### 4.4 Cuánto cuesta un MWh firme

El costo anual por MWh firme de cada bloque candidato (sección 3.2), en el escenario central de costos, con el rango entre el bajo y el alto. Las líneas son los precios de cierre de la subasta del cargo 2029-2030, que pagan solo la firmeza.

In [53]:
costos = pd.concat([bloques(e).assign(escenario_costos=e) for e in ["bajo", "central", "alto"]])
c = costos.pivot(index="tecnologia", columns="escenario_costos", values="costo_firme_usd_mwh").sort_values("central")
nombres = c.index.map(NOMBRES_TECNOLOGIA)
fig = go.Figure(go.Bar(x=c["central"], y=nombres, orientation="h", marker_color="#2a78d6", name="Escenario central",
                       error_x=dict(type="data", symmetric=False, array=c["alto"] - c["central"],
                                    arrayminus=c["central"] - c["bajo"], color="#555"),
                       hovertemplate="%{y}: %{x:.0f} USD/MWh firme<extra></extra>"))
cierres = leer("referencias_suficiencia.csv").set_index("concepto")["valor"]
for concepto, nombre, estilo in [("precio_cierre_cxc_renovables", "Cierre del cargo 2029-2030, renovables", "dash"),
                                 ("precio_cierre_cxc_gas_liquidos", "Cierre del cargo 2029-2030, gas y líquidos", "dot")]:
    fig.add_trace(go.Scatter(x=[cierres[concepto]] * 2, y=[nombres[0], nombres[-1]], mode="lines",
                             name=f"{nombre}: {cierres[concepto]:.1f}", line=dict(color="#eb6834", dash=estilo)))
fig.update_layout(title="Costo anual por MWh de energía firme de cada bloque candidato (USD de 2025)",
                  xaxis_title="USD por MWh firme (costo fijo + costo variable esperado)", yaxis_title="Bloque candidato",
                  height=420, legend=LEYENDA, separators=",.")
fig.show()
fijo = bloques(objetivo="solo_fijo").set_index("tecnologia")["costo_firme_usd_mwh"]
print(f"Solo con el costo fijo: ciclo abierto {fijo['gas_ciclo_abierto']:.1f} y bagazo {fijo['biomasa_bagazo']:.1f} USD/MWh firme")

Solo con el costo fijo: ciclo abierto 16.8 y bagazo 20.5 USD/MWh firme


**Lectura.** Ordenados por costo por MWh firme: bagazo (21 USD), ciclo combinado (39), motores diésel (47), ciclo abierto (51), solar (95) y eólica (145). Las líneas no se comparan directamente con las barras: el cierre de la subasta del cargo (22,0 USD/MWh para renovables y 16,4 para gas y líquidos) paga solo la firmeza, porque la planta vende aparte su energía, mientras que las barras cubren todo el costo de la planta. Con solo el costo fijo, el ciclo abierto cuesta 16,8 USD por MWh firme, casi el cierre para gas, y el bagazo 20,5, cerca del cierre para renovables. El modelo reproduce, en esencia, los precios del cargo.

### 4.5 El portafolio de mínimo costo

Los MW nuevos en operación por tecnología y período que escoge el MILP en cada escenario, en el caso central (sección 3.4).

In [54]:
filas = []
for escenario, (_, det) in corridas.items():
    for k, tec in enumerate(det["bl"]["tecnologia"]):
        for t, periodo in enumerate(PERIODOS):
            filas.append({"escenario": NOMBRES_ESCENARIO[escenario], "periodo": periodo, "tecnologia": NOMBRES_TECNOLOGIA[tec],
                          "mw": det["x"][k, t] * det["bl"].at[k, "tamano_mw"], "firme_gwh": det["x"][k, t] * det["pr"]["F"][k, t]})
portafolio = pd.DataFrame(filas)
fig = px.bar(portafolio[portafolio["mw"] > 0], x="periodo", y="mw", color="tecnologia", facet_col="escenario",
             color_discrete_map=COLORES_TECNOLOGIA, category_orders={"periodo": PERIODOS, "escenario": list(NOMBRES_ESCENARIO.values())},
             labels={"periodo": "Período del cargo", "mw": "MW nuevos en operación", "tecnologia": "Tecnología"},
             title="Expansión de mínimo costo: MW nuevos en operación por escenario (caso central)")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_xaxes(categoryorder="array", categoryarray=PERIODOS)
fig.update_layout(height=460, legend=LEYENDA, separators=",.")
fig.show()

**Lectura.** Nada nuevo entra antes de 2028-2029 en ningún escenario: en 2026-2027 no llega ninguna tecnología, y en 2027-2028 la solar con OEF ya llena el tope. Desde 2028-2029 el modelo escoge sobre todo gas de ciclo abierto: 600 MW en el base, 1.200 en el estrés y 500 en el optimista. En el base y el estrés agrega bagazo hasta su cupo (40 MW) y 200 MW solares; en el estrés, además, 100 MW eólicos en 2029-2030. El ciclo combinado no entra en ningún escenario: llega en 2029-2030, cuando lo ya construido casi cubre la brecha. En el estrés, el gas usa 1.200 de sus 1.400 MW de cupo y depende de GNL importado.

### 4.6 Con qué se cierra la brecha del estrés

La energía firme que aportan los bloques nuevos frente a la brecha del estrés, y la energía que ningún bloque alcanza a cubrir a tiempo.

In [55]:
_, det = corridas["estres"]
cobertura = portafolio[(portafolio["escenario"] == NOMBRES_ESCENARIO["estres"]) & (portafolio["firme_gwh"] > 0)]
fig = px.bar(cobertura, x="periodo", y="firme_gwh", color="tecnologia", color_discrete_map=COLORES_TECNOLOGIA,
             category_orders={"periodo": PERIODOS},
             labels={"periodo": "Período del Cargo por Confiabilidad", "firme_gwh": "GWh por período", "tecnologia": "Tecnología"},
             title="¿Con qué se cierra la brecha? Estrés: demanda alta, con retrasos (caso central)")
fig.add_bar(x=PERIODOS, y=det["u"], name="Energía no cubierta (ningún bloque llega a tiempo)",
            marker=dict(color="#c9c7bf", pattern_shape="/"))
fig.add_scatter(x=PERIODOS, y=brecha_de("estres")["brecha_gwh"], name="Brecha firme (sección 3.3)",
                mode="lines+markers", line=dict(color="#1f1f1f", width=2))
fig.update_xaxes(categoryorder="array", categoryarray=PERIODOS)
fig.update_layout(barmode="stack", height=460, legend=LEYENDA, separators=",.")
fig.show()

**Lectura.** En el estrés, toda la brecha de 2026-2027 y 2027-2028 (4.493 y 6.532 GWh) queda sin cubrir: ningún bloque alcanza a llegar. Desde 2028-2029 los bloques nuevos la cubren, con un pequeño exceso porque son indivisibles: 9.764 GWh firmes para una brecha de 9.740, y 10.000 para una de 9.933 en 2029-2030. La energía no cubierta no es un costo de construir: es la exposición al racionamiento que deja el calendario.

#### En movimiento: los tres escenarios, período a período

La misma cuenta para los tres escenarios a la vez. Cada cuadro es un período: la barra apila la energía firme de los bloques nuevos y la energía no cubierta, y el rombo marca la brecha que tienen que cubrir. Al avanzar se ve cuándo llegan los bloques y cuánto le sobra o le falta a cada escenario.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `px.bar(..., animation_frame=)` | Una figura de barras por período, unidas en una animación | `animation_frame="periodo"`, `range_y` fijo | Con la escala fija, la altura de las barras se compara entre períodos |
| `pd.MultiIndex.from_product()` + `reindex(fill_value=0)` | Completa la tabla con todas las combinaciones de escenario, período y componente | Los componentes con algún GWh | Si a un cuadro le falta una combinación, plotly deja la barra del cuadro anterior |
| `fig.frames[i].data +=` | Añade una traza a cada cuadro | Los rombos de un `px.scatter` con el mismo `animation_frame` | Así el rombo de la brecha se mueve con las barras |

In [56]:
# Energía firme de los bloques nuevos y energía no cubierta, por escenario, período y componente
no_cubierta = [{"escenario": NOMBRES_ESCENARIO[e], "periodo": p, "tecnologia": "Energía no cubierta", "firme_gwh": u}
               for e, (_, det) in corridas.items() for p, u in zip(PERIODOS, det["u"])]
largo = pd.concat([portafolio, pd.DataFrame(no_cubierta)]).groupby(["escenario", "periodo", "tecnologia"])["firme_gwh"].sum()
totales = largo.groupby(level="tecnologia").sum()
usadas = [c for c in [*COLORES_TECNOLOGIA, "Energía no cubierta"] if totales.get(c, 0) > 0]
rejilla = pd.MultiIndex.from_product([NOMBRES_ESCENARIO.values(), PERIODOS, usadas], names=largo.index.names)
largo = largo.reindex(rejilla, fill_value=0).reset_index()
brechas = pd.concat([brecha_de(e).reset_index().assign(escenario=NOMBRES_ESCENARIO[e]) for e in ESCENARIOS])
tope = max(largo.groupby(["escenario", "periodo"])["firme_gwh"].sum().max(), brechas["brecha_gwh"].max()) * 1.1
orden = {"periodo": PERIODOS, "escenario": list(NOMBRES_ESCENARIO.values()), "tecnologia": usadas}
largo.pivot_table(index="periodo", columns="escenario", values="firme_gwh", aggfunc="sum").round(0)

escenario,"Base: demanda media, con retrasos","Estrés: demanda alta, con retrasos","Optimista: demanda media, sin retrasos"
periodo,,,
2026-2027,1770.0,4493.0,1770.0
2027-2028,2809.0,6532.0,1499.0
2028-2029,5168.0,9764.0,3942.0
2029-2030,5147.0,10000.0,3942.0


In [57]:
fig = px.bar(largo, x="escenario", y="firme_gwh", color="tecnologia", animation_frame="periodo", category_orders=orden,
             color_discrete_map=COLORES_TECNOLOGIA | {"Energía no cubierta": "#c9c7bf"}, range_y=[0, tope],
             labels={"escenario": "Escenario", "firme_gwh": "GWh por período", "tecnologia": "Componente", "periodo": "Período"},
             title="¿Con qué se cierra la brecha? Los tres escenarios, período a período (caso central)")
rombos = px.scatter(brechas, x="escenario", y="brecha_gwh", animation_frame="periodo", category_orders=orden)
estilo = dict(name="Brecha firme (sección 3.3)", showlegend=True, hovertemplate="Brecha: %{y:,.0f} GWh<extra></extra>",
              marker=dict(symbol="diamond", size=16, color="#1f1f1f", line=dict(width=2, color="#ffffff")))
fig.add_trace(rombos.data[0].update(**estilo))
for cuadro, rombo in zip(fig.frames, rombos.frames):
    for traza in cuadro.data:
        traza.hovertemplate = f"{traza.name}: %{{y:,.0f}} GWh<extra></extra>"
    cuadro.data += (rombo.data[0].update(**estilo),)
fig.update_traces(hovertemplate="%{fullData.name}: %{y:,.0f} GWh<extra></extra>", selector=dict(type="bar"))
fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 1800
fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 600
fig.layout.sliders[0].currentvalue.prefix = "Período: "
fig.update_layout(barmode="stack", height=540, legend=dict(LEYENDA, y=1.0, yanchor="bottom"), margin=dict(t=110),
                  xaxis_title=None, yaxis_tickformat=",.0f", separators=",.")   # leyenda arriba: abajo, los controles
fig.show()

**Lectura.** La animación muestra el calendario. En 2026-2027 y 2027-2028 no entra ningún bloque en ningún escenario, y toda la brecha queda sin cubrir: 1.770 y 2.809 GWh en el base, 4.493 y 6.532 en el estrés. En 2028-2029 llegan los bloques, y el ciclo abierto pone casi toda la energía firme: 4.730 GWh en el base, 9.461 en el estrés y 3.942 en el optimista. En el base quedan 20 GWh sin cubrir ese período: al MILP le sale más barato pagar la penalidad por ellos que construir otro bloque. En 2029-2030 las barras del base y del optimista quedan por encima del rombo: lo construido para 2028-2029 sobra cuando la brecha baja (en el optimista, de 3.764 a 2.049 GWh). Solo el estrés construye en 2029-2030: 100 MW eólicos y otros 100 solares, que suman 237 GWh firmes, para una brecha que sube a 9.933 GWh.

### 4.7 Qué cambia con los supuestos

Los MW de cada tecnología en 2029-2030 y el costo en valor presente de cada caso de sensibilidad (sección 3.5), en el estrés. El costo se separa en el de los bloques y el de la penalidad de la energía no cubierta.

In [58]:
e = sens.xs("estres", level="escenario").iloc[::-1]          # el caso central arriba
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.03, column_widths=[0.55, 0.45],
                    subplot_titles=("MW nuevos en operación en 2029-2030", "Costo en valor presente (MUSD de 2025)"))
for tec, nombre in NOMBRES_TECNOLOGIA.items():
    if e[f"mw_{tec}"].max() > 0:
        fig.add_trace(go.Bar(y=e.index, x=e[f"mw_{tec}"], orientation="h", name=nombre,
                             marker_color=COLORES_TECNOLOGIA[nombre]), row=1, col=1)
fig.add_trace(go.Bar(y=e.index, x=e["costo_bloques_vp_musd"], orientation="h", name="Costo de los bloques",
                     marker_color="#555"), row=1, col=2)
fig.add_trace(go.Bar(y=e.index, x=e["costo_total_vp_musd"] - e["costo_bloques_vp_musd"], orientation="h",
                     name="Penalidad de la energía no cubierta", marker=dict(color="#c9c7bf", pattern_shape="/")), row=1, col=2)
fig.update_layout(barmode="stack", title="Sensibilidades del MILP en el estrés: capacidad y costo de cada caso",
                  height=640, legend=dict(LEYENDA, y=-0.1), margin=dict(l=280), separators=",.")
fig.update_xaxes(title_text="MW", row=1, col=1)
fig.update_xaxes(title_text="MUSD de 2025", row=1, col=2)
fig.update_yaxes(title_text="Caso de sensibilidad", row=1, col=1)
fig.show()

**Lectura.** El portafolio es estable. El gas de ciclo abierto pone entre 1.000 y 1.300 MW en todos los casos salvo uno: con el tope solar máximo del PIEG baja a 800, porque entran 3.100 MW solares, que llegan antes. Lo que cambia la energía sin cubrir es el calendario: con plazos un período más largos sube a 19.513 GWh y la penalidad domina el costo; con el tope solar del PIEG baja a 10.111 GWh (referencia) o 7.828 (máximo), y con 200 MW de diésel, a 9.479. Los costos, la tasa de descuento y los cupos de bagazo y eólica apenas mueven los MW. La penalidad sin tope (10 veces) no cambia el portafolio: solo multiplica el costo de la energía que de todos modos no alcanza a cubrirse.

Para mover estos supuestos sin correr el notebook, el proyecto tiene una [app interactiva](https://cterrassa.github.io/suficiencia-sin/app/) que corre en el navegador.

### 4.8 Respuesta a la pregunta

Aquí se juntan las piezas de las secciones 3 y 4 para responder la pregunta de la sección 1, en el orden en que la plantea: cuánta energía falta, qué se construye y cuánto cuesta.

**La respuesta.** En el caso base, con la demanda media de la UPME y los retrasos observados en la subasta de 2019, la expansión de mínimo costo es de 840 MW que entran en 2028-2029: 600 MW de gas de ciclo abierto, 200 MW solares y 40 MW de bagazo, por 423 MUSD en valor presente. Esa cifra es el valor presente de sus costos anuales dentro del horizonte, no la inversión, que ronda los 803 MUSD. Desde ese período, la energía firme cubre la demanda; antes no hay con qué, y quedan 4.599 GWh sin cubrir. Sus tres partes:

**1. Cuánta energía firme falta y cuándo.** Falta desde 2026-2027: 1.770 GWh, el 1,9% de la demanda. La brecha sube a 2.809 GWh en 2027-2028, llega a su máximo en 2028-2029 (5.168 GWh, el 5,2%) y queda en 4.585 (4,6%) en 2029-2030. Con la demanda alta llega a 9.933 GWh (9,4%); sin retrasos baja, pero no se cierra en ningún período. Es la brecha del año seco: la energía firme mide lo que el parque garantiza con El Niño. En un año de hidrología normal no falta: en el base sobran entre 5.682 y 9.080 GWh por período, y con la demanda alta, alrededor de 1.000 GWh en 2028-2030 (sección 3.3).

**2. Qué se construye y cuándo.** Nada nuevo llega antes de diciembre de 2028: en 2027-2028 solo podría entrar la solar, y los proyectos solares con OEF ya ocupan el tope de construcción. Desde 2028-2029, la expansión es sobre todo gas de ciclo abierto: de lo que puede llegar ese período, es lo más barato por MWh firme después del bagazo, que solo puede sumar 40 MW, y el ciclo combinado, más barato aún, no llega antes de 2029-2030. En el estrés, el portafolio sube a 1.200 MW de ciclo abierto, 200 solares, 100 eólicos y 40 de bagazo.

**3. Cuánto cuesta y qué queda sin cubrir.** Los bloques cuestan 423 MUSD en valor presente en el base y 810 en el estrés. La energía que no alcanza a cubrirse es, casi toda, la de 2026-2028, cuando nada nuevo puede llegar: 4.599 GWh en el base y 11.025 en el estrés.

**Qué cambia la respuesta.** El plazo: si cada tecnología llega un período más tarde, la energía sin cubrir del estrés pasa de 11.025 a 19.513 GWh. Los costos y la tasa de descuento mueven poco el portafolio.

**El riesgo.** No es el costo: el costo fijo del ciclo abierto por MWh firme (16,8 USD) es casi el cierre de la subasta del cargo para gas (16,4). El riesgo está en si los proyectos llegan (de la OEF nueva de la subasta de 2019, el 47% entró tarde o no ha entrado) y en si las térmicas tienen el combustible que respalda su ENFICC, algo que el balance da por hecho.

**Con la brecha como distribución.** Esta respuesta usa el valor esperado de la brecha. La sección 5 la trata como una distribución: el plan de 840 MW cubre 2029-2030 en la mitad de los casos (53%), y cubrirlo con 80% de confianza pide 1.340 MW y 755 MUSD.

### 4.9 Limitaciones

La respuesta depende de supuestos que conviene tener presentes al leer las cifras. Estas son sus limitaciones, por tipo.

**Supuestos.**
- **Energía, no potencia.** El balance es de energía firme por período; no hay un balance hora a hora, y por eso las baterías no entran al modelo.
- **El combustible se da por hecho.** El balance supone que las térmicas tienen el gas que respalda su ENFICC; buena parte depende de GNL importado, y el modelo limita los MW de gas, no el combustible.
- **Un bloque por tecnología,** sin economías de escala ni ubicación. La red solo entra como cupo, y el único tope de construcción es el solar, tomado del récord observado y no de un estudio de la cadena de suministro.
- **El año normal es una aproximación.** La hidráulica del año normal es la que generó, por unidad de ENFICC, en los períodos normales observados desde 2010. Como en un año normal esa generación la limita la demanda, es una cota baja de lo que podría generar, y con seis períodos normales la mediana es frágil. Las plantas menores mantienen su factor firme también en el año normal.

**Sesgos.**
- **La muestra de retrasos es pequeña y no se parece a los proyectos que faltan.** Sale de una sola subasta, con nueve plantas nuevas, y el 53% que entró a tiempo es casi todo de una sola planta, Tesorito: la muestra equivale a unas tres plantas iguales (ejercicio 3). Además, la cohorte de 2019 era sobre todo eólica en La Guajira, con problemas de transmisión y de consultas previas. Si los proyectos que faltan, en su mayoría solares, se retrasan menos, la brecha estaría sobreestimada.
- **Una cifra es una inferencia nuestra:** la energía firme que suma la obra de Ituango en 2029-2030 (2,17 GWh-día). El ejercicio 2 mide cuánto cambia la respuesta si no llega.

**Alcance de la inferencia.**
- **El cálculo de las secciones 3 y 4 es determinista:** valores esperados para tres escenarios. La sección 5 muestra que la incertidumbre de la brecha es mayor que la brecha, y sus percentiles dependen de los supuestos de la simulación: no son intervalos de confianza de un modelo estimado.
- **Algunos insumos vienen procesados.** Las plantas del mes de referencia, los proyectos con OEF y los costos los armó el pipeline del proyecto con descargas que tardan horas; este notebook rehace el método a partir de ellos y comprueba que llega a las mismas cifras.

**Causalidad.**
- **La relación entre la hidrología, El Niño y el precio de bolsa (sección 2.5) es una asociación,** no un efecto causal: la hidrología mueve a la vez los aportes, el embalse y el precio.
- **El MILP es normativo:** dice qué convendría construir al menor costo, no predice qué se construirá ni qué decidirán los inversionistas o el regulador.

**Validez externa.**
- **Las cifras valen para el SIN, con los datos al 4 de noviembre de 2026 y las reglas vigentes de la CREG.** Una nueva subasta, un cambio de reglas o una nueva proyección de la UPME cambian la respuesta. El método se puede llevar a otro sistema que use energía firme; sus números, no.

**Lo que sigue.** La respuesta usa el valor esperado de los retrasos y tres escenarios fijos. La sección 5 le pregunta a una IA cómo mejorar el método e implementa dos mejoras que tratan la brecha como incierta.

<a id="mas-alla" name="mas-alla"></a>
## 5. Uso de IA para ir más allá de la metodología

Le preguntamos a una IA cómo mejorar la metodología de las secciones 2 a 4. Aquí están la consulta y la respuesta completas, lo que va más allá de lo que recomendaron los profesores y dos mejoras implementadas, con su efecto sobre la respuesta.

### 5.1 La consulta

La hicimos el 18 de noviembre de 2026, hacia las 9:50 a. m., en un chat nuevo de Claude (modelo Fable 5.2 Max, de Anthropic), en modo incógnito y sin búsqueda web ni conectores, para que la IA no conociera el proyecto.

El prompt tiene ocho partes: contexto, pregunta, metodología actual, resultados, lo que ya recomendaron los profesores, lo que le pedimos, restricciones y formato de la respuesta. No incluye nuestras limitaciones (sección 4.9), para ver si la IA encontraba sola las debilidades del método. Este es el prompt, tal como se envió:

```text
Contexto
Somos un equipo de tres estudiantes de la Maestría en Economía Aplicada de la Universidad de los Andes (Colombia). Para el trabajo final del Taller de Programación en Python preparamos un tutorial en un Jupyter Notebook que analiza la suficiencia de energía firme del sistema eléctrico colombiano. Queremos su concepto crítico sobre cómo mejorar la metodología.

El Sistema Interconectado Nacional (SIN) depende sobre todo de la generación hidráulica. Para garantizar el suministro en los años secos, como los de El Niño, la CREG creó en 2006 el Cargo por Confiabilidad. Cada planta tiene una energía firme (ENFICC): la que puede garantizar con la hidrología crítica, si es hidráulica, o con su disponibilidad y su combustible, si es térmica. En subastas se asignan obligaciones de energía firme (OEF) por períodos de diciembre a noviembre; los proyectos nuevos que ganan OEF se comprometen a entrar en operación en una fecha.

La pregunta
¿Qué expansión de mínimo costo necesita el SIN para que su energía firme cubra la demanda proyectada por la UPME en cada período del Cargo por Confiabilidad entre 2026-2027 y 2029-2030? El horizonte va de diciembre de 2026 a noviembre de 2030, con datos al 4 de noviembre de 2026. El análisis es de energía firme (año seco); no incluye potencia hora a hora, red de transmisión ni precio de bolsa.

La metodología actual
1. Datos. XM, el operador del sistema (API y SIMEM): ENFICC, OEF y capacidad por planta, y series mensuales de demanda, aportes hídricos y precio desde 2000. UPME: proyección mensual de demanda 2026-2040 (revisión de julio de 2026), Catálogo Tecnológico de costos, Plan Indicativo de Expansión de la Generación (PIEG 2025-2039) y registro de proyectos. DANE: IPC. NOAA: índice ONI. Todo se agrega a períodos de diciembre a noviembre. La demanda es la total: SIN + grandes consumidores especiales + movilidad eléctrica − generación distribuida.
2. Homologación. Las bases de proyectos con OEF, de proyectos registrados en la UPME y de plantas de XM no comparten un código. Las emparejamos con RapidFuzz: normalizamos los nombres, puntuamos los pares dentro de la misma tecnología, descartamos los que tienen atributos incompatibles (capacidad, etapa, fechas) y escogemos el umbral (token_set_ratio >= 86) con 205 pares revisados a mano: precisión de 0,92 y exhaustividad de 0,71.
3. Retrasos. Con la homologación medimos qué pasó con las nueve plantas nuevas de la subasta de 2019, la única cuyos compromisos ya vencieron. Medido por OEF, el 53% entró a tiempo, el 19% entró tarde (20,7 meses en promedio, unos dos períodos) y el 28% no ha entrado. A cada uno de los 41 proyectos con OEF que aún no operan le aplicamos esas fracciones como valor esperado: cuenta el 53% de su OEF desde su período comprometido y el 72% desde dos períodos después; el resto nunca. Si su compromiso ya venció sin entrar, no cuenta.
4. Balance de energía firme. Oferta de cada período = ENFICC de las plantas que operan (237,7 GWh-día en octubre de 2026) + plantas menores sin ENFICC, con su capacidad por el factor firme del PIEG de su tecnología (13,65 GWh-día) + entradas sin subasta al ritmo observado (pequeñas centrales hidráulicas y solares de 1 a 20 MW) + OEF esperada de los proyectos nuevos − retiros. Brecha = demanda − oferta. Tres escenarios deterministas: base (demanda media, con retrasos), estrés (borde superior de la banda de 68% de la UPME, con retrasos) y optimista (demanda media, sin retrasos).
5. Expansión de mínimo costo. Un programa lineal entero mixto (scipy.optimize.milp, con el solver HiGHS). Variables enteras: los bloques que entran por tecnología y período (gas de ciclo abierto, gas de ciclo combinado, solar, eólica, bagazo y motores diésel). Variable continua: la energía firme que queda sin cubrir. Objetivo: minimizar el valor presente de los costos anualizados (inversión con factor de recuperación de capital al 10% real, O&M y costo variable) más una penalidad de 770 USD/MWh por la energía no cubierta (primer tramo del costo de racionamiento de la UPME). Restricciones: la energía firme de los bloques más la no cubierta cubre la brecha de cada período; cada tecnología entra desde su primer período factible; cupos de bagazo (40 MW), diésel (0 MW) y eólica (464 MW), y un tope de construcción solar de 1.400 MW por año, el récord observado. La energía firme de un bloque usa el factor firme del PIEG de su tecnología. Costo anual por MWh firme: bagazo 21 USD, ciclo combinado 39, diésel 47, ciclo abierto 51, solar 95 y eólica 145.
6. Sensibilidades. 17 casos por escenario: costos alto y bajo, tasa de descuento, cupos, factores firmes observados en XM en lugar de los del PIEG, plazos un período más largos, el tope solar del PIEG, 200 MW de diésel, la penalidad igual a los precios de escasez y un objetivo de solo costo fijo.

Resultados principales
- Brecha del caso base: 1.770, 2.809, 5.168 y 4.585 GWh en los cuatro períodos (entre 1,9% y 5,2% de la demanda). En el estrés llega a 9.933 GWh en 2029-2030.
- Antes de diciembre de 2028 no alcanza a llegar nada nuevo, salvo la solar dentro del tope, que ya llenan los proyectos con OEF.
- Expansión del caso base: 600 MW de gas de ciclo abierto, 200 MW solares y 40 MW de bagazo desde 2028-2029, por 423 millones de USD en valor presente; quedan 4.599 GWh sin cubrir, casi todos en 2026-2028. En el estrés: 1.200 MW de ciclo abierto, 200 solares, 100 eólicos y 40 de bagazo, por 810 millones de USD.
- La sensibilidad que más pesa son los plazos: un período más de demora sube la energía sin cubrir del estrés de 11.025 a 19.513 GWh.

Lo que ya nos recomendaron los profesores (por favor no lo repita)
- No usar el precio de bolsa como resultado del modelo: dejarlo como contexto descriptivo.
- Documentar la fuente de los costos de cada tecnología y justificar los supuestos de cada una.
- Delimitar el horizonte.
- Crear una llave común entre las bases y alinear el año calendario con los períodos de diciembre a noviembre.
- Darle más peso a la homologación con RapidFuzz.
- Plantear la pregunta como cuánta energía falta y para cuándo, con un escenario de hidrología normal y uno de El Niño.
- Reducir el alcance del trabajo.

Lo que le pedimos
¿Cómo mejoraría esta metodología para responder mejor la misma pregunta? Sea crítico: señale también las debilidades que vea, aunque no tengan una solución sencilla.

Restricciones
- La pregunta y el horizonte no cambian.
- Solo Python, con los paquetes que ya usamos (pandas, NumPy, SciPy, RapidFuzz, Plotly y requests); sin solvers comerciales.
- Solo datos públicos como los de la lista. No tenemos datos horarios de demanda ni de generación para el horizonte, ni datos de la red de transmisión.
- El notebook debe correr completo en pocos minutos en un computador portátil y seguir siendo un tutorial que entienda un estudiante de maestría con conocimientos básicos de Python.
- Tenemos pocos días de trabajo.

Formato de la respuesta
Entre cinco y siete sugerencias, de la más a la menos importante. Para cada una:
a) qué cambiaría;
b) qué debilidad de la metodología corrige;
c) cómo implementarla en Python con los datos disponibles;
d) qué efecto esperaría sobre los resultados;
e) qué tan difícil es (baja, media o alta).
Si una sugerencia necesita datos que no están en la lista, dígalo. Al final, diga cuál implementaría primero y por qué.
```

Después del prompt, el equipo envió un segundo mensaje: «Activa el pensamiento extendido para dar respuesta al prompt anterior». Ese mensaje no cambia la configuración del modelo, y la IA dio una sola respuesta, que se reproduce completa a continuación.

### 5.2 La respuesta completa

La respuesta, sin cambios. La IA la escribió en Markdown, así que se reproduce con su formato.

> El punto débil no es el optimizador. Con una penalidad de 770 USD/MWh frente a costos de 21 a 145, el MILP solo llena la brecha en orden de costo hasta donde lo dejan los topes y los plazos. La respuesta la fijan entonces tres insumos: los retrasos, las fechas de entrada y el menú de tecnologías. Además, la brecha es la resta de dos números grandes: un error de 1% en la demanda o en la ENFICC existente (unos 900 GWh) mueve la brecha de 2026-2027 cerca de 50%.
> 1. Simular los retrasos y la demanda en lugar de usar fracciones esperadas y tres escenarios
>
> * a) En cada simulación, cada proyecto con OEF recibe un desenlace (a tiempo, tarde con un retraso en meses, o no entra) y la demanda se sortea dentro de la banda de la UPME. Las tres fracciones también se sortean, porque vienen de nueve plantas. La brecha pasa a ser una distribución y el MILP actual se corre sobre sus percentiles 50, 80 y 95.
> * b) Ningún proyecto entrega el 53% de su OEF, y como la energía sin cubrir es max(0, brecha), promediar antes de optimizar la subestima. Con nueve plantas, el 53% tiene un margen de unos ±30 puntos. Las reglas además contradicen sus propios datos: un proyecto vencido cuenta cero, aunque 19/(19+28) = 40% de la OEF que no llegó a tiempo terminó entrando. Y uno con compromiso en diciembre de 2026 que el 4 de noviembre aún no operaba cuenta 53%. El «estrés» es solo el percentil 84 de la demanda con los retrasos del caso base.
> * c) Con NumPy: `rng.dirichlet(conteos + 1, size=S)` da las fracciones de cada simulación, compartidas por todos los proyectos, lo que correlaciona sus retrasos. `rng.random` asigna el desenlace y `rng.choice` un retraso de los observados. Los sorteos que contradicen lo que ya se sabe al 4 de noviembre se descartan y se repiten: eso es condicionar. Trabajen por mes y agreguen al final, para no redondear 20,7 meses a dos períodos. Demanda: media + z × (borde superior − media), con el mismo z en los cuatro períodos. Corre en segundos con datos que ya tienen.
> * d) No sé hacia dónde se mueve la mediana, porque las dos correcciones de las reglas van en sentidos opuestos. Sí espero un rango muy ancho: solo por demanda, la brecha de 2029-2030 es 4.585 ± 5.350 GWh, con cerca de 20% de probabilidad de que no haya brecha. El plan base, que hoy deja unos 20 GWh sin cubrir en 2028-2030, dejaría del orden de 1.900 GWh esperados solo en 2029-2030. La respuesta pasa de «600 MW de gas» a una curva de MW contra nivel de confianza.
> * e) Media.
>
> 2. Medir los bloques del modelo con la misma vara que los proyectos con OEF
>
> * a) Aplicar a los bloques candidatos el mismo riesgo de retraso y de no entrada, y fijar su primer período factible con evidencia. Reportar dos cifras: los MW efectivos que hacen falta y los MW que habría que contratar.
> * b) Por lo que describen, un proyecto con OEF y garantías cuenta 53%, mientras un bloque que aún no existe llega completo y puntual. Eso hace ver la expansión más pequeña y barata de lo que es, justo en el parámetro que su sensibilidad señala como el más importante. Las dos últimas subastas dejaron casi cuatro años entre la adjudicación y el inicio de la obligación (lo cito de memoria), frente a los 25 meses que supone entrar en diciembre de 2028. Revisen también si el tope solar se llena con los MW nominales de proyectos que en la oferta cuentan 53% a 72%.
> * c) Versión corta: multiplicar el coeficiente de energía firme de cada bloque por la fracción esperada de su período. Versión completa: reutilizar los sorteos de la sugerencia 1 y darle a la energía no cubierta un índice de escenario. Son 24 variables enteras más 4 continuas por escenario, que HiGHS resuelve en segundos con 200 a 500 escenarios (matriz con `scipy.sparse`).
> * d) Si los bloques fallan como la cohorte de 2019, cubrir 2028-2029 exige contratar hasta 1/0,53 ≈ 1,9 veces la energía firme. El plan base (840 MW) se acerca al del estrés (1.540 MW) y su costo casi se duplica.
> * e) Baja la versión corta, media la completa.
>
> 3. Revisar el menú: gas sin límite de combustible y ninguna opción de corto plazo
>
> * a) Añadir una restricción de combustible a los bloques de gas. Dejar entrar opciones rápidas (generación de emergencia o diésel, demanda desconectable voluntaria) con costo alto y cupo explícito, en vez de un cupo de 0 MW.
> * b) El resultado depende de que el ciclo abierto sea la única tecnología barata sin cupo que llega a tiempo. Sus 600 MW necesitan del orden de 120 a 140 GBTUD de gas en firme, y los 1.200 MW del estrés el doble. La demanda nacional ronda 1.000 GBTUD y ya se cubre en parte con GNL importado (cifras de memoria). Para 2026-2028 el modelo no tiene variables de decisión: los 4.579 GWh sin cubrir son la brecha misma.
> * c) Una fila más en el MILP: bloques de gas acumulados × MW × factor firme × 24 h × consumo específico de calor ≤ gas disponible. El gas disponible está en el balance de gas natural de la UPME, que no está en su lista. Sin datos nuevos, barran el cupo de gas (0 a 1.200 MW) y el del diésel (0 a 1.000 MW) y grafiquen MW, costo y energía sin cubrir. El costo de la generación de emergencia y el potencial de demanda desconectable tampoco están en su lista.
> * d) Con gas limitado, el modelo pasa a solar, eólica y energía sin cubrir, y el costo marginal por MWh firme sube de 51 a entre 95 y 145 USD. Con opciones rápidas, la energía sin cubrir de 2026-2028 deja de ser un dato y aparece su precio.
> * e) Media (baja si se quedan en los barridos).
>
> 4. Separar el costo de tener la planta del costo de usarla
>
> * a) Dividir el costo en fijo (anualidad y O&M fijo) y variable (combustible), y cargar el variable por la generación esperada, no por la energía firme. Reportar por separado la inversión comprometida y el costo dentro del horizonte.
> * b) La energía firme es una capacidad para el año seco. Una térmica de respaldo genera mucho menos que eso en un año normal, y una solar o eólica genera más, y ese excedente hoy vale cero. El «costo por MWh firme» no dice cuántas horas de combustible supone. Que el diésel (47) quede por debajo del ciclo abierto (51) sugiere que no son las mismas para todas. Los 423 millones no son la inversión: por lo que reconstruyo, son dos anualidades de unos 270 millones, descontadas. Y la penalidad trata como racionamiento seguro un déficit que solo se materializa en un período seco.
> * c) Dos columnas en la tabla de costos. La probabilidad de año seco sale del ONI. Los factores de uso por tecnología salen de la generación por planta de la API de XM en meses secos y normales (`groupby`). La penalidad se usa como simple holgura y se reportan GWh, o se multiplica por esa probabilidad.
> * d) Los MW casi no cambian, porque el gas sigue siendo la firmeza más barata. Si hoy cargan el combustible por toda la energía firme, el costo cae hacia su sensibilidad de solo costo fijo. Con la penalidad ponderada, la eólica a 145 USD puede dejar de justificarse.
> * e) Baja.
>
> 5. Contrastar la contabilidad de energía firme con los años secos
>
> * a) Comparar, por tecnología, lo que el método cuenta como firme con lo generado en los meses críticos de 2015-2016 y 2023-2024. Recalcular así el factor firme de las plantas menores y dejar de suponer constante la ENFICC existente.
> * b) El balance nunca se compara con un resultado observado. Las plantas menores aportan 13,65 GWh-día, unos 4.980 GWh al año, casi tanto como la mayor brecha. Buena parte debe de ser hidráulica a filo de agua, que cae justo en el año seco. Los 237,7 GWh-día se mantienen cuatro años, aunque la ENFICC térmica depende de contratos de combustible que se renuevan.
> * c) Calculen el factor de planta mensual por tecnología y su mínimo móvil de 12 meses (`rolling(12).mean().min()`). Ya tienen los factores observados como sensibilidad; falta definirlos sobre la ventana seca. Dos comprobaciones baratas:
>    * 2027-2028 tiene 366 días, y un día de oferta son 251 GWh, casi 9% de esa brecha.
>    * La generación distribuida que resta la UPME no debe solaparse con las solares de 1 a 20 MW que ustedes suman.
> * d) Espero un factor menor para las pequeñas hidráulicas. Cada 10% de error en ese renglón son unos 500 GWh al año, 28% de la brecha de 2026-2027.
> * e) Media.
>
> 6. Medir el error de la homologación en GWh, no en pares
>
> * a) Evaluar ponderando por OEF y sobre pares no usados para elegir el umbral. Verificar a mano los 50 proyectos con OEF (9 + 41).
> * b) Según entiendo, la precisión de 0,92 y la exhaustividad de 0,71 se midieron sobre los mismos 205 pares con que se escogió el umbral, así que son optimistas. También pesan igual una planta de 5 MW y una de 500 MW. Una planta que ya opera y no se empareja queda como pendiente: se cuenta dos veces o infla el 28% de «no ha entrado».
> * c) Partir los 205 pares al azar, elegir el umbral con una mitad y reportar con la otra. Usar `np.average(acierto, weights=oef)` y una tabla CSV de correcciones manuales que prevalezca sobre RapidFuzz.
> * d) Poco efecto en la brecha central si los proyectos grandes ya están bien. Evita un doble conteo silencioso y da una barra de error al paso 3.
> * e) Baja.
>
> Debilidades que quedan aunque hagan todo lo anterior
>
> * Horizonte: cuatro años es menos que el plazo de casi cualquier tecnología. Para 2026-2028 no hay decisiones que optimizar, y una planta de 25 años se elige con dos años de costos.
> * Muestra de retrasos: nueve plantas de una sola subasta, afectadas por causas que el modelo excluye (transmisión, licencias, consultas). La simulación muestra esa ignorancia, no la reduce. Creo recordar que las subastas de 2008 y 2011-2012 también tienen plantas nuevas con compromisos vencidos (Ituango, Gecelca 3, Termocol, Porce IV). Si aparecen en sus datos, ampliarían la muestra.
> * Energía firme: es una cifra declarada y anual. Cubrirla no garantiza el suministro, y no cubrirla no equivale a racionar.
>
> Cuál implementaría primero
> La 1, en su versión mínima: sorteos, percentiles y el MILP sin tocar. La incertidumbre de la brecha es mayor que la brecha, así que una cifra única es lo que más debilita la respuesta. Son unas decenas de líneas de NumPy que un estudiante puede seguir, y permite retirar buena parte de las sensibilidades, lo que ayuda con el alcance. Enseguida haría la versión corta de la 2, que reutiliza esos sorteos y es la que más cambia los MW.

### 5.3 Qué va más allá de lo que recomendaron los profesores

Cada sugerencia frente a las recomendaciones que hicieron los profesores al evaluar la propuesta, y lo que hicimos con ella. Implementamos las dos que más cambian la respuesta sin pedir datos nuevos.

| Sugerencia de la IA | ¿La habían recomendado los profesores? | Qué hicimos |
|---|---|---|
| 1. Simular los retrasos y la demanda | No: la evaluación de la propuesta no habla de incertidumbre | **Implementada** (5.4 y 5.5): la respuesta pasa de una cifra a un nivel de confianza, con datos que ya teníamos |
| 2. Medir los bloques nuevos con la misma vara que los proyectos con OEF | No | **Implementada en su versión corta** (5.6): corrige una incoherencia del modelo |
| 3. Límite de combustible para el gas y opciones de corto plazo | No | Descartada: pide el balance de gas de la UPME y costos de generación de emergencia que no están en los datos. Queda como limitación (4.9) |
| 4. Separar el costo fijo del variable | En parte: pidieron documentar y justificar los costos | Descartada: el costo variable ya se carga por la generación esperada. Sí corregimos cómo se presenta el costo (5.7) |
| 5. Contrastar la energía firme con los años secos observados | En parte: pidieron un escenario normal y uno de El Niño | En parte, dentro del año normal (3.3): en los períodos secos observados la hidráulica generó 1,05 veces su ENFICC (mediana), así que la ENFICC describe bien el año seco. No recalculamos el factor de las plantas menores, porque el paquete no trae su generación |
| 6. Medir el error de la homologación en GWh y fuera de muestra | No en esa forma; profundiza la recomendación de darle más peso a RapidFuzz | No implementada: la propia IA espera poco efecto en la brecha. Es la mejora siguiente |
| Debilidades que quedan: horizonte, muestra de retrasos y alcance de la energía firme | No | Las reconocemos en 4.9 y 5.7 |

### 5.4 Mejora 1: la brecha como distribución

La sección 3.3 le aplica a cada proyecto el 53% de su OEF, pero ningún proyecto entrega el 53%: entra o no entra. Aquí cada simulación decide qué pasa con cada proyecto y sortea también las probabilidades y la demanda.

Cada simulación tiene cuatro piezas:
- **Las probabilidades también son inciertas,** porque salen de nueve plantas. Cada simulación las sortea de una distribución de Dirichlet con parámetros 1 + 9 × (las fracciones por OEF de la subasta de 2019): el "conteos + 1" que propone la IA, con los conteos medidos en OEF, como en la sección 3.1. Con nueve plantas, la fracción a tiempo se mueve unos ±27 puntos.
- **Todos los proyectos comparten las probabilidades de su simulación.** Si a uno le va mal, es más probable que a los demás también: los retrasos quedan correlacionados.
- **El retraso se cuenta en meses.** Quien entra tarde se demora 15,7, 18,8 o 27,6 meses, como las tres plantas que entraron tarde en 2019, y en cada período cuenta los meses que opera. La sección 3.3 redondea ese retraso a dos períodos completos.
- **La demanda se sortea** dentro de la banda de 68% de la UPME: media + *z* × (alta − media), con *z* normal estándar y el mismo *z* en los cuatro períodos, porque una demanda alta suele seguir alta.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `rng.dirichlet(alfa, size)` | Sortea vectores de probabilidades que suman 1 | `alfa` = 1 + 9 × fracciones de 2019 | Representa que las probabilidades salen de una muestra pequeña |
| `rng.random(forma)` | Números uniformes entre 0 y 1 | Uno por simulación y proyecto | Se comparan con las probabilidades para decidir el desenlace |
| `rng.choice(valores, size)` | Escoge al azar entre unos valores | Los tres retrasos observados | El retraso de quien entra tarde sale de lo observado |
| `rng.standard_normal(forma)` | Sortea una normal estándar | Una por simulación | El *z* de la demanda |
| `np.clip(a, 0, 1)` | Recorta los valores al intervalo [0, 1] | La fracción de cada período que opera un proyecto | No puede operar menos de cero meses ni más de doce |
| `operando()` y `simular_brecha()` (propias) | Fracción operando de cada proyecto y brecha simulada | 20.000 simulaciones con la semilla `SEMILLA` | Con la semilla fija, el notebook da siempre lo mismo |

In [59]:
FRACCIONES_2019 = resumen_2019["fraccion_oef"].reindex(["entró a tiempo", "entró tarde", "no ha entrado"]).to_numpy()
RETRASOS_2019 = tarde.to_numpy()                                   # meses de las tres plantas que entraron tarde
por_periodo = demanda.pivot(index="periodo", columns="escenario_demanda", values="demanda_gwh").reindex(PERIODOS)
DEMANDA_MEDIA, DEMANDA_ALTA = por_periodo["media"].to_numpy(), por_periodo["alta"].to_numpy()
base_33 = balance("base").set_index("periodo").reindex(PERIODOS)
SIN_NUEVOS = (base_33["oferta_firme_gwh"] - base_33["nuevas_gwh"]).to_numpy()   # oferta sin proyectos nuevos (GWh)


def operando(retraso, k, meses=True):
    """Fracción de cada período en que opera cada proyecto (simulación × proyecto × período).

    retraso: meses de retraso de cada proyecto en cada simulación (np.inf si no entra). k: período comprometido.
    Con meses=False solo cuentan los períodos completos, como en la regla de la sección 3.3.
    """
    i = np.arange(len(PERIODOS))[None, None, :] - k[None, :, None]     # períodos desde el compromiso
    fraccion = np.clip((12 * (i + 1) - retraso[:, :, None]) / 12, 0, 1) * (i >= 0)
    return fraccion if meses else np.floor(fraccion)

In [60]:
def simular_brecha(n=20_000, semilla=SEMILLA, tamano_muestra=9, demanda_incierta=True, meses=True):
    """Simula n veces la brecha firme del caso base (GWh por período), con la entrada de cada proyecto sorteada.

    tamano_muestra: plantas detrás de las probabilidades; cada simulación las sortea de una Dirichlet con
    parámetros 1 + tamano_muestra × fracciones de 2019. Con None quedan fijas, las de MODELO_RETRASOS.
    demanda_incierta: sortea la demanda en la banda de 68% de la UPME, con el mismo z en los cuatro períodos.
    meses: retraso en meses, de las plantas de 2019; con False, dos períodos completos (sección 3.3).
    """
    rng = np.random.default_rng(semilla)
    if tamano_muestra is None:
        p = np.tile([MODELO_RETRASOS["a_tiempo"], MODELO_RETRASOS["tarde"]], (n, 1))
    else:
        p = rng.dirichlet(1 + tamano_muestra * FRACCIONES_2019, size=n)[:, :2]
    azar = rng.random((n, len(nuevos)))                  # un sorteo por proyecto, con las p de su simulación
    a_tiempo = azar < p[:, [0]]
    entra_tarde = ~a_tiempo & (azar < p[:, [0]] + p[:, [1]])
    meses_tarde = (rng.choice(RETRASOS_2019, size=azar.shape) if meses
                   else np.full(azar.shape, 12.0 * MODELO_RETRASOS["periodos_retraso"]))
    retraso = np.where(a_tiempo, 0.0, np.where(entra_tarde, meses_tarde, np.inf))
    k = nuevos["periodo_entrada"].map(PERIODOS.index).to_numpy()
    oef = nuevos[COLUMNAS_OEF].to_numpy()[None]          # GWh-día de cada proyecto por período
    nuevas = (operando(retraso, k, meses) * oef).sum(axis=1) * dias.to_numpy()
    z = rng.standard_normal((n, 1)) if demanda_incierta else np.zeros((n, 1))
    demanda_simulada = DEMANDA_MEDIA + z * (DEMANDA_ALTA - DEMANDA_MEDIA)
    return pd.DataFrame(demanda_simulada - SIN_NUEVOS - nuevas, columns=PERIODOS)

Antes de usarla, una prueba: con las probabilidades fijas, el retraso de dos períodos completos y la demanda media, la media de las simulaciones tiene que reproducir la brecha de la sección 3.3. Lo que queda de diferencia es ruido del azar.

In [61]:
prueba = simular_brecha(tamano_muestra=None, demanda_incierta=False, meses=False)
diferencia = (prueba.mean() - base_33["brecha_gwh"]).abs().max()
assert diferencia < 30, f"La simulación no reproduce la brecha de la sección 3.3 (diferencia de {diferencia:,.0f} GWh)"
print(f"✓ Sin la incertidumbre nueva, la media reproduce la brecha de la sección 3.3 (diferencia máxima {diferencia:.1f} GWh)")

sim = simular_brecha()
percentiles = pd.DataFrame({"valor esperado (sección 3.3)": base_33["brecha_gwh"], "P5": sim.quantile(0.05),
                            "P50": sim.median(), "P80": sim.quantile(0.8), "P95": sim.quantile(0.95)}).round(0)
percentiles["probabilidad de que no falte"] = (sim <= 0).mean().round(2)
percentiles

✓ Sin la incertidumbre nueva, la media reproduce la brecha de la sección 3.3 (diferencia máxima 6.7 GWh)


,valor esperado (sección 3.3),P5,P50,P80,P95,probabilidad de que no falte
2026-2027,1770.0,-2696.0,1765.0,4062.0,6234.0,0.26
2027-2028,2809.0,-3210.0,2930.0,6103.0,9139.0,0.22
2028-2029,5168.0,-2439.0,5077.0,8947.0,12679.0,0.13
2029-2030,4585.0,-4077.0,4809.0,9427.0,13877.0,0.19


In [62]:
fig = go.Figure()
for (bajo, alto), opacidad, nombre in [((0.05, 0.95), 0.18, "90% central de las simulaciones"),
                                       ((0.2, 0.8), 0.38, "60% central de las simulaciones")]:
    fig.add_trace(go.Scatter(x=PERIODOS + PERIODOS[::-1], y=list(sim.quantile(alto)) + list(sim.quantile(bajo))[::-1],
                             mode="lines", fill="toself", fillcolor=f"rgba(42,120,214,{opacidad})",
                             line=dict(width=0), name=nombre, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=PERIODOS, y=sim.median(), mode="lines+markers", name="Mediana",
                         line=dict(color="#184f95", width=2)))
fig.add_trace(go.Scatter(x=PERIODOS, y=base_33["brecha_gwh"], mode="markers", name="Valor esperado (sección 3.3)",
                         marker=dict(symbol="diamond", size=11, color="#eb6834")))
fig.add_hline(y=0, line_color="#888", line_width=1)
fig.update_layout(title="La brecha firme del caso base como distribución (20.000 simulaciones)",
                  xaxis_title="Período del cargo", yaxis_title="Brecha firme (GWh)", legend=LEYENDA, height=460,
                  separators=",.", yaxis_tickformat=",.0f")
fig.show()

**Lectura.** La simulación pasa la prueba: sin la incertidumbre nueva, su media reproduce la brecha de la sección 3.3 (diferencia máxima de 6,7 GWh). Con ella, la brecha deja de ser una cifra. En 2029-2030 la mediana es de 4.809 GWh, cerca de los 4.585 del valor esperado, pero entre el 5% y el 95% de las simulaciones va de −4.077 a 13.877 GWh. En lo central, la IA tenía razón: la incertidumbre es mayor que la brecha. En el 19% de las simulaciones no falta energía firme en 2029-2030, y en el 13% no falta en 2028-2029, el peor período.

La mayor parte de ese ancho viene de la demanda: en 2026-2027, cuando no entra ningún proyecto nuevo, la brecha va de −2.696 a 6.234 GWh solo por ella. La mediana se mueve poco frente al valor esperado, y en sentidos opuestos, como anticipó la IA. Sube en 2027-2028 y 2029-2030, porque la Dirichlet con nueve plantas y el «+1» acercan las tres probabilidades a un tercio: la fracción a tiempo baja en promedio del 53% al 48%. Baja en 2028-2029, porque contar el retraso por meses le suma a ese período la parte del año en que ya operan los proyectos que entraron tarde.

### 5.5 El plan del caso base frente a la distribución

El plan de la sección 3.4 se dimensionó para el valor esperado de la brecha. Aquí se mide con qué probabilidad la cubre y se resuelve el mismo MILP para los percentiles 50, 80 y 95.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `np.maximum(a, 0)` | Deja en cero los valores negativos | La brecha simulada menos la energía firme del plan | La energía sin cubrir es max(0, brecha − plan): un período con holgura no compensa otro con faltante |
| `evaluar_plan(det, sim)` (propia) | Energía firme de un plan, probabilidad de cubrir y energía esperada sin cubrir, por período | El detalle de `correr()` y las simulaciones | Mide cualquier plan contra la misma distribución |
| `correr("base", brecha_extra=...)` | El MILP de la sección 3.4 | La diferencia entre el percentil y el valor esperado | Resuelve el mismo modelo con otra brecha, sin tocarlo |

In [63]:
def evaluar_plan(det, sim):
    """Energía firme nueva de un plan en cada período y, frente a la brecha simulada, la probabilidad de cubrirla
    y la energía esperada sin cubrir (GWh). det: el detalle que devuelve correr()."""
    firme = (det["x"] * det["pr"]["F"]).sum(axis=0)
    falta = np.maximum(sim.to_numpy() - firme, 0)
    return pd.DataFrame({"energía firme nueva (GWh)": firme, "probabilidad de cubrir": (sim.to_numpy() <= firme).mean(axis=0),
                         "energía esperada sin cubrir (GWh)": falta.mean(axis=0)}, index=PERIODOS)


evaluar_plan(corridas["base"][1], sim).round(2)

,energía firme nueva (GWh),probabilidad de cubrir,energía esperada sin cubrir (GWh)
2026-2027,0.00,0.26,2187.50
2027-2028,0.00,0.22,3408.23
2028-2029,5147.38,0.51,1807.59
2029-2030,5147.38,0.53,2025.82


**Lectura.** El plan de 840 MW de la sección 3.4 aporta 5.147 GWh firmes desde 2028-2029. Frente a la brecha simulada, cubre 2028-2029 en el 51% de los casos y 2029-2030 en el 53%: es un plan para la mediana, no para el riesgo. La energía esperada sin cubrir de esos dos períodos suma 3.833 GWh, cuando el valor esperado decía que quedaban 20. La IA estimó del orden de 1.900 GWh esperados sin cubrir en 2029-2030, solo por la demanda; con los retrasos también sorteados, son 2.026. Es su punto central: como la energía sin cubrir es max(0, brecha − plan), promediar la brecha antes de optimizar la subestima. En 2026-2028 no hay plan posible, y la energía esperada sin cubrir es la brecha misma: 5.596 GWh entre los dos períodos, frente a los 4.579 del valor esperado.

In [64]:
planes, niveles = {}, [("Valor esperado (3.4)", None), ("P50", 0.5), ("P80", 0.8), ("P95", 0.95)]
for nivel, q in niveles:
    resumen, det = correr("base", brecha_extra=0 if q is None else sim.quantile(q) - base_33["brecha_gwh"])
    evaluacion = evaluar_plan(det, sim)
    mw = {NOMBRES_TECNOLOGIA[k[3:]]: v for k, v in resumen.items() if k.startswith("mw_")}
    planes[nivel] = mw | {"MW nuevos": sum(mw.values()), "costo de los bloques (MUSD, VP)": resumen["costo_bloques_vp_musd"],
                          "probabilidad de cubrir 2029-2030": evaluacion.at["2029-2030", "probabilidad de cubrir"],
                          "energía esperada sin cubrir 2028-2030 (GWh)":
                              evaluacion["energía esperada sin cubrir (GWh)"].iloc[2:].sum()}
planes = pd.DataFrame(planes).T
planes = planes.loc[:, (planes != 0).any()]
planes.round(2)

,Bagazo,Eólica,"Gas, ciclo abierto",Solar FV,MW nuevos,"costo de los bloques (MUSD, VP)",probabilidad de cubrir 2029-2030,energía esperada sin cubrir 2028-2030 (GWh)
Valor esperado (3.4),40.0,0.0,600.0,200.0,840.0,422.88,0.53,3833.41
P50,40.0,0.0,600.0,200.0,840.0,422.88,0.53,3833.41
P80,40.0,0.0,1200.0,100.0,1340.0,754.93,0.82,1070.34
P95,40.0,400.0,1400.0,1200.0,3040.0,1148.62,0.93,266.28


In [65]:
tecnologias = [t for t in NOMBRES_TECNOLOGIA.values() if t in planes]
largo = planes[tecnologias].rename_axis("nivel").reset_index().melt(id_vars="nivel", var_name="tecnología", value_name="MW")
fig = px.bar(largo, x="nivel", y="MW", color="tecnología", color_discrete_map=COLORES_TECNOLOGIA,
             title="MW nuevos del caso base según la brecha que se quiere cubrir",
             labels={"nivel": "Brecha que se cubre", "MW": "MW nuevos en 2029-2030", "tecnología": "Tecnología"})
for nivel, fila in planes.iterrows():
    texto = (f"{fila['costo de los bloques (MUSD, VP)']:,.0f} MUSD".replace(",", ".")
             + f"<br>cubre 2029-2030 en el {fila['probabilidad de cubrir 2029-2030']:.0%}")
    fig.add_annotation(x=nivel, y=fila["MW nuevos"], text=texto, showarrow=False, yshift=22)
fig.update_layout(height=480, legend=LEYENDA, separators=",.")
fig.update_yaxes(range=[0, planes["MW nuevos"].max() * 1.22], tickformat=",.0f")
fig.show()

**Lectura.** El plan para la mediana (P50) es el mismo de la sección 3.4: 840 MW por 423 MUSD. Cubrir la brecha con 80% de confianza pide 1.340 MW (1.200 de ciclo abierto, 100 solares y 40 de bagazo) por 755 MUSD, un 79% más; ese plan cubre 2029-2030 en el 82% de las simulaciones y deja 1.070 GWh esperados sin cubrir en 2028-2030. Con 95% de confianza hacen falta 3.040 MW y 1.149 MUSD: el ciclo abierto llena su cupo de 1.400 MW y entran 1.200 MW solares y 400 eólicos, más caros por MWh firme. Aun así, el plan cubre 2029-2030 en el 93% de los casos, porque los cupos y el tope solar ya no dejan construir más. Pasar del 80% al 95% cuesta más que pasar de la mediana al 80%. La respuesta deja de ser «600 MW de gas» y pasa a ser una curva: cuántos MW hacen falta según el riesgo que se esté dispuesto a correr.

### 5.6 Mejora 2: los bloques nuevos con el mismo riesgo

Un proyecto con OEF entra con retraso, pero un bloque del MILP llega completo y a tiempo. La versión corta de la sugerencia 2 le aplica a cada bloque la regla de los proyectos: 53% de su energía firme en sus dos primeros períodos y 72% desde el tercero.

El bloque se paga completo aunque no llegue, así que el MILP compensa contratando más. La tabla separa los MW que habría que contratar de la energía firme que se espera de ellos. Es un caso pesimista: la cohorte de 2019 era sobre todo eólica en La Guajira, con problemas de transmisión y de consultas previas, y su tasa de fracaso exagera la de una térmica de ciclo abierto.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `con_riesgo(pr)` (propia) | Copia el MILP y multiplica la energía firme de cada bloque por la fracción que se espera que opere | `MODELO_RETRASOS` | Es la regla de la sección 3.3, aplicada ahora a los bloques |
| `dict \| dict` | Une dos diccionarios; el de la derecha manda | El problema y su nueva matriz | Deja intacto el problema original |

In [66]:
def con_riesgo(pr, modelo=MODELO_RETRASOS):
    """Copia del MILP en la que un bloque que entra en τ aporta en t solo la fracción esperada de su energía firme:
    la de entrar a tiempo en sus primeros `periodos_retraso` períodos y, desde ahí, también la de entrar tarde."""
    A, nt = pr["A"].copy(), len(PERIODOS)
    nb = (A.shape[1] - nt) // nt
    for t in range(nt):                                   # las primeras nt filas son las de la energía firme
        for tau in range(t + 1):
            fraccion = modelo["a_tiempo"] + (modelo["tarde"] if t - tau >= modelo["periodos_retraso"] else 0)
            A[t, [k * nt + tau for k in range(nb)]] *= fraccion
    return pr | {"A": A}


riesgo, nt = {}, len(PERIODOS)
for nivel, q in [("Valor esperado", None), ("P80", 0.8)]:
    det = correr("base", brecha_extra=0 if q is None else sim.quantile(q) - base_33["brecha_gwh"])[1]
    for bloques_llegan, pr in [("a tiempo (3.4)", det["pr"]), ("con el riesgo de 2019", con_riesgo(det["pr"]))]:
        res = resolver(pr)
        y = np.round(res.x[:-nt])
        x = y.reshape(-1, nt).cumsum(axis=1)
        mw = x[:, -1] * det["bl"]["tamano_mw"].to_numpy()
        riesgo[(nivel, bloques_llegan)] = {
            "MW a contratar": mw.sum(), "de ellos, ciclo abierto": mw[(det["bl"]["tecnologia"] == "gas_ciclo_abierto").to_numpy()].sum(),
            "energía firme esperada 2029-2030 (GWh)": (pr["A"][:nt, :-nt] @ y)[-1],
            "costo de los bloques (MUSD, VP)": float(pr["delta"] @ (pr["C"] @ x)), "energía sin cubrir (GWh)": res.x[-nt:].sum()}
pd.DataFrame(riesgo).T.rename_axis(["brecha", "los bloques llegan"]).round(0)

MW a contratar  de ellos, ciclo abierto  energía firme esperada 2029-2030 (GWh)  costo de los bloques (MUSD, VP)  \
brecha         los bloques llegan                                                                                                                        
Valor esperado a tiempo (3.4)                  840.0                    600.0                                  5147.0                            423.0   
               con el riesgo de 2019          1340.0                   1200.0                                  5175.0                            788.0   
P80            a tiempo (3.4)                 1340.0                   1200.0                                  9764.0                            755.0   
               con el riesgo de 2019          3040.0                   1400.0                                  6934.0                           1149.0   

                                      energía sin cubrir (GWh)  
brecha         los bloques llegan                               
Valor esperado a tiempo (3.4)                           4599.0  
               con el riesgo de 2019                    4579.0  
P80            a tiempo (3.4)                          10165.0  
               con el riesgo de 2019                   14991.0

**Lectura.** La IA anticipó bien el efecto. Si los bloques nuevos se retrasan como los proyectos de 2019, cubrir el valor esperado de la brecha pide contratar 1.340 MW en lugar de 840 (1.200 de ciclo abierto en lugar de 600), y el costo de los bloques pasa de 423 a 788 MUSD, un 86% más. La energía firme que se espera de ellos en 2029-2030 es casi la misma (5.175 frente a 5.147 GWh): se contrata más para recibir lo mismo. Con la brecha del P80 ya no alcanza: el MILP contrata 3.040 MW, con el cupo de gas lleno, y solo espera 6.934 GWh firmes en 2029-2030, frente a una brecha de 9.427. Las dos mejoras apuntan al mismo lado: el plan de la sección 3.4 es el mínimo, no el prudente.

### 5.7 Lo que descartamos y por qué

Una respuesta de IA no se copia: se revisa. Esto es lo que no aplicaba a nuestros datos, lo que la IA no podía ver y lo que simplificamos al implementar.

- **Lo que no aplicaba a nuestros datos.** La IA razonó sobre la descripción del método, sin los datos, y dos de sus críticas a las reglas no cambian ninguna cifra: ningún proyecto está vencido y ninguno tiene compromiso de entrada en diciembre de 2026 (23 entran en 2027-2028, 2 en 2028-2029 y 16 en 2029-2030). Tampoco aplican otras dos sospechas. El tope solar no se llena con MW nominales: ya usa los MW esperados con retrasos (sección 3.4). Y el combustible no se carga por toda la energía firme, sino por la generación esperada, con un factor de planta de 0,32 para el ciclo abierto y de 0,15 para el diésel; por eso el diésel sale más barato por MWh firme, como la IA intuyó. El año de 366 días ya se contaba.
- **Un error de nuestro prompt.** No mencionamos el cupo de gas: 1.400 MW compartidos entre ciclo abierto y ciclo combinado (PIEG 2025-2039, Tabla 23). Por eso la IA supuso que el gas no tenía límite. Su punto de fondo sigue en pie: el modelo limita los MW de gas, no el combustible, y en el P95 ese cupo ya se llena.
- **Lo que la IA no podía ver.** Las nueve plantas de 2019 no pesan lo mismo: Tesorito tiene el 53% de la OEF, así que el 53% que entró a tiempo es, en la práctica, una sola planta. Con el tamaño efectivo de Kish, la muestra equivale a 3,2 plantas iguales, y la incertidumbre de los retrasos es mayor que la que supusimos con nueve. El ejercicio 3 mide cuánto cambia el P80.
- **Lo que simplificamos.** Cada percentil se calcula período por período, así que el plan del P80 cubre el P80 de cada período, no el 80% de los horizontes completos. Los retrasos en meses salen de solo tres plantas. El «+1» de la Dirichlet acerca las probabilidades a un tercio y mueve la mediana. Y la mejora 2 está en su versión corta: el riesgo de los bloques entra como valor esperado, no sorteado junto con la mejora 1.
- **Lo que no usamos.** Las cifras que la IA dio de memoria: casi cuatro años entre la adjudicación y el inicio de la obligación en las últimas subastas, el gas que consume el ciclo abierto, la demanda nacional de gas y las plantas de las subastas de 2008 y 2011-2012. No las verificamos, así que no entran en ningún cálculo.
- **Cómo se presenta el costo.** La IA notó que los 423 MUSD no son la inversión, y tiene razón: son el valor presente de los costos anuales dentro del horizonte. La inversión del portafolio base ronda los 803 MUSD, y la sección 4.8 ahora lo dice.

### 5.8 La respuesta, con las mejoras

Las dos mejoras no cambian la pregunta: cambian la forma de la respuesta. Esta es la respuesta de la sección 4.8, con la brecha tratada como una distribución.

**Cuánta y cuándo.** La brecha del caso base tiene un valor esperado de 4.585 GWh en 2029-2030, pero entre el 5% y el 95% de las simulaciones va de −4.077 a 13.877 GWh. La mayor parte de ese rango viene de la demanda.

**Qué se construye y cuánto cuesta, según el riesgo.** El plan de la sección 3.4 (840 MW por 423 MUSD) cubre 2029-2030 en la mitad de los casos (53%). Para cubrirlo con 80% de confianza hacen falta 1.340 MW, sobre todo de ciclo abierto, por 755 MUSD. Con 95%, 3.040 MW y 1.149 MUSD, y ni así se cubre del todo, porque se llenan los cupos. Si además los bloques nuevos se retrasan como los proyectos de 2019, el plan del valor esperado ya pide 1.340 MW y 788 MUSD.

**Lo que no cambia.** Antes de diciembre de 2028 no hay con qué, y lo que más mueve la respuesta son los plazos. La expansión de mínimo costo sigue siendo sobre todo gas de ciclo abierto: las mejoras cambian cuánto hay que construir según el riesgo que se acepte, no con qué.

**Lo que sigue.** Los ejercicios de la sección 6 repiten y extienden lo hecho: otras medidas de similitud, un proyecto que no llega y el tamaño real de la muestra de retrasos.

<a id="ejercicios" name="ejercicios"></a>
## 6. Ejercicios para el lector

Tres ejercicios, de menor a mayor dificultad. Usan las tablas y las funciones que el notebook ya definió, así que hay que correrlo completo antes. Las soluciones están en la [sección siguiente](#soluciones).

**Los ejercicios se revisan solos.** Debajo de cada enunciado hay una celda para la respuesta: se reemplaza cada `None` por un número y se corre. La función `verificar_ejercicio_N()` calcula la respuesta esperada con las funciones del tutorial, la compara con la suya y dice qué está bien y qué no, con una pista, sin mostrar la solución. Las celdas de verificación revelan parte del método: conviene no leerlas antes de intentarlo.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `revisar(que, dado, esperado, tolerancia, pista)` (propia) | Compara una respuesta con el valor esperado e imprime ✓, ✗ con una pista, o · si todavía no hay respuesta | La tolerancia de cada respuesta | Una respuesta vacía o mal escrita no detiene el notebook |
| `verificar_ejercicio_1()`, `_2()` y `_3()` (propias) | Calculan las respuestas esperadas y llaman a `revisar()` | Los valores de la celda de respuesta | En el ejercicio 3 la tolerancia acepta otra semilla u otro número de simulaciones |

In [67]:
def revisar(que, dado, esperado, tolerancia, pista):
    """Compara una respuesta con el valor esperado e imprime ✓, ✗ con una pista, o · si falta la respuesta."""
    if dado is None:
        print(f"·  {que}: sin respuesta todavía")
        return
    try:
        correcto = abs(float(dado) - float(esperado)) <= tolerancia
    except (TypeError, ValueError):                   # texto u otro objeto en lugar de un número
        print(f"✗  {que}: la respuesta debe ser un número")
        return
    print(f"✓  {que}" if correcto else f"✗  {que}: no coincide. Pista: {pista}")

### Ejercicio 1 · Otras medidas de similitud *(fácil)*

Un ejercicio fácil: repetir la comparación de nombres de la sección 3.1 con otras medidas de RapidFuzz.

**Enunciado.** En la sección 3.1 se usaron `token_set_ratio` y `WRatio`. Compare cuatro medidas de RapidFuzz, `fuzz.ratio`, `fuzz.partial_ratio`, `fuzz.token_set_ratio` y `fuzz.WRatio`, en estos cinco pares:

| a | b | ¿Es la misma planta? |
|---|---|---|
| La Loma Solar | PARQUE SOLAR LA LOMA | Sí |
| La Loma Solar | SOLAR LA LOMA II | No: es una segunda etapa |
| Termo Tasajero II | TASAJERO 1 | No: son unidades distintas |
| Hidroituango | ITUANGO | Sí |
| Central Hidroeléctrica Guatapé | GUATAPE | Sí |

Normalice los nombres con `normalizar()` y marque qué medidas pasan el umbral elegido en la sección 3.1 (`elegido["umbral"]`). ¿Alguna medida acierta en los cinco pares? ¿Qué regla de la sección 3.1 corrige los errores que quedan?

**Pistas:**
- Arme una lista de diccionarios, una fila por par, y conviértala en `DataFrame`.
- `fuzz.ratio` compara las dos cadenas completas; `fuzz.partial_ratio` compara la cadena corta con el pedazo más parecido de la larga.
- `numeros()` devuelve los números de un nombre normalizado: dos etapas distintas tienen números distintos.

**Su respuesta:** cuántos de los cinco pares acierta cada medida, solo con el puntaje y sumando la prueba de números (un par se acepta si además los números coinciden).

In [68]:
def verificar_ejercicio_1(aciertos_solo, aciertos_con_numeros):
    """Revisa el ejercicio 1: aciertos de 5 de cada medida, solo con el puntaje y con la prueba de números."""
    pares = [("La Loma Solar", "PARQUE SOLAR LA LOMA", True), ("La Loma Solar", "SOLAR LA LOMA II", False),
             ("Termo Tasajero II", "TASAJERO 1", False), ("Hidroituango", "ITUANGO", True),
             ("Central Hidroeléctrica Guatapé", "GUATAPE", True)]
    medidas = {"ratio": fuzz.ratio, "partial_ratio": fuzz.partial_ratio,
               "token_set_ratio": fuzz.token_set_ratio, "WRatio": fuzz.WRatio}
    for m, f in medidas.items():
        acepta = [f(normalizar(a), normalizar(b)) >= elegido["umbral"] for a, b, _ in pares]
        iguales = [numeros(normalizar(a)) == numeros(normalizar(b)) for a, b, _ in pares]
        solo = sum(ac == misma for ac, (_, _, misma) in zip(acepta, pares))
        con_numeros = sum((ac and ig) == misma for ac, ig, (_, _, misma) in zip(acepta, iguales, pares))
        revisar(f"{m}, solo con el puntaje", (aciertos_solo or {}).get(m), solo, 0,
                f"normalice antes de puntuar y acepte con puntaje >= {elegido['umbral']:.0f}")
        revisar(f"{m}, con la prueba de números", (aciertos_con_numeros or {}).get(m), con_numeros, 0,
                "un par se acepta solo si pasa el umbral y sus números coinciden")

In [69]:
# Su respuesta al ejercicio 1: aciertos de 5 por medida (reemplace cada None por un número y corra la celda)
aciertos_solo = {"ratio": None, "partial_ratio": None, "token_set_ratio": None, "WRatio": None}
aciertos_con_numeros = {"ratio": None, "partial_ratio": None, "token_set_ratio": None, "WRatio": None}
verificar_ejercicio_1(aciertos_solo, aciertos_con_numeros)

·  ratio, solo con el puntaje: sin respuesta todavía
·  ratio, con la prueba de números: sin respuesta todavía
·  partial_ratio, solo con el puntaje: sin respuesta todavía
·  partial_ratio, con la prueba de números: sin respuesta todavía
·  token_set_ratio, solo con el puntaje: sin respuesta todavía
·  token_set_ratio, con la prueba de números: sin respuesta todavía
·  WRatio, solo con el puntaje: sin respuesta todavía
·  WRatio, con la prueba de números: sin respuesta todavía


### Ejercicio 2 · ¿Y si la obra de Ituango no llega? *(intermedio)*

Un ejercicio intermedio: medir cuánto depende la respuesta de un solo proyecto, combinando el balance de la sección 3.3 con el MILP de la 3.4.

**Enunciado.** La oferta firme de 2029-2030 incluye 2,17 GWh-día por la obra del embalse de Ituango: es uno de los proyectos de `nuevos` (sección 3.3). Suponga que la obra no llega:
1. Quite Ituango de `nuevos` y recalcule la OEF nueva con retrasos. ¿Cuántos GWh firmes se pierden en cada período?
2. Sume esa pérdida a la brecha del base y del estrés, y resuelva el MILP de cada escenario.
3. ¿Cambia el portafolio? ¿Cuánto cuestan los bloques en valor presente? Explique por qué la respuesta del base y la del estrés no son iguales.

**Pistas:**
- `oferta_nuevos(nuevos_sin_ituango, True)` da la OEF nueva con retrasos en GWh-día; réstela de la original y multiplíquela por `dias`.
- Con retrasos, solo una parte de la OEF de un proyecto entra en su período (sección 3.3), así que la pérdida no es 2,17 × 365.
- `correr(escenario, brecha_extra=perdida)` suma la pérdida a la brecha antes de resolver.
- Mire cuánto le sobra al portafolio del base en 2029-2030.

**Su respuesta:** los GWh firmes que se pierden en 2029-2030 y el costo de los bloques en valor presente (MUSD) del base y del estrés sin la obra.

In [70]:
def verificar_ejercicio_2(perdida_2029_2030, costo_base, costo_estres):
    """Revisa el ejercicio 2: GWh firmes que se pierden en 2029-2030 sin la obra de Ituango y costo de los
    bloques (MUSD en valor presente) del base y del estrés sin ella."""
    sin_obra = nuevos[~nuevos["nombre"].str.contains("ITUANGO", case=False, na=False)]
    perdida = (oferta_nuevos(nuevos, True) - oferta_nuevos(sin_obra, True)) * dias
    revisar("GWh que se pierden en 2029-2030", perdida_2029_2030, perdida["2029-2030"], 1,
            "con retrasos, en su primer período solo cuenta una parte de la OEF del proyecto")
    for nombre, dado, escenario in [("base", costo_base, "base"), ("estrés", costo_estres, "estres")]:
        esperado = correr(escenario, brecha_extra=perdida)[0]["costo_bloques_vp_musd"]
        revisar(f"costo de los bloques del {nombre} sin la obra", dado, esperado, 1,
                "sume la pérdida a la brecha con correr(escenario, brecha_extra=...) y lea costo_bloques_vp_musd")

In [71]:
# Su respuesta al ejercicio 2 (reemplace cada None por un número y corra la celda)
perdida_2029_2030 = None       # GWh firmes que se pierden en 2029-2030
costo_base_sin_obra = None     # costo de los bloques del base sin la obra, MUSD en valor presente
costo_estres_sin_obra = None   # lo mismo, en el estrés
verificar_ejercicio_2(perdida_2029_2030, costo_base_sin_obra, costo_estres_sin_obra)

·  GWh que se pierden en 2029-2030: sin respuesta todavía
·  costo de los bloques del base sin la obra: sin respuesta todavía
·  costo de los bloques del estrés sin la obra: sin respuesta todavía


### Ejercicio 3 · ¿Nueve plantas o tres? *(difícil)*

Un ejercicio difícil, que extiende la sección 5.4: cuánto cambia la incertidumbre si la muestra de retrasos de 2019 vale por menos plantas de las que parece.

**Enunciado.** La mejora 1 de la sección 5.4 sortea las probabilidades de entrada como si vinieran de nueve plantas iguales. Pero no pesan lo mismo: Tesorito tiene más de la mitad de la OEF de las nueve plantas nuevas de la subasta de 2019. El **tamaño efectivo de Kish**, $n_{ef} = (\sum_i w_i)^2 / \sum_i w_i^2$, con $w_i$ la OEF de cada planta, dice a cuántas plantas iguales equivale la muestra.
1. Calcule $n_{ef}$ con la OEF de las nueve plantas.
2. Repita la simulación de la sección 5.4 con $n_{ef}$ plantas en lugar de nueve: primero sin la incertidumbre de la demanda y después con ella. Compare el P80 de 2029-2030 con el de nueve plantas.
3. ¿Por qué la diferencia entre nueve plantas y $n_{ef}$ es menor cuando se incluye la demanda?

**Pistas:**
- `s2019["oef_registrada_gwh_dia"]` (sección 3.1) trae la OEF de las nueve plantas.
- `simular_brecha(tamano_muestra=..., demanda_incierta=False)` deja fija la demanda en el escenario medio.
- El P80 de 2029-2030 es `sim["2029-2030"].quantile(0.8)`.
- Para la pregunta 3, compare cuánto se dispersa la brecha por los retrasos y cuánto por la demanda.

**Su respuesta:** $n_{ef}$ y el P80 de la brecha de 2029-2030 con $n_{ef}$ plantas, sin y con la incertidumbre de la demanda (GWh). Como el resultado es aleatorio, la revisión acepta otra semilla.

In [72]:
def verificar_ejercicio_3(n_ef, p80_sin_demanda, p80_con_demanda):
    """Revisa el ejercicio 3: tamaño efectivo de Kish de las nueve plantas de 2019 y el P80 de la brecha de
    2029-2030 con ese tamaño, sin y con la incertidumbre de la demanda (referencia de 50.000 simulaciones)."""
    w = s2019["oef_registrada_gwh_dia"]
    esperado = w.sum() ** 2 / (w ** 2).sum()
    revisar("tamaño efectivo de Kish", n_ef, esperado, 0.01, "(suma de la OEF)² dividido por la suma de la OEF al cuadrado")
    for que, dado, incierta, tolerancia in [("P80 de 2029-2030 sin la demanda", p80_sin_demanda, False, 120),
                                            ("P80 de 2029-2030 con la demanda", p80_con_demanda, True, 250)]:
        referencia = simular_brecha(50_000, SEMILLA + 1, tamano_muestra=esperado, demanda_incierta=incierta)
        revisar(que, dado, referencia["2029-2030"].quantile(0.8), tolerancia,
                "simular_brecha(tamano_muestra=n_ef, demanda_incierta=...) y el cuantil 0,8 de 2029-2030")

In [73]:
# Su respuesta al ejercicio 3 (reemplace cada None por un número y corra la celda)
n_ef = None              # tamaño efectivo de Kish de las nueve plantas de 2019
p80_sin_demanda = None   # P80 de la brecha de 2029-2030 con n_ef plantas y la demanda media, GWh
p80_con_demanda = None   # lo mismo, con la incertidumbre de la demanda, GWh
verificar_ejercicio_3(n_ef, p80_sin_demanda, p80_con_demanda)

·  tamaño efectivo de Kish: sin respuesta todavía


·  P80 de 2029-2030 sin la demanda: sin respuesta todavía


·  P80 de 2029-2030 con la demanda: sin respuesta todavía


<a id="soluciones" name="soluciones"></a>
## Soluciones de los ejercicios

Cada solución trae su código, la revisión automática que la confirma y una lectura del resultado.

### Solución 1

Se normalizan los cinco pares, se puntúan con las cuatro medidas y se cuenta cuántos acierta cada una, con y sin la prueba de números.

In [74]:
# Solución 1: cuatro medidas de RapidFuzz sobre cinco pares, después de normalizar
umbral = elegido["umbral"]
pares = [("La Loma Solar", "PARQUE SOLAR LA LOMA", True), ("La Loma Solar", "SOLAR LA LOMA II", False),
         ("Termo Tasajero II", "TASAJERO 1", False), ("Hidroituango", "ITUANGO", True),
         ("Central Hidroeléctrica Guatapé", "GUATAPE", True)]
medidas = {"ratio": fuzz.ratio, "partial_ratio": fuzz.partial_ratio,
           "token_set_ratio": fuzz.token_set_ratio, "WRatio": fuzz.WRatio}

filas = []
for a, b, misma in pares:
    na, nb = normalizar(a), normalizar(b)
    fila = {"a": a, "b": b, "normalizados": f"{na} | {nb}", "misma planta": misma,
            "números iguales": numeros(na) == numeros(nb)}
    fila.update({m: round(f(na, nb), 1) for m, f in medidas.items()})
    filas.append(fila)
sol1 = pd.DataFrame(filas)
display(sol1)

# ¿Acierta cada medida? Solo con el puntaje (>= umbral) y sumando la prueba de números
acepta = sol1[list(medidas)].ge(umbral)
aciertos = pd.DataFrame({
    "solo con el puntaje": acepta.eq(sol1["misma planta"], axis=0).sum(),
    "con la prueba de números": (acepta & sol1[["números iguales"]].to_numpy()).eq(sol1["misma planta"], axis=0).sum()})
aciertos.rename_axis(f"aciertos de 5 (umbral {umbral:.0f})")

,a,b,normalizados,misma planta,números iguales,ratio,partial_ratio,token_set_ratio,WRatio
0,La Loma Solar,PARQUE SOLAR LA LOMA,LOMA | LOMA,True,True,100.0,100.0,100.0,100.0
1,La Loma Solar,SOLAR LA LOMA II,LOMA | LOMA 2,False,False,80.0,100.0,100.0,90.0
2,Termo Tasajero II,TASAJERO 1,TERMO TASAJERO 2 | TASAJERO 1,False,False,69.2,90.0,88.9,85.5
3,Hidroituango,ITUANGO,HIDROITUANGO | ITUANGO,True,True,73.7,100.0,73.7,90.0
4,Central Hidroeléctrica Guatapé,GUATAPE,GUATAPE | GUATAPE,True,True,100.0,100.0,100.0,100.0


,solo con el puntaje,con la prueba de números
aciertos de 5 (umbral 86),,
ratio,4,4
partial_ratio,3,5
token_set_ratio,2,4
WRatio,4,5


In [75]:
# La solución pasa la revisión del ejercicio 1
verificar_ejercicio_1(aciertos["solo con el puntaje"].to_dict(), aciertos["con la prueba de números"].to_dict())

✓  ratio, solo con el puntaje
✓  ratio, con la prueba de números
✓  partial_ratio, solo con el puntaje
✓  partial_ratio, con la prueba de números
✓  token_set_ratio, solo con el puntaje
✓  token_set_ratio, con la prueba de números
✓  WRatio, solo con el puntaje
✓  WRatio, con la prueba de números


**Lectura.** Solo con el puntaje, ninguna medida acierta en los cinco pares: `ratio` y `WRatio` fallan en uno, `partial_ratio` en dos y `token_set_ratio`, la medida que escogió la sección 3.1, en tres. Las dos trampas son de número: LOMA frente a LOMA 2 y TASAJERO 2 frente a TASAJERO 1 se parecen casi del todo. Con la prueba de números (la regla `numeros_ok` de la sección 3.1, aquí simplificada, porque la del tutorial también acepta un 1 de un solo lado), `partial_ratio` y `WRatio` aciertan los cinco. Hidroituango es el caso contrario: es la misma planta, con el prefijo HIDRO pegado al nombre, y `ratio` y `token_set_ratio` lo rechazan porque comparan cadenas o palabras completas.

Cinco pares no bastan para escoger una medida. La sección 3.1 la escoge con 205 pares revisados, donde lo que pesa son los cientos de autogeneradores con nombres parecidos a los de los proyectos. Una extensión natural es agregar `partial_ratio` a esa curva.

### Solución 2

Se quita la obra de Ituango de los proyectos nuevos, se mide la OEF que se pierde y se resuelve el MILP con esa pérdida sumada a la brecha.

In [76]:
# Solución 2: la OEF que se pierde sin la obra de Ituango y el portafolio con y sin ella
sin_ituango = nuevos[~nuevos["nombre"].str.contains("ITUANGO", case=False, na=False)]
perdida = (oferta_nuevos(nuevos, True) - oferta_nuevos(sin_ituango, True)) * dias
print("GWh firmes que se pierden sin la obra:", perdida.round(1).to_dict())

filas = {}
for escenario in ["base", "estres"]:
    for caso, extra in [("con la obra", 0), ("sin la obra", perdida)]:
        resumen, _ = correr(escenario, brecha_extra=extra)
        mw = {NOMBRES_TECNOLOGIA[k[3:]]: v for k, v in resumen.items() if k.startswith("mw_") and v > 0}
        filas[(escenario, caso)] = mw | {"costo de los bloques (MUSD, VP)": round(resumen["costo_bloques_vp_musd"], 1)}
pd.DataFrame(filas).fillna(0).T.rename_axis(["escenario", "Ituango"])

GWh firmes que se pierden sin la obra: {'2026-2027': 0.0, '2027-2028': 0.0, '2028-2029': 0.0, '2029-2030': 420.3}


Bagazo  Gas, ciclo abierto  Solar FV  costo de los bloques (MUSD, VP)  Eólica
escenario Ituango                                                                                   
base      con la obra    40.0               600.0     200.0                            422.9     0.0
          sin la obra    40.0               600.0     200.0                            422.9     0.0
estres    con la obra    40.0              1200.0     200.0                            809.9   100.0
          sin la obra    40.0              1300.0     100.0                            818.7     0.0

In [77]:
# La solución pasa la revisión del ejercicio 2
verificar_ejercicio_2(perdida["2029-2030"], filas[("base", "sin la obra")]["costo de los bloques (MUSD, VP)"],
                      filas[("estres", "sin la obra")]["costo de los bloques (MUSD, VP)"])

✓  GWh que se pierden en 2029-2030
✓  costo de los bloques del base sin la obra
✓  costo de los bloques del estrés sin la obra


**Lectura.** Sin la obra se pierden 420 GWh firmes en 2029-2030: con retrasos solo cuenta el 53% de los 2,17 GWh-día de Ituango (0,53 × 2,17 × 365). En el base, el portafolio y su costo no cambian (600 MW de ciclo abierto, 200 solares y 40 de bagazo, por 423 MUSD): lo que se construye en 2028-2029 para una brecha de 5.168 GWh sobra en 2029-2030, cuando la brecha sube de 4.585 a 5.005 GWh. En el estrés no hay esa holgura: la brecha de 2029-2030 pasa de 9.933 a 10.354 GWh, y el MILP cambia los 100 MW eólicos y un bloque solar de 100 MW por un bloque más de ciclo abierto (1.300 MW en total). El costo de los bloques sube de 810 a 819 MUSD, un 1,1%. La misma pérdida cambia el portafolio o no según cuánto le sobre al período, porque el MILP construye con bloques indivisibles.

### Solución 3

Se calcula el tamaño efectivo de Kish de las nueve plantas y se repite la simulación de la sección 5.4 con ese tamaño, sin y con la incertidumbre de la demanda.

In [78]:
# Solución 3: el tamaño efectivo de la muestra de 2019 y su efecto sobre el P80 de 2029-2030
w = s2019["oef_registrada_gwh_dia"]
n_ef = w.sum() ** 2 / (w ** 2).sum()
print(f"Tamaño efectivo de Kish: {n_ef:.2f} plantas iguales (Tesorito tiene el {w.max() / w.sum():.0%} de la OEF)")

filas = {"9 plantas (sección 5.4)": 9, "plantas efectivas (Kish)": n_ef}
p80 = pd.DataFrame({demanda_txt: {fila: simular_brecha(tamano_muestra=tamano, demanda_incierta=incierta)["2029-2030"]
                                  .quantile(0.8) for fila, tamano in filas.items()}
                    for demanda_txt, incierta in [("sin la demanda", False), ("con la demanda", True)]})
p80.loc["diferencia"] = p80.loc["plantas efectivas (Kish)"] - p80.loc["9 plantas (sección 5.4)"]
p80.round(0).rename_axis("P80 de la brecha de 2029-2030 (GWh)")

Tamaño efectivo de Kish: 3.19 plantas iguales (Tesorito tiene el 53% de la OEF)


,sin la demanda,con la demanda
P80 de la brecha de 2029-2030 (GWh),,
9 plantas (sección 5.4),5737.0,9427.0
plantas efectivas (Kish),6131.0,9675.0
diferencia,394.0,248.0


In [79]:
# La solución pasa la revisión del ejercicio 3
kish = p80.loc["plantas efectivas (Kish)"]
verificar_ejercicio_3(n_ef, kish["sin la demanda"], kish["con la demanda"])

✓  tamaño efectivo de Kish


✓  P80 de 2029-2030 sin la demanda


✓  P80 de 2029-2030 con la demanda


**Lectura.** Las nueve plantas valen por 3,19: Tesorito tiene el 53% de la OEF. Con esa muestra efectiva, las probabilidades sorteadas se dispersan más, y el P80 de la brecha de 2029-2030 sube 394 GWh sin la incertidumbre de la demanda (de 5.737 a 6.131), un 7%. Con la demanda, la diferencia es menor: 248 GWh (de 9.427 a 9.675), un 3%.

La razón es que la demanda dispersa la brecha mucho más que los retrasos: el escenario alto de la UPME queda 5.348 GWh por encima del medio en 2029-2030, mientras que medir mejor los retrasos mueve el P80 unos cientos de GWh. Cuando se suman dos fuentes de incertidumbre, la más ancha domina los percentiles. Para la respuesta, eso quiere decir que el plan del P80 depende sobre todo de cuánto se crea en la proyección de la UPME.

<a id="evaluacion" name="evaluacion"></a>
## Cómo atendimos la evaluación de la propuesta

Los profesores evaluaron la propuesta con 82 sobre 100 y dejaron comentarios por criterio y tres recomendaciones. Esta tabla dice qué hicimos con cada uno y dónde se ve en este notebook.

| Comentario de la evaluación | Qué hicimos | Dónde se ve |
|---|---|---|
| La pregunta tenía dos objetivos, y un MILP de mínimo costo no produce el precio de bolsa | Una sola pregunta, de suficiencia y expansión, sin el precio de bolsa | Sección 1 (la pregunta) y 4.8 |
| El horizonte («corto y mediano plazo») no estaba delimitado | Los cuatro períodos del Cargo por Confiabilidad, de diciembre de 2026 a noviembre de 2030 | Sección 1 (qué abarca) |
| Recomendación: preguntar cuánta energía y para cuándo, con un escenario normal y uno de El Niño | La brecha de cada período con El Niño (energía firme) y con un año de hidrología normal; la expansión se dimensiona para El Niño | Secciones 3.3, 4.2 y 4.8 |
| Con estos datos no se puede modelar el precio de bolsa | El precio queda como contexto descriptivo, por condición hidrológica y fase de El Niño, y se lee como asociación, no como causalidad | Secciones 2.5 y 4.9 |
| Para escoger bloques de mínimo costo hacen falta costos por tecnología, con su fuente | Cada bloque candidato trae su costo, con una jerarquía de fuentes: Catálogo Tecnológico de la UPME, PIEG 2025-2039, CNE (Chile), EPE (Brasil) e IRENA | Secciones 3.2 y 4.4 |
| Recomendación: si mantienen el MILP, documentar y justificar los costos y los demás supuestos de cada tecnología | Costos, factores firmes, primer período factible, cupos con su fuente, tope solar y penalidad; cada supuesto se prueba en las sensibilidades | Secciones 3.2, 3.4 y 3.5 |
| Faltaba una llave común entre las bases, y el año calendario no coincide con el período de diciembre a noviembre | La homologación con RapidFuzz da la llave común, y todo se agrega por período del cargo | Secciones 3.1 y 2.5 |
| Recomendación: la homologación con RapidFuzz es un muy buen aporte; darle más peso | Subsección propia: normalización, puntajes, revisión de atributos y umbral escogido con 205 pares revisados. De ahí salen los retrasos de la subasta de 2019 | Sección 3.1 y ejercicio 1 |
| El alcance era demasiado grande para un tutorial | El notebook responde la pregunta con el método central y deja por fuera lo que no aporta a la respuesta: el pipeline completo, un simulador y dos consultas en vivo. La app del proyecto queda como un enlace | Todo el notebook; sección 4.7 |
| La fuente debe ser pública y verificable; el gestor normativo de la CREG es normativa, no datos | Los datos son de XM, la UPME, NOAA y el DANE, verificados con su huella SHA-256 y revisados tabla por tabla; la CREG entra solo como contexto | Secciones 2.2 y 2.4 |
| Bien que declaren el uso de IA y cómo se valida | Declaración de uso de IA, comparación de cada resultado con el pipeline (`comprobar()`) y una sección sobre cómo usamos la IA para ir más allá de la metodología | Secciones 3.3 a 3.5 y 5; declaración de uso de IA |
| La página de la propuesta estaba muy cargada y decía «Phyton» | Una introducción de máximo tres líneas en cada sección y una revisión de ortografía de todo el texto | Todo el notebook |

<a id="ia" name="ia"></a>
## Declaración de uso de IA

Usamos IA generativa (Claude, de Anthropic, a través de Claude Code) en todo el proyecto, bajo la dirección del equipo:

- **Código.** Claude escribió la mayor parte del código del pipeline del proyecto y de este notebook, a partir del plan de trabajo y de nuestras instrucciones. Este notebook rehace con funciones propias el método central del pipeline. El equipo definió la pregunta, las fuentes y las decisiones de modelación, y revisó los resultados.
- **Fuentes.** Buscó fuentes de costos y referencias (XM, UPME, CREG y prensa) y transcribió datos, siempre con URL, página y estado de verificación. Las cifras de prensa se identifican como tales.
- **Homologación.** Sugirió las etiquetas de los 205 pares revisados de la sección 3.1. Después, a pedido del equipo, un agente de Claude las revisó de forma independiente contra los atributos de las fuentes, par por par y con evidencia.
- **Plantas, proyectos y cupos.** A pedido del equipo, agentes de Claude verificaron con fuentes primarias (UPME, XM, SIMEM, CREG, ANLA y promotores) los MW de los proyectos con OEF, las ampliaciones con obras y los cupos de bagazo, diésel y eólica. El equipo escogió los valores y aprobó los supuestos.
- **Redacción.** Los textos de este notebook.
- **Ir más allá de la metodología.** La consulta a la IA sobre cómo mejorar el método, con el prompt, la respuesta completa y las dos mejoras que implementamos, está en la [sección 5](#mas-alla).

**Cómo se controla:**
- Toda cifra se calcula en Python desde los datos.
- Este notebook compara sus resultados con los del pipeline, que los calculó con su propio código (`comprobar()`, en 3.3, 3.4 y 3.5).
- Los datos del paquete se verifican con su huella SHA-256, y los datos transcritos traen fuente y página.

<a id="referencias" name="referencias"></a>
## Referencias

**Académicas**

- Cramton, P. y Stoft, S. (2007). *Colombia Firm Energy Market*. Proceedings of the 40th Annual Hawaii International Conference on System Sciences (HICSS'07). IEEE. [doi.org/10.1109/HICSS.2007.133](https://doi.org/10.1109/HICSS.2007.133) ([versión de los autores](https://www.cramton.umd.edu/papers2005-2009/cramton-stoft-colombia-firm-energy-market.pdf)).
- Cramton, P., Ockenfels, A. y Stoft, S. (2013). *Capacity Market Fundamentals*. Economics of Energy & Environmental Policy, 2(2), 27-46. [doi.org/10.5547/2160-5890.2.2.2](https://doi.org/10.5547/2160-5890.2.2.2)
- Fellegi, I. P. y Sunter, A. B. (1969). *A Theory for Record Linkage*. Journal of the American Statistical Association, 64(328), 1183-1210. [doi.org/10.1080/01621459.1969.10501049](https://doi.org/10.1080/01621459.1969.10501049)
- Harbord, D. y Pagnozzi, M. (2014). *Britain's Electricity Capacity Auctions: Lessons from Colombia and New England*. The Electricity Journal, 27(5), 54-62. [doi.org/10.1016/j.tej.2014.05.004](https://doi.org/10.1016/j.tej.2014.05.004)
- Juvinao Navarro, J. C. (2021). *Lecciones del Cargo por Confiabilidad en Colombia como un mecanismo de incentivo a la generación de energía eléctrica*. Desarrollo y Sociedad, (87), 113-148. Universidad de los Andes. [doi.org/10.13043/dys.87.4](https://doi.org/10.13043/dys.87.4)
- Koltsaklis, N. E. y Dagoumas, A. S. (2018). *State-of-the-art generation expansion planning: A review*. Applied Energy, 230, 563-589. [doi.org/10.1016/j.apenergy.2018.08.087](https://doi.org/10.1016/j.apenergy.2018.08.087)
- Poveda, G., Álvarez, D. M. y Rueda, Ó. A. (2011). *Hydro-climatic variability over the Andes of Colombia associated with ENSO: a review of climatic processes and their impact on one of the Earth's most important biodiversity hotspots*. Climate Dynamics, 36(11-12), 2233-2249. [doi.org/10.1007/s00382-010-0931-y](https://doi.org/10.1007/s00382-010-0931-y)
- Ríos, D., Pérez, A., Carabalí, J. y Meneses, L. (2024). *Market power and adverse weather events in a hydro-dominated electricity market*. Borradores de Economía, 1266. Banco de la República. [doi.org/10.32468/be.1266](https://doi.org/10.32468/be.1266)

**Regulación, planeación y datos**

- CREG (2006). *Resolución 071 de 2006: metodología para la remuneración del Cargo por Confiabilidad en el Mercado Mayorista de Energía*. [gestornormativo.creg.gov.co](https://gestornormativo.creg.gov.co/gestor/entorno/docs/resolucion_creg_0071_2006.htm)
- UPME (2026). *Plan Indicativo de Expansión de la Generación 2025-2039*. [docs.upme.gov.co](https://docs.upme.gov.co/SIMEC/Energia%20Electrica/PIEG/2025-2039/PIEG_2025_2039_Version_final.pdf)
- UPME (2026). *Proyección de la demanda de energía eléctrica y potencia máxima 2026-2040*, rev. julio de 2026 (tablas de agosto de 2026). [upme.gov.co](https://www.upme.gov.co/simec/planeacion-energetica/proyeccion-de-demanda-v2/)
- MinEnergía, UPME y Agencia Danesa de Energía (2025). *Catálogo Tecnológico Colombiano*, API abierta. [api.upme.gov.co](https://api.upme.gov.co/catalogo_tecnologias/api/techs_year/)
- XM (2026). *XM informa los resultados de la subasta de asignación de Obligaciones de Energía Firme, OEF, para la vigencia diciembre 2029 – noviembre 2030* (22 de mayo de 2026). [xm.com.co](https://www.xm.com.co/noticias/9118-xm-informa-los-resultados-de-la-subasta-de-asignacion-de-obligaciones-de-energia)
- XM. API de datos (`servapibi.xm.com.co`) y SIMEM, consultados en octubre y noviembre de 2026 con `requests` y [pydataxm](https://github.com/EquipoAnaliticaXM/API_XM). Reporte de ENFICC verificada frente a OEF asignadas (octubre de 2026) y recursos no despachados centralmente con contratos para 2029-2030 (junio de 2026).
- NOAA Climate Prediction Center. *Oceanic Niño Index (ONI)*. [cpc.ncep.noaa.gov](https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt)
- FRED (CPI de EE. UU.), DANE (IPC) y datos.gov.co (TRM), para llevar los costos y precios a una misma moneda y año.
- Asocaña (2026). *Informe Anual 2025-2026* (p. 39 y Cuadro 5, p. 63). [asocana.org](https://www.asocana.org/documentos/362026-9AEF4DC0-00FF00,000A000,878787,C3C3C3,0F0F0F,B4B4B4,FF00FF,FFFFFF,2D2D2D,A3C4B5.pdf). Las demás fuentes de los cupos están en `data/manual/fuentes_cupos.csv`.

**Costos de tecnologías**

- National Laboratory of the Rockies, antes NREL (2025). *Annual Technology Baseline 2025*. [atb.nlr.gov](https://atb.nlr.gov/electricity/2025/index)
- EIA (2026). *Assumptions to the Annual Energy Outlook 2026: Electricity Market Module* (abril de 2026). [eia.gov](https://www.eia.gov/outlooks/aeo/assumptions/pdf/EMM_Assumptions.pdf)
- IRENA (2026). *Renewable Power Generation Costs in 2025*. [irena.org](https://www.irena.org/Publications/2026/Jul/Renewable-Power-Generation-Costs-in-2025)
- Lazard (2026). *Levelized Cost of Energy+* (LCOE, versión 19.0). [lazard.com](https://www.lazard.com/research-insights/financial-advisory/levelized-cost-of-energyplus-lcoeplus/)
- CNE Chile (2026). *Informe de Costos de Tecnologías de Generación y Almacenamiento* (mayo de 2026). [cne.cl](https://www.cne.cl/wp-content/uploads/2026/06/ICTGA-2026.pdf)
- EPE y MME (2025). *PDE 2035: Caderno de Parâmetros de Custos de Geração e Transmissão*. [epe.gov.br](https://www.epe.gov.br/pt/publicacoes-dados-abertos/publicacoes/plano-decenal-de-expansao-de-energia-2035)

**Prensa**

- Portafolio (2026). Nota sobre el Boletín 364 de XM: faltantes de energía firme de hasta 7.669 GWh en 2026-2027. [portafolio.co](https://www.portafolio.co/energia/colombia-podria-enfrentar-un-deficit-de-energia-en-2027-xm-advierte-faltantes-de-hasta-7-669-gwh-504633)

La documentación oficial de cada paquete está en la tabla de la sección 2.1. El detalle de cada fuente, con su edición y su estado de verificación, está en el plan de trabajo del proyecto.